-------------------

с учётом также видео

In [2]:
# ============================================================  
# ПОЛНЫЙ ФАЙЛ: av_train_infer.py
# Аудио + Видео (AVConformerCTC) → текст с таймкодами
# ============================================================

import os, io, cv2, subprocess, tempfile, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T

# import torchvision.transforms as VT

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from datasets import load_dataset

from datasets import load_dataset, Audio

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ============================================================
# 1. TOKENIZER
# ============================================================
class CharTokenizer:
    def __init__(self):
        chars = "абвгдеёжзийклмнопрстуфхцчшщъыьэюя "
        chars += "abcdefghijklmnopqrstuvwxyz0123456789.,!?-"
        self.vocab     = ["<blank>", "<unk>"] + list(chars)
        self.char2id   = {c: i for i, c in enumerate(self.vocab)}
        self.id2char   = {i: c for i, c in enumerate(self.vocab)}

    def encode(self, text):
        return [self.char2id.get(c, 1) for c in text.lower()]

    def decode_greedy(self, ids):
        result, prev = [], -1
        for i in ids:
            if i != prev and i != 0:
                result.append(self.id2char.get(i, ""))
            prev = i
        return "".join(result)

tokenizer  = CharTokenizer()
VOCAB_SIZE = len(tokenizer.vocab)

# ============================================================
# 2. FEATURE EXTRACTORS
# ============================================================
class LogMelFeatureExtractor:
    def __init__(self, sample_rate=16000, n_mels=80):
        self.target_sr = sample_rate
        self.transform = T.MelSpectrogram(
            sample_rate=sample_rate, n_mels=n_mels,
            win_length=400, hop_length=160, n_fft=512,
        )
        self.to_db = T.AmplitudeToDB(top_db=80)

    def __call__(self, audio_bytes):
        waveform, sr = torchaudio.load(io.BytesIO(audio_bytes))
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        mel = self.transform(waveform)
        log_mel = self.to_db(mel)
        log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-9)
        return log_mel.squeeze(0).T   # (T, 80)

# ============================================================
# FEATURE EXTRACTORS — VideoFrameExtractor без torchvision
# ============================================================
class VideoFrameExtractor:
    """Извлекает кадры через cv2 — без torchvision/torchcodec"""
    def __init__(self, target_fps=10, img_size=112):
        self.target_fps = target_fps
        self.img_size   = img_size
        # Нормализация ImageNet вручную
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def _preprocess(self, frame_bgr):
        """BGR numpy → нормализованный тензор (3, H, W)"""
        frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (self.img_size, self.img_size))
        frame = frame.astype(np.float32) / 255.0
        frame = (frame - self.mean) / self.std       # (H, W, 3)
        return torch.from_numpy(frame).permute(2, 0, 1)  # (3, H, W)

    def extract(self, video_path, start_sec, end_sec, max_frames=64):
        cap       = cv2.VideoCapture(video_path)
        video_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        step      = max(1, int(video_fps / self.target_fps))

        start_frame = int(start_sec * video_fps)
        end_frame   = int(end_sec   * video_fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        frames = []
        frame_idx = start_frame
        while frame_idx < end_frame:
            ret, frame = cap.read()
            if not ret:
                break
            if (frame_idx - start_frame) % step == 0:
                frames.append(self._preprocess(frame))
            frame_idx += 1
            if len(frames) >= max_frames:
                break

        cap.release()
        if not frames:
            return None
        return torch.stack(frames)

audio_extractor = LogMelFeatureExtractor()
video_extractor = VideoFrameExtractor()

# ============================================================
# 3. DATASET — sova_rudevices (только аудио, видео синтезируем как заглушку)
# ============================================================
class SovaAVDataset(Dataset):
    """
    sova_rudevices не содержит видео — для обучения аудио-ветки
    передаём video=None. Когда будет видео-датасет, подключишь сюда.
    """
    def __init__(self, hf_dataset, max_duration_sec=15.0):
        self.data       = hf_dataset
        self.max_frames = int(max_duration_sec * 100)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample   = self.data[idx]
        features = audio_extractor(sample["audio"]["bytes"])
        if features.shape[0] > self.max_frames:
            features = features[:self.max_frames]
        target = torch.tensor(
            tokenizer.encode(sample["transcription"]), dtype=torch.long
        )
        # video=None — модель работает в audio-only режиме
        return features, None, target, sample["transcription"]

def collate_fn(batch):
    features, videos, targets, texts = zip(*batch)

    feat_lengths   = torch.tensor([f.shape[0] for f in features])
    target_lengths = torch.tensor([len(t) for t in targets])

    feat_padded = torch.zeros(len(features), feat_lengths.max(), features[0].shape[1])
    tgt_padded  = torch.zeros(len(targets), target_lengths.max(), dtype=torch.long)

    for i, (f, t) in enumerate(zip(features, targets)):
        feat_padded[i, :f.shape[0]] = f
        tgt_padded[i,  :len(t)]     = t

    # videos остаются None если датасет без видео
    return feat_padded, videos, tgt_padded, feat_lengths, target_lengths, texts

# ============================================================
# 4. МОДЕЛЬ — AVConformerCTC
# ============================================================
class ConvolutionModule(nn.Module):
    def __init__(self, d_model, kernel_size=31):
        super().__init__()
        self.norm     = nn.LayerNorm(d_model)
        self.pw_conv1 = nn.Conv1d(d_model, 2 * d_model, 1)
        self.dw_conv  = nn.Conv1d(d_model, d_model, kernel_size,
                                  padding=(kernel_size - 1) // 2, groups=d_model)
        self.bn       = nn.BatchNorm1d(d_model)
        self.pw_conv2 = nn.Conv1d(d_model, d_model, 1)
        self.dropout  = nn.Dropout(0.1)

    def forward(self, x):
        residual = x
        x = self.norm(x).transpose(1, 2)
        x = F.glu(self.pw_conv1(x), dim=1)
        x = F.silu(self.bn(self.dw_conv(x)))
        x = self.pw_conv2(x)
        return self.dropout(x).transpose(1, 2) + residual

class ConformerBlock(nn.Module):
    def __init__(self, d_model=256, n_heads=4, ff_dim=1024, kernel_size=31):
        super().__init__()
        self.ff1_norm  = nn.LayerNorm(d_model)
        self.ff1       = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.SiLU(), nn.Dropout(0.1),
            nn.Linear(ff_dim, d_model), nn.Dropout(0.1),
        )
        self.norm_attn = nn.LayerNorm(d_model)
        self.attn      = nn.MultiheadAttention(d_model, n_heads,
                                               dropout=0.1, batch_first=True)
        self.conv      = ConvolutionModule(d_model, kernel_size)
        self.ff2_norm  = nn.LayerNorm(d_model)
        self.ff2       = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.SiLU(), nn.Dropout(0.1),
            nn.Linear(ff_dim, d_model), nn.Dropout(0.1),
        )
        self.norm_out  = nn.LayerNorm(d_model)
        self.dropout   = nn.Dropout(0.1)

    def forward(self, x, key_padding_mask=None):
        x        = x + 0.5 * self.ff1(self.ff1_norm(x))
        residual = x
        x_norm   = self.norm_attn(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm,
                                key_padding_mask=key_padding_mask)
        x = residual + self.dropout(attn_out)
        x = self.conv(x)
        x = x + 0.5 * self.ff2(self.ff2_norm(x))
        return self.norm_out(x)

class ConvSubsampling(nn.Module):
    def __init__(self, n_mels=80, d_model=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, d_model, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(d_model, d_model, 3, stride=2, padding=1), nn.ReLU(),
        )
        self.proj = nn.Linear(d_model * (n_mels // 4), d_model)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        B, C, Tv, F_ = x.shape
        return self.proj(x.transpose(1, 2).reshape(B, Tv, C * F_))

class VideoEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3,   32,  3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32,  64,  3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64,  128, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.proj = nn.Linear(128 * 4 * 4, d_model)

    def forward(self, frames):          # (T_v, 3, 112, 112)
        x = self.cnn(frames)            # (T_v, 128, 4, 4)
        x = x.view(x.shape[0], -1)
        return self.proj(x)             # (T_v, d_model)

class AudioVisualFusion(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            d_model, num_heads=4, batch_first=True
        )

    def forward(self, audio_feat, video_feat):
        # audio_feat: (B, T_a, d)  video_feat: (B, T_v, d)
        # выравниваем видео по длине аудио
        video_feat = F.interpolate(
            video_feat.transpose(1, 2),
            size=audio_feat.shape[1],
            mode="linear", align_corners=False,
        ).transpose(1, 2)                          # (B, T_a, d)

        fused, _ = self.cross_attn(
            query=audio_feat,
            key=video_feat,
            value=video_feat,
        )
        return audio_feat + fused                  # residual

class AVConformerCTC(nn.Module):
    def __init__(self, vocab_size, n_mels=80, d_model=256,
                 n_layers=6, n_heads=4):
        super().__init__()
        self.subsampling = ConvSubsampling(n_mels, d_model)
        self.pos_enc     = nn.Embedding(5000, d_model)
        self.conformer   = nn.ModuleList([
            ConformerBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        self.video_enc   = VideoEncoder(d_model)
        self.fusion      = AudioVisualFusion(d_model)
        self.ctc_head    = nn.Linear(d_model, vocab_size)

    def forward(self, audio, video_frames=None):
        x   = self.subsampling(audio)
        pos = torch.arange(x.size(1), device=x.device)
        x   = x + self.pos_enc(pos)

        for block in self.conformer:
            x = block(x)
        # x: (B, T_a, d_model)

        if video_frames is not None:
            v = self.video_enc(video_frames)        # (T_v, d_model)
            v = v.unsqueeze(0)                      # (1, T_v, d_model)
            x = self.fusion(x, v)                   # (B, T_a, d_model)

        return F.log_softmax(self.ctc_head(x), dim=-1)

# ============================================================
# 5. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================
def wer(reference, hypothesis):
    ref = reference.lower().split()
    hyp = hypothesis.lower().split()
    d   = np.zeros((len(ref) + 1, len(hyp) + 1))
    for i in range(len(ref) + 1): d[i][0] = i
    for j in range(len(hyp) + 1): d[0][j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost   = 0 if ref[i-1] == hyp[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(ref)][len(hyp)] / max(len(ref), 1)

def seconds_to_timestamp(sec):
    """123.4 → '00:02:03'"""
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"



Device: cuda


In [ ]:
# ============================================================
# 6. ОБУЧЕНИЕ
# ============================================================
def train(n_epochs=1, batch_size=8, lr=3e-4,
          checkpoint_path="av_conformer_best.pt"):

    print("Loading dataset...")
    dataset    = load_dataset("bond005/sova_rudevices")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    train_ds   = SovaAVDataset(dataset["train"].select(range(30_000))) # для быстрого теста
    val_ds     = SovaAVDataset(dataset["test"])

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, collate_fn=collate_fn)

    model     = AVConformerCTC(vocab_size=VOCAB_SIZE).to(DEVICE)
    n_params  = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Model params: {n_params:.1f}M")

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
    best_wer  = float("inf")

    for epoch in range(n_epochs):
        # ── TRAIN ──────────────────────────────────────────
        model.train()
        train_losses = []
        t0 = time.time()

        for step, (feats, videos, targets, feat_lens, tgt_lens, texts) in enumerate(train_loader):
            feats   = feats.to(DEVICE)
            targets = targets.to(DEVICE)

            # videos — None для sova (аудио-датасет), передаём как есть
            video_tensor = None
            if videos[0] is not None:
                video_tensor = torch.stack(videos).to(DEVICE)

            log_probs     = model(feats, video_tensor)
            input_lengths = (feat_lens // 4).clamp(max=log_probs.size(1))

            loss = F.ctc_loss(
                log_probs.transpose(0, 1),
                targets, input_lengths, tgt_lens,
                blank=0, zero_infinity=True,
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_losses.append(loss.item())

            if step % 100 == 0:
                elapsed = time.time() - t0
                print(f"  [{epoch+1}/{n_epochs}] step {step:4d}/{len(train_loader)} "
                      f"| loss {loss.item():.4f} | {elapsed:.0f}s")

        # ── VALIDATION ─────────────────────────────────────
        model.eval()
        val_losses, wer_scores = [], []

        with torch.no_grad():
            for feats, videos, targets, feat_lens, tgt_lens, texts in val_loader:
                feats   = feats.to(DEVICE)
                targets = targets.to(DEVICE)

                log_probs     = model(feats)
                input_lengths = (feat_lens // 4).clamp(max=log_probs.size(1))

                loss = F.ctc_loss(
                    log_probs.transpose(0, 1),
                    targets, input_lengths, tgt_lens,
                    blank=0, zero_infinity=True,
                )
                val_losses.append(loss.item())

                preds = log_probs.argmax(dim=-1).cpu()
                for i, pred_ids in enumerate(preds):
                    hyp = tokenizer.decode_greedy(pred_ids.tolist())
                    wer_scores.append(wer(texts[i], hyp))

        avg_train = np.mean(train_losses)
        avg_val   = np.mean(val_losses)
        avg_wer   = np.mean(wer_scores)
        scheduler.step()

        print(f"\n{'─'*60}")
        print(f"Epoch {epoch+1:2d}/{n_epochs} | "
              f"train_loss={avg_train:.4f} | "
              f"val_loss={avg_val:.4f} | "
              f"WER={avg_wer:.3f}")
        print(f"{'─'*60}\n")

        # Сохраняем если WER улучшился
        if avg_wer < best_wer:
            best_wer = avg_wer
            torch.save({
                "epoch":        epoch + 1,
                "model_state":  model.state_dict(),
                "optimizer":    optimizer.state_dict(),
                "val_loss":     avg_val,
                "wer":          avg_wer,
                "vocab_size":   VOCAB_SIZE,
            }, checkpoint_path)
            print(f"  ✓ Checkpoint saved → {checkpoint_path}  (WER={avg_wer:.3f})\n")

    return model

# ============================================================
# 7. ИНФЕРЕНС НА РЕАЛЬНОМ ВИДЕО
# ============================================================
def extract_audio_from_video(video_path, sample_rate=16000):
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_path = tmp.name
    subprocess.run([
        "ffmpeg", "-y", "-i", video_path,
        "-ar", str(sample_rate), "-ac", "1", "-f", "wav", tmp_path,
    ], check=True, capture_output=True)
    with open(tmp_path, "rb") as f:
        audio_bytes = f.read()
    os.unlink(tmp_path)
    return audio_bytes

def split_into_segments(audio_bytes, segment_sec=20.0,
                        overlap_sec=1.0, sample_rate=16000):
    waveform, sr = torchaudio.load(io.BytesIO(audio_bytes))
    if sr != sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, sample_rate)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    seg_samples  = int(segment_sec  * sample_rate)
    step_samples = int((segment_sec - overlap_sec) * sample_rate)
    total        = waveform.shape[1]
    segments     = []
    start        = 0

    while start < total:
        end   = min(start + seg_samples, total)
        chunk = waveform[:, start:end]
        rms   = chunk.pow(2).mean().sqrt().item()
        if rms > 0.001:                            # пропускаем тишину
            buf = io.BytesIO()
            torchaudio.save(buf, chunk, sample_rate, format="wav")
            segments.append({
                "bytes":     buf.getvalue(),
                "start_sec": start / sample_rate,
                "end_sec":   end   / sample_rate,
            })
        start += step_samples
        if end == total:
            break

    return segments

@torch.no_grad()
def transcribe_video(video_path, model,
                     segment_sec=20.0,
                     use_video=True,
                     reference_text=None):
    """
    Транскрибирует видео и возвращает текст с таймкодами.

    Параметры:
        video_path     — путь к .mp4 / .avi / любому видео
        model          — обученная AVConformerCTC
        segment_sec    — длина сегментов для обработки (по умолчанию 20 секунд)
        use_video      — использовать видеокадры (если False — только аудио)
        reference_text — если передать эталонный текст, посчитает WER
    """
    model.eval()
    print(f"\nTranscribing: {video_path}")

    # Шаг 1: аудио
    print("  [1/3] Extracting audio...")
    audio_bytes = extract_audio_from_video(video_path)

    # Шаг 2: сегменты
    print("  [2/3] Splitting into segments...")
    segments = split_into_segments(audio_bytes, segment_sec=segment_sec)
    print(f"         {len(segments)} segments found")

    # Шаг 3: транскрибируем каждый сегмент
    print("  [3/3] Transcribing...")
    results = []

    for seg in segments:
        start = seg["start_sec"]
        end   = seg["end_sec"]

        # Аудио фичи
        features = audio_extractor(seg["bytes"])       # (T, 80)
        features = features.unsqueeze(0).to(DEVICE)    # (1, T, 80)

        # Видео кадры для этого отрезка
        video_tensor = None
        if use_video:
            frames = video_extractor.extract(video_path, start, end)
            if frames is not None:
                video_tensor = frames.to(DEVICE)

        # Прогон через модель
        log_probs = model(features, video_tensor)      # (1, T, vocab)
        pred_ids  = log_probs.argmax(dim=-1)[0]        # (T,)
        text      = tokenizer.decode_greedy(pred_ids.tolist()).strip()

        ts_start = seconds_to_timestamp(start)
        ts_end   = seconds_to_timestamp(end)

        if text:
            print(f"    [{ts_start} – {ts_end}]  {text}")
            results.append({
                "start":      start,
                "end":        end,
                "start_str":  ts_start,
                "end_str":    ts_end,
                "text":       text,
            })

    # Собираем итог
    full_text = " ".join(r["text"] for r in results)

    output = {
        "segments":  results,
        "full_text": full_text,
    }

    # WER если есть референс
    if reference_text:
        score = wer(reference_text, full_text)
        output["wer"] = score
        print(f"\n  WER = {score:.3f}  ({score*100:.1f}%)")

    # Красивый вывод итога
    print("\n" + "═"*60)
    print("РЕЗУЛЬТАТ:")
    print("─"*60)
    for r in results:
        print(f"[{r['start_str']} – {r['end_str']}]  {r['text']}")
    print("─"*60)
    print(f"ПОЛНЫЙ ТЕКСТ:\n{full_text}")
    print("═"*60)

    return output

# ============================================================
# 8. ЗАГРУЗКА ЧЕКПОИНТА
# ============================================================
def load_model(checkpoint_path="av_conformer_best.pt"):
    ckpt  = torch.load(checkpoint_path, map_location=DEVICE)
    model = AVConformerCTC(vocab_size=ckpt["vocab_size"]).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"Loaded checkpoint: epoch={ckpt['epoch']}, WER={ckpt['wer']:.3f}")
    return model



In [13]:
# ============================================================
# ЗАПУСК
# ============================================================
# if __name__ == "__main__":

    # --- ОБУЧЕНИЕ ---
model = train(
        n_epochs=3,
        batch_size=8,
        lr=3e-4,
        checkpoint_path="av_conformer_best.pt",
    )

    # --- ИНФЕРЕНС НА РЕАЛЬНОМ ВИДЕО ---
    # model = load_model("av_conformer_best.pt")  # ← если уже обучена



Loading dataset...


Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Model params: 13.2M
  [1/3] step    0/3750 | loss 9.6175 | 0s
  [1/3] step  100/3750 | loss 3.1564 | 17s
  [1/3] step  200/3750 | loss 6.2449 | 33s
  [1/3] step  300/3750 | loss 3.2440 | 50s
  [1/3] step  400/3750 | loss 2.9627 | 66s
  [1/3] step  500/3750 | loss 2.9154 | 83s
  [1/3] step  600/3750 | loss 2.7928 | 100s
  [1/3] step  700/3750 | loss 2.6338 | 117s
  [1/3] step  800/3750 | loss 2.3100 | 136s
  [1/3] step  900/3750 | loss 2.5938 | 154s
  [1/3] step 1000/3750 | loss 2.2939 | 171s
  [1/3] step 1100/3750 | loss 2.2602 | 189s
  [1/3] step 1200/3750 | loss 2.3350 | 205s
  [1/3] step 1300/3750 | loss 2.1969 | 217s
  [1/3] step 1400/3750 | loss 2.4425 | 229s
  [1/3] step 1500/3750 | loss 2.2757 | 245s
  [1/3] step 1600/3750 | loss 1.9095 | 263s
  [1/3] step 1700/3750 | loss 1.9225 | 280s
  [1/3] step 1800/3750 | loss 2.2192 | 298s
  [1/3] step 1900/3750 | loss 1.9088 | 316s
  [1/3] step 2000/3750 | loss 2.1636 | 335s
  [1/3] step 2100/3750 | loss 1.9862 | 353s
  [1/3] step 2200/3

In [ ]:


result = transcribe_video(
        video_path="videos/trushin.mp4",
        model=model,
        segment_sec=20.0,
        use_video=True,
        reference_text=None,      # передай строку для подсчёта WER
    )



Transcribing: videos/trushin.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
         13 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  ну вцелон модри это слочей когба есе потплоха чтустовоышьше тепи да можно вграносонети да пижеть сто вос о здравусо втоесяте на писно косенус а сколько вут стиградусаотв на
    [00:00:19 – 00:00:39]  на косе нус посмидистеградусяов на накосенусвскаки ста срокандранус так вот ну дальше чечуть приведеним припривести
    [00:00:38 – 00:00:58]  припривести ну на прмер сказаеь что бута этоэтовужу самаеш что синус деси тиг ранус да а ву то это тоже сама что мину косинус торакан дранабус
    [00:00:57 – 00:01:17]  градус так сабася нуот ри тебя получается уже боь что нинус всину сдесять
    [00:01:16 – 00:01:36]  ть накойсн ну тватцсять накойснусорак перопруть еслиьд знаешь  пормолу симустрайно мубла сказать что есль на зват всиму здестигралнуся зата кто симу стринцстиградуса проспладвать спа ормо н прйнол
    [00:01:35 

1) Feature extaction (audio+video)
2) **Sequence** modeling (audio+video): words, context
3) Multimodal fusion (суммируя/склеивая аудио и видео)
4) Decoding to text

- Gulati et al. (2020) – "Conformer: Convolution-augmented Transformer for Speech Recognition" (Google).
- Graves et al., 2006 – "Connectionist Temporal Classification" (CTC)

Компонент|	Обязательный элемент|	Ключевые статьи|
---------------|-----------------|--------------------|
Аудио-признаки|	Мел-спектрограмма|	Davis & Mermelstein, 1980; H. Purwins et al. (2019) — "Deep Learning for Audio Signal Processing" |
Видео-признаки|	Область рта + CNN|	Viola & Jones, 2001; Chung & Zisserman, 2016|
Аудиоэнкодер|	**Conformer или Transformer**|	Gulati et al., 2020; Vaswani et al., 2017|
Видеоэнкодер|	3D CNN / CNN+LSTM|	Tran et al., 2015; Stafylakis & Tzimiropoulos, 2017|
Слияние|	Кросс-модальное внимание|	Sterpu et al., 2018; Shi et al., 2022|
Декодирование|	CTC / Attention|	Graves et al., 2006; Watanabe et al., 2017|

написать в пт:
- прислать презу, структуру доклада
- почему такие старые статьи? нет ли более свежих:


рассказать по структуре в целом, что за чем последовательно.

Показать табличку с источниками, можно упомянуть мел-спектограмму как раз (впервые была предложена в 1980, в 2019 претерпела некоторые изменения - которую как раз сегодня и используют) - в частности статья по Conformer 2020-года года, 

(картинка с переменными по Conformer - последний информативный слайд): \*показываю на слайде\* 
первый этап - половина feed-forward network (ffn), просто рассматриваем принимаем данные, половину какого-то грубо говоря кадра, 
потом второй этап - механизм внимания Multi-Head Self-Attention (MHSA) для получения информации и связей с другими данными,
третий - свертка Convolution для получения локальной информации с другими данными,
и четвертый - опять половина ffn с нормализацией
(не факт, что всё буду говорить)


Структура сэндвича:

Представьте, что вы анализируете аудиозапись:

    FFN – это как увеличительное стекло, которое рассматривает каждый звук отдельно.
    Attention – это карта связей между звуками: вы понимаете, что предыдущее слово влияет на следующее.
    Convolution – это сканирование окрестностей каждого звука, чтобы уловить интонацию или акцент.

    Сэндвич – это когда вы сначала чуть-чуть увеличиваете (половинный FFN), потом строите карту связей (Attention), потом сканируете окрестности (Convolution), и снова увеличиваете (второй половинный FFN). И так несколько раз, чтобы постепенно строить всё более глубокое понимание речи.



алгоритм как это как понял, должно работать:


    Dataset ->
        Video ->
            Face detection and Lip localization
            Lip coordinates extraction ->
                Training Deep learning Model ->
                    --->

        Audio -> 
            Audio Features Extraction -> 
                Training Deep learning Model -> 
                    --->
                    
                    [COMMON] Final Lip Reading Model ->
                        (Measuring Performance)

In [1]:
# ============================================================
# ПОЛНЫЙ ФАЙЛ: av_train_infer.py
# Аудио + Видео (AVConformerCTC) → текст с таймкодами
# ============================================================

import os, io, cv2, subprocess, tempfile, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T

# import torchvision.transforms as VT

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from datasets import load_dataset

from datasets import load_dataset, Audio

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ============================================================
# 1. TOKENIZER
# ============================================================
class CharTokenizer:
    def __init__(self):
        chars = "абвгдеёжзийклмнопрстуфхцчшщъыьэюя "
        chars += "abcdefghijklmnopqrstuvwxyz0123456789.,!?-"
        self.vocab     = ["<blank>", "<unk>"] + list(chars)
        self.char2id   = {c: i for i, c in enumerate(self.vocab)}
        self.id2char   = {i: c for i, c in enumerate(self.vocab)}

    def encode(self, text):
        return [self.char2id.get(c, 1) for c in text.lower()]

    def decode_greedy(self, ids):
        result, prev = [], -1
        for i in ids:
            if i != prev and i != 0:
                result.append(self.id2char.get(i, ""))
            prev = i
        return "".join(result)

tokenizer  = CharTokenizer()
VOCAB_SIZE = len(tokenizer.vocab)

# ============================================================
# 2. FEATURE EXTRACTORS
# ============================================================
class LogMelFeatureExtractor:
    def __init__(self, sample_rate=16000, n_mels=80):
        self.target_sr = sample_rate
        self.transform = T.MelSpectrogram(
            sample_rate=sample_rate, n_mels=n_mels,
            win_length=400, hop_length=160, n_fft=512,
        )
        self.to_db = T.AmplitudeToDB(top_db=80)

    def __call__(self, audio_bytes):
        waveform, sr = torchaudio.load(io.BytesIO(audio_bytes))
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        mel = self.transform(waveform)
        log_mel = self.to_db(mel)
        log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-9)
        return log_mel.squeeze(0).T   # (T, 80)

# ============================================================
# FEATURE EXTRACTORS — VideoFrameExtractor без torchvision
# ============================================================
class VideoFrameExtractor:
    """Извлекает кадры через cv2 — без torchvision/torchcodec"""
    def __init__(self, target_fps=10, img_size=112):
        self.target_fps = target_fps
        self.img_size   = img_size
        # Нормализация ImageNet вручную
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def _preprocess(self, frame_bgr):
        """BGR numpy → нормализованный тензор (3, H, W)"""
        frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (self.img_size, self.img_size))
        frame = frame.astype(np.float32) / 255.0
        frame = (frame - self.mean) / self.std       # (H, W, 3)
        return torch.from_numpy(frame).permute(2, 0, 1)  # (3, H, W)

    def extract(self, video_path, start_sec, end_sec, max_frames=32):
        cap       = cv2.VideoCapture(video_path)
        video_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        step      = max(1, int(video_fps / self.target_fps))

        start_frame = int(start_sec * video_fps)
        end_frame   = int(end_sec   * video_fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        frames = []
        frame_idx = start_frame
        while frame_idx < end_frame:
            ret, frame = cap.read()
            if not ret:
                break
            if (frame_idx - start_frame) % step == 0:
                frames.append(self._preprocess(frame))
            frame_idx += 1
            if len(frames) >= max_frames:
                break

        cap.release()
        if not frames:
            return None
        return torch.stack(frames)

audio_extractor = LogMelFeatureExtractor()
video_extractor = VideoFrameExtractor()

import torch
import torch.nn as nn

class SpecAugment(nn.Module):
    """
        
    Регуляризация — модель учится восстанавливать речь при потере части информации.
    """
    def __init__(self,
                 time_mask_width=40,    # макс ширина маски по времени (в фреймах)
                 freq_mask_width=15,    # макс ширина маски по частоте (мел-фильтрам)
                 n_time_masks=2,        # сколько масок по времени
                 n_freq_masks=2):       # сколько масок по частоте
        super().__init__()
        self.time_mask_width = time_mask_width
        self.freq_mask_width = freq_mask_width
        self.n_time_masks    = n_time_masks
        self.n_freq_masks    = n_freq_masks
    
    def forward(self, x):
        """
        x: (B, T, F) — мел-спектрограмма
        """
        if not self.training:
            return x   # при инференсе ничего не делаем
        
        x = x.clone()
        B, T, F = x.shape
        
        for b in range(B):
            # Маски по времени
            for _ in range(self.n_time_masks):
                t = torch.randint(0, self.time_mask_width + 1, (1,)).item()
                if t == 0 or T - t <= 0:
                    continue
                t0 = torch.randint(0, T - t, (1,)).item()
                x[b, t0:t0+t, :] = 0
            
            # Маски по частоте
            for _ in range(self.n_freq_masks):
                f = torch.randint(0, self.freq_mask_width + 1, (1,)).item()
                if f == 0 or F - f <= 0:
                    continue
                f0 = torch.randint(0, F - f, (1,)).item()
                x[b, :, f0:f0+f] = 0
        
        return x

# ============================================================================================================

# ============================================================
# 4. МОДЕЛЬ — AVConformerCTC
# ============================================================
class ConvolutionModule(nn.Module):
    def __init__(self, d_model, kernel_size=31):
        super().__init__()
        self.norm     = nn.LayerNorm(d_model)
        self.pw_conv1 = nn.Conv1d(d_model, 2 * d_model, 1)
        self.dw_conv  = nn.Conv1d(d_model, d_model, kernel_size,
                                  padding=(kernel_size - 1) // 2, groups=d_model)
        self.bn       = nn.BatchNorm1d(d_model)
        self.pw_conv2 = nn.Conv1d(d_model, d_model, 1)
        self.dropout  = nn.Dropout(0.1)

    def forward(self, x):
        residual = x
        x = self.norm(x).transpose(1, 2)
        x = F.glu(self.pw_conv1(x), dim=1)
        x = F.silu(self.bn(self.dw_conv(x)))
        x = self.pw_conv2(x)
        return self.dropout(x).transpose(1, 2) + residual

class ConformerBlock(nn.Module):
    def __init__(self, d_model=256, n_heads=4, ff_dim=1024, kernel_size=31):
        super().__init__()
        self.ff1_norm  = nn.LayerNorm(d_model)
        self.ff1       = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.SiLU(), nn.Dropout(0.1),
            nn.Linear(ff_dim, d_model), nn.Dropout(0.1),
        )
        self.norm_attn = nn.LayerNorm(d_model)
        self.attn      = nn.MultiheadAttention(d_model, n_heads,
                                               dropout=0.1, batch_first=True)
        self.conv      = ConvolutionModule(d_model, kernel_size)
        self.ff2_norm  = nn.LayerNorm(d_model)
        self.ff2       = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.SiLU(), nn.Dropout(0.1),
            nn.Linear(ff_dim, d_model), nn.Dropout(0.1),
        )
        self.norm_out  = nn.LayerNorm(d_model)
        self.dropout   = nn.Dropout(0.1)

    def forward(self, x, key_padding_mask=None):
        x        = x + 0.5 * self.ff1(self.ff1_norm(x))
        residual = x
        x_norm   = self.norm_attn(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm,
                                key_padding_mask=key_padding_mask)
        x = residual + self.dropout(attn_out)
        x = self.conv(x)
        x = x + 0.5 * self.ff2(self.ff2_norm(x))
        return self.norm_out(x)

class ConvSubsampling(nn.Module):
    def __init__(self, n_mels=80, d_model=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, d_model, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(d_model, d_model, 3, stride=2, padding=1), nn.ReLU(),
        )
        self.proj = nn.Linear(d_model * (n_mels // 4), d_model)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        B, C, Tv, F_ = x.shape
        return self.proj(x.transpose(1, 2).reshape(B, Tv, C * F_))

class VideoEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3,   32,  3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32,  64,  3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64,  128, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.proj = nn.Linear(128 * 4 * 4, d_model)

    def forward(self, frames):          # (T_v, 3, 112, 112)
        x = self.cnn(frames)            # (T_v, 128, 4, 4)
        x = x.view(x.shape[0], -1)
        return self.proj(x)             # (T_v, d_model)

class AudioVisualFusion(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            d_model, num_heads=4, batch_first=True
        )
    
    def forward(self, audio_feat, video_feat):
        # audio_feat: (B, T_a, d)  video_feat: (B, T_v, d)
        # выравниваем видео по длине аудио
        video_feat = F.interpolate(
            video_feat.transpose(1, 2),
            size=audio_feat.shape[1],
            mode="linear", align_corners=False,
        ).transpose(1, 2)                          # (B, T_a, d)

        fused, _ = self.cross_attn(
            query=audio_feat,
            key=video_feat,
            value=video_feat,
        )
        return audio_feat + fused                  # residual

# ============================================================
# ИСПРАВЛЕННАЯ МОДЕЛЬ с Attention Decoder
# ============================================================
class AVConformerCTC(nn.Module):
    def __init__(self, vocab_size, n_mels=80, d_model=256,
                 n_layers=6, n_heads=4, n_decoder_layers=3):
        super().__init__()
        self.spec_augment = SpecAugment()
        self.subsampling = ConvSubsampling(n_mels, d_model)
        self.pos_enc     = nn.Embedding(5000, d_model)
        self.conformer   = nn.ModuleList([
            ConformerBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        self.video_enc   = VideoEncoder(d_model)
        self.fusion      = AudioVisualFusion(d_model)
        self.ctc_head    = nn.Linear(d_model, vocab_size)

        # Attention Decoder
        self.decoder_embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.decoder_pos   = nn.Embedding(1_000, d_model)   # позиции для декодера
        decoder_layer      = nn.TransformerDecoderLayer(
            d_model, n_heads,
            dim_feedforward=1024,
            dropout=0.1,
            batch_first=True,
        )
        self.decoder     = nn.TransformerDecoder(decoder_layer,
                                                  num_layers=n_decoder_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

        self._init_decoder_weights()

    def _encoder(self, audio, video_frames=None):
        """Общая часть: субдискретизация + Conformer + fusion"""

        # применяется только при обучении
        x = self.spec_augment(audio)
        
        x   = self.subsampling(x)
        pos = torch.arange(x.size(1), device=x.device)
        x   = x + self.pos_enc(pos)

        for block in self.conformer:
            x = block(x)

        if video_frames is not None:
            if video_frames.dim() == 4:
                v = self.video_enc(video_frames).unsqueeze(0)
            elif video_frames.dim() == 5:
                B, T_v, C, H, W = video_frames.shape
                v = self.video_enc(
                    video_frames.view(B * T_v, C, H, W)
                ).view(B, T_v, -1)
            else:
                v = None

            if v is not None:
                if v.size(0) == 1 and x.size(0) > 1:
                    v = v.expand(x.size(0), -1, -1)
                x = self.fusion(x, v)

        return x   # (B, T//4, d_model)

    def _init_decoder_weights(self):
        nn.init.normal_(self.decoder_embed.weight, std=0.02)
        nn.init.normal_(self.decoder_pos.weight,   std=0.02)
        nn.init.normal_(self.output_proj.weight,   std=0.02)
        nn.init.zeros_(self.output_proj.bias)

    def forward(self, audio, video_frames=None, target_tokens=None):
        """
        audio:         (B, T, 80)
        video_frames:  (T_v, 3, H, W) или (B, T_v, 3, H, W) или None
        target_tokens: (B, L) — сдвинутые вправо токены для teacher forcing
                       None   — только CTC (инференс или pretrain)
        """
        x = self._encoder(audio, video_frames)   # (B, T//4, d_model)

        # CTC выход — всегда
        ctc_logits = F.log_softmax(self.ctc_head(x), dim=-1)

        # print(f"  forward: target_tokens={target_tokens if target_tokens is None else target_tokens.shape}")

        # Attention выход — только при обучении
        attn_logits = None
        if target_tokens is not None:
            L   = target_tokens.size(1)
            pos = torch.arange(L, device=x.device)

            tgt_emb = (self.decoder_embed(target_tokens)
                       + self.decoder_pos(pos))   # (B, L, d_model)

            # Causal mask — декодер не видит будущие токены
            causal_mask = nn.Transformer.generate_square_subsequent_mask(
                L, device=x.device
            )

            # Padding mask — игнорируем <blank>=0 в таргете
            tgt_key_padding_mask = (target_tokens == 0)  # (B, L)
            tgt_key_padding_mask[:, 0] = False  # всегда разрешаем первый токен - не маскируем BOS

            attn_out    = self.decoder(
                tgt_emb, x,
                tgt_mask=causal_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
            )                                             # (B, L, d_model)
            attn_logits = self.output_proj(attn_out)      # (B, L, vocab_size)

        return ctc_logits, attn_logits



# ============================================================
# 5. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================
def wer(reference, hypothesis):
    ref = reference.lower().split()
    hyp = hypothesis.lower().split()
    d   = np.zeros((len(ref) + 1, len(hyp) + 1))
    for i in range(len(ref) + 1): d[i][0] = i
    for j in range(len(hyp) + 1): d[0][j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost   = 0 if ref[i-1] == hyp[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(ref)][len(hyp)] / max(len(ref), 1)

def seconds_to_timestamp(sec):
    """123.4 → '00:02:03'"""
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


Device: cuda


In [ ]:
# import os, io, cv2, time
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torchaudio
# import torchaudio.transforms as T

# # import torchvision.transforms as VT

# # from torch.utils.data import Dataset, DataLoader
# # from torch.optim import AdamW
# # from torch.optim.lr_scheduler import CosineAnnealingLR

# # from datasets import load_dataset, Audio

# # ============================================================
# # 2. FEATURE EXTRACTORS
# # ============================================================
# class LogMelFeatureExtractor:
#     def __init__(self, sample_rate=16000, n_mels=80):
#         self.target_sr = sample_rate
#         self.transform = T.MelSpectrogram(
#             sample_rate=sample_rate, n_mels=n_mels,
#             win_length=400, hop_length=160, n_fft=512,
#         )
#         self.to_db = T.AmplitudeToDB(top_db=80)

#     def __call__(self, audio_bytes):
#         waveform, sr = torchaudio.load(io.BytesIO(audio_bytes))
#         if sr != self.target_sr:
#             waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)
#         if waveform.shape[0] > 1:
#             waveform = waveform.mean(dim=0, keepdim=True)
#         mel = self.transform(waveform)
#         log_mel = self.to_db(mel)
#         log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-9)
#         return log_mel.squeeze(0).T   # (T, 80)

# # ============================================================
# # FEATURE EXTRACTORS — VideoFrameExtractor без torchvision
# # ============================================================
# class VideoFrameExtractor:
#     """Извлекает кадры через cv2 — без torchvision/torchcodec"""
#     def __init__(self, target_fps=10, img_size=112):
#         self.target_fps = target_fps
#         self.img_size   = img_size
#         # Нормализация ImageNet вручную
#         self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
#         self.std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

#     def _preprocess(self, frame_bgr):
#         """BGR numpy → нормализованный тензор (3, H, W)"""
#         frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
#         frame = cv2.resize(frame, (self.img_size, self.img_size))
#         frame = frame.astype(np.float32) / 255.0
#         frame = (frame - self.mean) / self.std       # (H, W, 3)
#         return torch.from_numpy(frame).permute(2, 0, 1)  # (3, H, W)

#     def extract(self, video_path, start_sec, end_sec, max_frames=64):
#         cap       = cv2.VideoCapture(video_path)
#         video_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
#         step      = max(1, int(video_fps / self.target_fps))

#         start_frame = int(start_sec * video_fps)
#         end_frame   = int(end_sec   * video_fps)
#         cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

#         frames = []
#         frame_idx = start_frame
#         while frame_idx < end_frame:
#             ret, frame = cap.read()
#             if not ret:
#                 break
#             if (frame_idx - start_frame) % step == 0:
#                 frames.append(self._preprocess(frame))
#             frame_idx += 1
#             if len(frames) >= max_frames:
#                 break

#         cap.release()
#         if not frames:
#             return None
#         return torch.stack(frames)



# # ============================================================
# # 7. PYTORCH DATASET ДЛЯ ОБУЧЕНИЯ
# # ============================================================
# class YouTubeAVDataset(torch.utils.data.Dataset):
#     """
#     Читает собранный датасет для обучения AVConformerCTC.
#     Возвращает: аудио-фичи, видео-кадры, транскрипцию.
#     """
#     def __init__(self, metadata_path: str, max_duration_sec=15.0):
#         with open(metadata_path, encoding="utf-8") as f:
#             self.records = json.load(f)

#         # Фильтруем по длине
#         self.records = [r for r in self.records
#                         if r["duration"] <= max_duration_sec]

#         self.audio_extractor = LogMelFeatureExtractor()
#         self.video_extractor = VideoFrameExtractor()
#         self.max_frames      = int(max_duration_sec * 100)
#         # print(f"YouTubeAVDataset: {len(self.records)} segments loaded")

#     def __len__(self):
#         return len(self.records)

#     def __getitem__(self, idx):
#         rec = self.records[idx]

#         # Аудио
#         with open(rec["audio_path"], "rb") as f:
#             audio_bytes = f.read()
#         features = self.audio_extractor(audio_bytes)
#         if features.shape[0] > self.max_frames:
#             features = features[:self.max_frames]

#         # Видео
#         frames = self.video_extractor.extract(
#             rec["video_path"], 0, rec["duration"]
#         )

#         # Текст
#         target = torch.tensor(
#             tokenizer.encode(rec["transcription"]), dtype=torch.long
#         )

#         return features, frames, target, rec["transcription"]

In [ ]:
# from urls import URLS

# # ============================================================
# # ЗАПУСК — собрать датасет
# # ============================================================
# if __name__ == "__main__":
#     # Список видео с хорошими русскими субтитрами
#     # Лучше брать видео с РУЧНЫМИ субтитрами (не авто)


#     # records = build_dataset(
#     #     urls          = URLS,
#     #     download_dir  = "./yt_downloads",
#     #     segments_dir  = "./yt_segments",
#     #     metadata_path = "./yt_dataset.json",
#     # )

#     # Проверяем датасет
    # ds = YouTubeAVDataset("./yt_dataset.json")
#     features, frames, target, text = ds[0]
#     print(f"Audio features: {features.shape}")   # (T, 80)
#     print(f"Video frames:   {frames.shape}")      # (T_v, 3, 112, 112)
#     print(f"Transcription:  {text}")

Audio features: torch.Size([307, 80])
Video frames:   torch.Size([46, 3, 112, 112])
Transcription:  всем привет с вами снова борис трушин и


In [13]:
target

tensor([ 4, 20,  7, 15, 35, 18, 19, 11,  4,  7, 21, 35, 20, 35,  4,  2, 15, 11,
        35, 20, 16, 17,  4,  2, 35,  3, 17, 19, 11, 20, 35, 21, 19, 22, 27, 11,
        16, 35, 11])

In [7]:
print(tokenizer.decode_greedy(target.tolist()))


всем привет с вами снова борис трушин и


161 min 

pretrain_sova

In [ ]:
import json
import os

# from torch.utils.data import DataLoader, coll

# ============================================================
# 3. DATASET — sova_rudevices (только аудио, видео синтезируем как заглушку)
# ============================================================
class SovaAVDataset(Dataset):
    """
    sova_rudevices не содержит видео — для обучения аудио-ветки
    передаём video=None. Когда будет видео-датасет, подключишь сюда.
    """
    def __init__(self, hf_dataset, max_duration_sec=15.0, max_samples=None):
        self.data = hf_dataset
        self.max_frames = int(max_duration_sec * 100)

        if max_samples and max_samples < len(self.data):
            self.data = self.data.select(range(max_samples))

        print(f"SovaAVDataset: {len(self.data)} samples")     
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample   = self.data[idx]
        features = audio_extractor(sample["audio"]["bytes"])
        if features.shape[0] > self.max_frames:
            features = features[:self.max_frames]
        target = torch.tensor(
            tokenizer.encode(sample["transcription"]), dtype=torch.long
        )
        # video=None — модель работает в audio-only режиме
        return features, None, target, sample["transcription"]

def collate_fn(batch):
    features, videos, targets, texts = zip(*batch)

    feat_lengths   = torch.tensor([f.shape[0] for f in features])
    target_lengths = torch.tensor([len(t) for t in targets])

    feat_padded = torch.zeros(len(features), feat_lengths.max(), features[0].shape[1])
    tgt_padded  = torch.zeros(len(targets), target_lengths.max(), dtype=torch.long)

    for i, (f, t) in enumerate(zip(features, targets)):
        feat_padded[i, :f.shape[0]] = f
        tgt_padded[i,  :len(t)]     = t

    # videos остаются None если датасет без видео
    return feat_padded, videos, tgt_padded, feat_lengths, target_lengths, texts



# ============================================================
# Youtube датасет
# ============================================================
class YouTubeAVDataset(torch.utils.data.Dataset):
    def __init__(self, metadata_path: str, language=None,
                 max_duration_sec=15.0):
        with open(metadata_path, encoding="utf-8") as f:
            self.records = json.load(f)

        # Фильтр по длине
        self.records = [
            r for r in self.records
            if len(r['transcription'].strip()) >= 3 
            and r["duration"] >= 0.5
            and r["duration"] <= max_duration_sec
            and os.path.exists(r["audio_path"])
            and os.path.exists(r["video_path"])
            ]

        # Фильтр по языку
        if language:
            self.records = [r for r in self.records
                            if r.get("language", "ru") == language]

        # Фильтр по существующим файлам — убираем битые записи
        self.records = [
            r for r in self.records
            if os.path.exists(r["audio_path"])
            and os.path.exists(r["video_path"])
        ]

        self.audio_extractor = LogMelFeatureExtractor()
        self.video_extractor = VideoFrameExtractor()
        self.max_frames      = int(max_duration_sec * 100)
        print(f"YouTubeAVDataset: {len(self.records)} segments "
              f"(language={language or 'all'})")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]

        # Аудио — с защитой от ошибок
        try:
            with open(rec["audio_path"], "rb") as f:
                audio_bytes = f.read()
            features = self.audio_extractor(audio_bytes)
            if features.shape[0] > self.max_frames:
                features = features[:self.max_frames]
        except Exception as e:
            print(f"  ⚠ Аудио ошибка [{idx}]: {e}")
            # Возвращаем пустой тензор — collate_fn отфильтрует
            features = torch.zeros(10, 80)

        # Видео — с защитой от ошибок
        frames = None
        # try:
        #     frames = self.video_extractor.extract(
        #         rec["video_path"], 0, rec["duration"]
        #     )
        # except Exception as e:
        #     print(f"  ⚠ Видео ошибка [{idx}]: {e}")
        #     frames = None

        # # Если видео не извлеклось — заглушка из нулей
        # if frames is None:
        #     frames = torch.zeros(1, 3, 112, 112)

        # Текст
        target = torch.tensor(
            tokenizer.encode(rec["transcription"]), dtype=torch.long
        )

        return features, frames, target, rec["transcription"]

In [ ]:
from datasets import load_dataset


# ============================================================
# ЭТАП 1 — Pretrain на sova_rudevices (только аудио)
# ============================================================
def pretrain_sova(n_epochs=10, batch_size=8, lr=3e-4,
                  checkpoint_path="conformer_sova.pt"):

    print("=" * 60)
    print("ЭТАП 1: Pretrain на sova_rudevices")
    print("=" * 60)

    dataset = load_dataset("bond005/sova_rudevices")
    dataset["train"] = dataset["train"].cast_column("audio", Audio(decode=False))
    dataset["test"]  = dataset["test"].cast_column("audio",  Audio(decode=False))

    train_loader = DataLoader(
        SovaAVDataset(dataset["train"], max_samples=20_000),
        batch_size=batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=0,
    )
    val_loader = DataLoader(
        SovaAVDataset(dataset["test"]),
        batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=0,
    )

    model = AVConformerCTC(vocab_size=VOCAB_SIZE).to(DEVICE)
    print(f"Параметров: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
    best_wer  = float("inf")

    for epoch in range(n_epochs):
        # ── TRAIN ──────────────────────────────────────────
        model.train()
        train_losses = []
        t0 = time.time()

        for step, (feats, videos, targets, feat_lens, tgt_lens, texts) in enumerate(train_loader):
            feats   = feats.to(DEVICE)
            targets = targets.to(DEVICE)

            log_probs     = model(feats, video_frames=None)
            input_lengths = (feat_lens // 4).clamp(max=log_probs.size(1))

            loss = F.ctc_loss(
                log_probs.transpose(0, 1),
                targets, input_lengths, tgt_lens,
                blank=0, zero_infinity=True,
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_losses.append(loss.item())

            if step % 100 == 0:
                print(f"  [{epoch+1}/{n_epochs}] step {step:4d}/{len(train_loader)}"
                      f" | loss {loss.item():.4f} | {time.time()-t0:.0f}s")

        # ── VALIDATION ─────────────────────────────────────
        model.eval()
        val_losses, wer_scores = [], []

        with torch.no_grad():
            for feats, videos, targets, feat_lens, tgt_lens, texts in val_loader:
                feats   = feats.to(DEVICE)
                targets = targets.to(DEVICE)

                log_probs     = model(feats, video_frames=None)
                input_lengths = (feat_lens // 4).clamp(max=log_probs.size(1))

                loss = F.ctc_loss(
                    log_probs.transpose(0, 1),
                    targets, input_lengths, tgt_lens,
                    blank=0, zero_infinity=True,
                )
                val_losses.append(loss.item())

                preds = log_probs.argmax(dim=-1).cpu()
                for i, pred_ids in enumerate(preds):
                    hyp = tokenizer.decode_greedy(pred_ids.tolist())
                    wer_scores.append(wer(texts[i], hyp))

        avg_train = np.mean(train_losses)
        avg_val   = np.mean(val_losses)
        avg_wer   = np.mean(wer_scores)
        scheduler.step()

        print(f"\n{'─'*60}")
        print(f"Epoch {epoch+1:2d}/{n_epochs} | "
              f"train={avg_train:.4f} | val={avg_val:.4f} | WER={avg_wer:.3f}")
        print(f"{'─'*60}\n")

        if avg_wer < best_wer:
            best_wer = avg_wer
            torch.save({
                "epoch": epoch + 1,
                "model_state": model.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "val_loss":    avg_val,
                "wer":         avg_wer,
                "vocab_size":  VOCAB_SIZE,
                "stage":       "pretrain_sova",
            }, checkpoint_path)
            print(f"  ✓ Сохранён → {checkpoint_path} (WER={avg_wer:.3f})\n")

    return model





# ============================================================
# ЭТАП 2 — Fine-tune на YouTube (аудио + видео)
# ============================================================
def finetune_youtube(checkpoint_path="conformer_sova.pt",
                     metadata_path="yt_dataset.json",
                     language="ru",
                     n_epochs=5,
                     batch_size=4,        # меньше батч — видео тяжелее
                     lr=1e-4,
                     freeze_encoder_epochs=2,
                     save_path="conformer_finetuned.pt",
                     hf_dataset_name=None,
                     hf_language="ru"):

    if hf_dataset_name:
        from datasets import load_dataset, Audio
        print(f"Loading {hf_dataset_name}...")
        hf_ds = load_dataset(hf_dataset_name, hf_language, 
                             trust_remote_code=True)
        hf_ds = hf_ds.cast_column("audio", Audio(decode=False))
        train_ds = SovaAVDataset(hf_ds["train"], max_samples=20_000)
        val_ds = SovaAVDataset(hf_ds["validation"], max_samples=5_000)
    else:

        print("=" * 60)
        print(f"ЭТАП 2: Fine-tune на YouTube ({language})")
        print("=" * 60)

        # Загружаем веса pretrain
        ckpt  = torch.load(checkpoint_path, map_location=DEVICE)
        model = AVConformerCTC(vocab_size=ckpt["vocab_size"]).to(DEVICE)
        model.load_state_dict(ckpt["model_state"])
        print(f"Загружен претрейн: epoch={ckpt['epoch']}, WER={ckpt['wer']:.3f}\n")

        # Датасет
        full_ds  = YouTubeAVDataset(metadata_path, language=language)
        val_size = max(1, int(len(full_ds) * 0.1))
        train_ds, val_ds = torch.utils.data.random_split(
            full_ds, [len(full_ds) - val_size, val_size]
        )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=collate_fn, 
        num_workers=0, # parallel
        pin_memory=True, # ускоряет при достаточной RAM
        # prefetch_factor=2 # загружать заранее
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=0,
    )
    print(f"Train: {len(train_ds)} | Val: {len(val_ds)}\n")

    optimizer = AdamW([
        {"params": model.subsampling.parameters(), "lr": lr * 0.1},
        {"params": model.conformer.parameters(),   "lr": lr * 0.1},
        {"params": model.pos_enc.parameters(),     "lr": lr * 0.1},
        {"params": model.video_enc.parameters(),   "lr": lr},
        {"params": model.fusion.parameters(),      "lr": lr},
        {"params": model.ctc_head.parameters(),    "lr": lr},
    ], weight_decay=1e-2)

    # optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
    best_wer  = float("inf")

    for epoch in range(n_epochs):

        # Freeze / Unfreeze энкодера
        if epoch < freeze_encoder_epochs:
            for name, param in model.named_parameters():
                param.requires_grad = "ctc_head" in name
            if epoch == 0:
                print("  🔒 Энкодер заморожен\n")
        else:
            for param in model.parameters():
                param.requires_grad = True
            if epoch == freeze_encoder_epochs:
                print("  🔓 Энкодер разморожен — end-to-end\n")

        # ── TRAIN ──────────────────────────────────────────
        model.train()
        train_losses = []
        t0 = time.time()

        for step, (feats, videos, targets, feat_lens, tgt_lens, texts) in enumerate(train_loader):
            feats   = feats.to(DEVICE)
            targets = targets.to(DEVICE)

            # Видео на GPU если есть
            video_tensor = None
            # if videos is not None:
            #     # print(videos[0].shape)  
            #     # video_tensor = videos.to(DEVICE) if videos is not None else None
            #     if videos is not None and not all(v is None for v in videos):
            #         # video_tensor = torch.stack([v for v in videos if v is not None]).to(DEVICE)
            #         valid_videos = [v for v in videos if v is not None]
            #         max_t = max(v.shape[0] for v in valid_videos)
            #         C, H, W = valid_videos[0].shape[1], valid_videos[0].shape[2], valid_videos[0].shape[3]

            #         video_tensor = torch.zeros(len(valid_videos), max_t, C, H, W)
            #         for i, v in enumerate(valid_videos):
            #             video_tensor[i, :v.shape[0]] = v
            #         video_tensor = video_tensor.to(DEVICE)

            #     else:
            #         video_tensor = None

            log_probs     = model(feats, video_tensor)
            input_lengths = (feat_lens // 4).clamp(max=log_probs.size(1))

            loss = F.ctc_loss(
                log_probs.transpose(0, 1),
                targets, input_lengths, tgt_lens,
                blank=0, zero_infinity=True,
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_losses.append(loss.item())

            if step % 50 == 0:
                print(f"  [{epoch+1}/{n_epochs}] step {step:3d}/{len(train_loader)}"
                      f" | loss {loss.item():.4f} | {time.time()-t0:.0f}s")

        # ── VALIDATION ─────────────────────────────────────
        model.eval()
        val_losses, wer_scores = [], []

        with torch.no_grad():
            for feats, videos, targets, feat_lens, tgt_lens, texts in val_loader:
                feats   = feats.to(DEVICE)
                targets = targets.to(DEVICE)

                log_probs     = model(feats, video_frames=None)
                input_lengths = (feat_lens // 4).clamp(max=log_probs.size(1))

                loss = F.ctc_loss(
                    log_probs.transpose(0, 1),
                    targets, input_lengths, tgt_lens,
                    blank=0, zero_infinity=True,
                )
                val_losses.append(loss.item())

                preds = log_probs.argmax(dim=-1).cpu()
                for i, pred_ids in enumerate(preds):
                    hyp = tokenizer.decode_greedy(pred_ids.tolist())

                    if i < 20:
                        print(f"REF: {texts[i]}")
                        print(f"HYP: {hyp}")
                        print(f"WER: {wer(texts[i], hyp):.3f}")
                        print()
                
                    wer_scores.append(wer(texts[i], hyp))

        avg_train = np.mean(train_losses)
        avg_val   = np.mean(val_losses)
        avg_wer   = np.mean(wer_scores)
        scheduler.step()

        print(f"\n{'─'*60}")
        print(f"Epoch {epoch+1:2d}/{n_epochs} | "
              f"train={avg_train:.4f} | val={avg_val:.4f} | WER={avg_wer:.3f}")
        print(f"{'─'*60}\n")

        if avg_wer < best_wer:
            best_wer = avg_wer
            torch.save({
                "epoch":       epoch + 1,
                "model_state": model.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "val_loss":    avg_val,
                "wer":         avg_wer,
                "vocab_size":  VOCAB_SIZE,
                "stage":       f"finetune_youtube_{language}",
            }, save_path)
            print(f"  ✓ Сохранён → {save_path} (WER={avg_wer:.3f})\n")

    return model


In [27]:
# ============================================================
# ЗАПУСК
# ============================================================
if __name__ == "__main__":

    # ── ЭТАП 1: Pretrain на sova ───────────────────────────
    # model = pretrain_sova(
    #     n_epochs=10,
    #     batch_size=8,
    #     lr=3e-4,
    #     checkpoint_path="conformer_sova_1.pt",
    # )
    ...
    # ── ЭТАП 2: Fine-tune на YouTube ──────────────────────
    # (раскомментируй после этапа 1)
    # model = finetune_youtube(
    #     checkpoint_path = "conformer_sova.pt",
    #     metadata_path   = "yt_dataset.json",
    #     language        = "ru",
    #     n_epochs        = 5,
    #     lr              = 1e-4,
    #     freeze_encoder_epochs = 2,
    #     save_path       = "conformer_youtube_ru.pt",
    # )

In [33]:
from huggingface_hub import list_datasets

# Ищем все датасеты mozilla
results = list(list_datasets(search="common_voice", author="mozilla-foundation"))
results

[DatasetInfo(id='mozilla-foundation/common_voice_17_0', author='mozilla-foundation', sha='11dc88355e899d1bf2df74f01b904a8544a17b33', created_at=datetime.datetime(2025, 10, 24, 17, 59, 42, tzinfo=datetime.timezone.utc), last_modified=datetime.datetime(2025, 10, 24, 18, 2, 20, tzinfo=datetime.timezone.utc), private=False, gated=False, disabled=False, downloads=7172, downloads_all_time=None, likes=13, paperswithcode_id=None, tags=['task_categories:automatic-speech-recognition', 'region:us'], trending_score=1, card_data=None, siblings=None, xet_enabled=None),
 DatasetInfo(id='mozilla-foundation/common_voice_13_0', author='mozilla-foundation', sha='ff2bbb54dcdb597100fe534a1b911ff9103f9e22', created_at=datetime.datetime(2025, 10, 24, 18, 4, 31, tzinfo=datetime.timezone.utc), last_modified=datetime.datetime(2025, 10, 24, 18, 5, 15, tzinfo=datetime.timezone.utc), private=False, gated=False, disabled=False, downloads=2050, downloads_all_time=None, likes=3, paperswithcode_id=None, tags=['region:

YouTube данные: нормально не обучается, loss с обучением вверх-вниз без четкой динамики

In [34]:
#  ── ЭТАП 2: Fine-tune на YouTube ──────────────────────
#     (раскомментируй после этапа 1)
model = finetune_youtube(
        checkpoint_path = "conformer_sova_1.pt",
        metadata_path   =  None,
        hf_dataset_name = "mozilla-foundation/common_voice_17_0",
        language        = "ru",
        n_epochs        = 3,
        lr              = 1e-4,
        freeze_encoder_epochs = 2,
        save_path       = "conformer_finetuned.pt",
    )

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mozilla-foundation/common_voice_17_0' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading mozilla-foundation/common_voice_17_0...


README.md:   0%|          | 0.00/414 [00:00<?, ?B/s]

c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tim\.cache\huggingface\hub\datasets--mozilla-foundation--common_voice_17_0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


EmptyDatasetError: The directory at hf://datasets/mozilla-foundation/common_voice_17_0@11dc88355e899d1bf2df74f01b904a8544a17b33 doesn't contain any data files


============================================================
ЭТАП 2: Fine-tune на YouTube (ru)
============================================================
Загружен претрейн: epoch=8, WER=0.882

YouTubeAVDataset: 15101 segments (language=ru)
Train: 13591 | Val: 1510

  🔒 Энкодер заморожен

  [1/5] step   0/3398 | loss 5.0001 | 2s
  [1/5] step  50/3398 | loss 4.8648 | 48s
  [1/5] step 100/3398 | loss 1.3860 | 93s
  [1/5] step 150/3398 | loss 2.6778 | 139s
  [1/5] step 200/3398 | loss 3.9074 | 181s
  [1/5] step 250/3398 | loss 0.0000 | 221s
  [1/5] step 300/3398 | loss 2.5585 | 263s
  [1/5] step 350/3398 | loss 3.0142 | 304s
  [1/5] step 400/3398 | loss 1.3808 | 347s
  [1/5] step 450/3398 | loss 2.7283 | 389s
  [1/5] step 500/3398 | loss 1.9228 | 431s
  [1/5] step 550/3398 | loss 3.5824 | 473s
  [1/5] step 600/3398 | loss 2.9126 | 622s
  [1/5] step 650/3398 | loss 2.0639 | 662s
  [1/5] step 700/3398 | loss 3.8365 | 704s


какьто не правильно обучение здесь идёт

лосс туда сюда, и по загрузке вижу по большей части процессор загружается а видеокарта совсем немного


TODO: что делаем дальше архитектурно?

In [ ]:
model = finetune_youtube(
        checkpoint_path = "conformer_sova_1.pt",
        metadata_path   = "yt_dataset.json",
        language        = "en",
        n_epochs        = 5,
        lr              = 1e-4,
        freeze_encoder_epochs = 2,
        save_path       = "conformer_youtube_en.pt",
    )

(потом нужно будет по архитектуре алгоритма пройтись ещё раз)

### Проверка на качество

In [2]:
import os, io, cv2, subprocess, tempfile, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T

# ============================================================
# 7. ИНФЕРЕНС НА РЕАЛЬНОМ ВИДЕО
# ============================================================
def extract_audio_from_video(video_path, sample_rate=16000):
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_path = tmp.name
    subprocess.run([
        "ffmpeg", "-y", "-i", video_path,
        "-ar", str(sample_rate), "-ac", "1", "-f", "wav", tmp_path,
    ], check=True, capture_output=True)
    with open(tmp_path, "rb") as f:
        audio_bytes = f.read()
    os.unlink(tmp_path)
    return audio_bytes

def split_into_segments(audio_bytes, segment_sec=20.0,
                        overlap_sec=1.0, sample_rate=16000):
    waveform, sr = torchaudio.load(io.BytesIO(audio_bytes))
    if sr != sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, sample_rate)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    seg_samples  = int(segment_sec  * sample_rate)
    step_samples = int((segment_sec - overlap_sec) * sample_rate)
    total        = waveform.shape[1]
    segments     = []
    start        = 0

    while start < total:
        end   = min(start + seg_samples, total)
        chunk = waveform[:, start:end]
        rms   = chunk.pow(2).mean().sqrt().item()
        if rms > 0.001:                            # пропускаем тишину
            buf = io.BytesIO()
            torchaudio.save(buf, chunk, sample_rate, format="wav")
            segments.append({
                "bytes":     buf.getvalue(),
                "start_sec": start / sample_rate,
                "end_sec":   end   / sample_rate,
            })
        start += step_samples
        if end == total:
            break

    return segments

@torch.no_grad()
def transcribe_video(video_path, model,
                     segment_sec=20.0,
                     use_video=True,
                     reference_text=None):
    """
    Транскрибирует видео и возвращает текст с таймкодами.

    Параметры:
        video_path     — путь к .mp4 / .avi / любому видео
        model          — обученная AVConformerCTC
        segment_sec    — длина сегментов для обработки (по умолчанию 20 секунд)
        use_video      — использовать видеокадры (если False — только аудио)
        reference_text — если передать эталонный текст, посчитает WER
    """
    model.eval()
    print(f"\nTranscribing: {video_path}")

    # Шаг 1: аудио
    print("  [1/3] Extracting audio...")
    audio_bytes = extract_audio_from_video(video_path)

    # Шаг 2: сегменты
    print("  [2/3] Splitting into segments...")
    segments = split_into_segments(audio_bytes, segment_sec=segment_sec)
    print(f"         {len(segments)} segments found")

    # Шаг 3: транскрибируем каждый сегмент
    print("  [3/3] Transcribing...")
    results = []

    for seg in segments:
        start = seg["start_sec"]
        end   = seg["end_sec"]

        # Аудио фичи
        features = audio_extractor(seg["bytes"])       # (T, 80)
        features = features.unsqueeze(0).to(DEVICE)    # (1, T, 80)

        # Видео кадры для этого отрезка
        video_tensor = None
        if use_video:
            frames = video_extractor.extract(video_path, start, end)
            if frames is not None:
                video_tensor = frames.to(DEVICE)

        # Прогон через модель
        log_probs = model(features, video_tensor)      # (1, T, vocab)
        pred_ids  = log_probs.argmax(dim=-1)[0]        # (T,)
        text      = tokenizer.decode_greedy(pred_ids.tolist()).strip()

        ts_start = seconds_to_timestamp(start)
        ts_end   = seconds_to_timestamp(end)

        if text:
            print(f"    [{ts_start} – {ts_end}]  {text}")
            results.append({
                "start":      start,
                "end":        end,
                "start_str":  ts_start,
                "end_str":    ts_end,
                "text":       text,
            })

    # Собираем итог
    full_text = " ".join(r["text"] for r in results)

    output = {
        "segments":  results,
        "full_text": full_text,
    }

    # WER если есть референс
    if reference_text:
        score = wer(reference_text, full_text)
        output["wer"] = score
        print(f"\n  WER = {score:.3f}  ({score*100:.1f}%)")

    # Красивый вывод итога
    print("\n" + "═"*60)
    print("РЕЗУЛЬТАТ:")
    print("─"*60)
    for r in results:
        print(f"[{r['start_str']} – {r['end_str']}]  {r['text']}")
    print("─"*60)
    print(f"ПОЛНЫЙ ТЕКСТ:\n{full_text}")
    print("═"*60)

    return output

# ============================================================
# 8. ЗАГРУЗКА ЧЕКПОИНТА
# ============================================================
def load_model(checkpoint_path="av_conformer_best.pt"):
    ckpt  = torch.load(checkpoint_path, map_location=DEVICE)
    model = AVConformerCTC(vocab_size=ckpt["vocab_size"]).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"Loaded checkpoint: epoch={ckpt['epoch']}, WER={ckpt['wer']:.3f}")
    return model




In [ ]:

result = transcribe_video(
        video_path="videos/trushin.mp4",
        model=model,
        segment_sec=20.0,
        use_video=True,
        reference_text=None,      # передай строку для подсчёта WER
    )


Transcribing: videos/trushin.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
         13 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  ну вцелон мотри это слоченй когда еслинтеплоха чуствоешь депи да можно вгранусо вытида пижеть стогось ь здратусо в оесите написнно косеус на сколько двудцатиградусо на
    [00:00:19 – 00:00:39]  на ковсенус восмидестиградусов на накосенс кскаких ста соракадрадс так вот ну и дальшо чиечуть привидениеими припривести
    [00:00:38 – 00:00:58]  припривести но на примерт сказаюь что быто этоэтогоже самаеш что синус десити граус да а вт это тоже сама что минус ко сенус зрака грабусв
    [00:00:57 – 00:01:17]  градусв таксагасявот им тебя получается уже бот что минус всин сздесять
    [00:01:16 – 00:01:36]  т накось на стдватьсять накос му ссоракак первыю пудть есль д знаешь   орву лу се му стронол вубла сказать что есл назвать всину здестегралисят зат кто сену стрицатиг радуся просладваетцсяпа порбол тронол
    [00:01:3

In [ ]:
from datasets import load_dataset

ds = load_dataset("bond005/openstt_public_speech")


DatasetNotFoundError: Dataset 'bond005/openstt_public_speech' doesn't exist on the Hub or cannot be accessed.

In [2]:
# Проверим что реально есть от bond005
from huggingface_hub import list_datasets

results = list(list_datasets(author="bond005"))
for r in results:
    print(r.id)

bond005/ru_llm_calibration
bond005/sberdevices_golos_100h_farfield
bond005/sberdevices_golos_10h_crowd
bond005/sova_rudevices
bond005/rulibrispeech
bond005/ru_texts_normalized
bond005/sberdevices_golos_10h_crowd_noised_2db
bond005/taiga_speech
bond005/podlodka_speech
bond005/taiga_speech_v2
bond005/audioset-nonspeech
bond005/NEREL_instruct
bond005/NEREL_bench


In [5]:
%pip install hf_xet --verbose

^C
Note: you may need to restart the kernel to use updated packages.


Using pip 25.1.1 from c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\pip (python 3.10)
  Obtaining dependency information for hf_xet from https://files.pythonhosted.org/packages/8a/21/75a6c175b4e79662ad8e62f46a40ce341d8d6b206b06b4320d07d55b188c/hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datasets import load_dataset

# ds = load_dataset("SberDevices/Golos", split="train") EmptyDatasetError

ds = load_dataset("bond005/sberdevices_golos_100h_farfield")


In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 9570
    })
    validation: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 933
    })
    test: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 1916
    })
})

#### Common Voice 17_0 обучение

In [6]:
import pandas as pd
# from datasets import Dataset
from torch.utils.data import Dataset, DataLoader

import os

class CommonVoiceLocalDataset(Dataset):
    def __init__(self, tsv_path, clips_dir, 
                 max_duration_sec=15.0, max_samples=None,
                 min_upvotes=1, shuffle=True, seed=42):
        """
        tsv_path  — путь к validated.tsv или train.tsv
        clips_dir — папка с .mp3 файлами (обычно рядом с tsv)
        """
        df = pd.read_csv(tsv_path, sep="\t")
        # Фильтруем по качеству — берём только с upvotes
        if "up_votes" in df.columns:
            df = df[df["up_votes"] >= min_upvotes]
        
        # Убираем пустые транскрипции
        df = df[df["sentence"].notna() & (df["sentence"].str.strip() != "")]
        
        # Фильтр по длине текста
        df = df[df["sentence"].str.split().str.len().between(2, 20)]
        
        # Оставляем нужные колонки
        self.records = df[["path", "sentence"]].reset_index(drop=True)
        
        if shuffle:
            self.records = self.records.sample(max_samples, random_state=seed)
            
        if max_samples and max_samples < len(self.records):
            self.records = self.records.reset_index(drop=True)
        
        self.clips_dir  = clips_dir
        self.max_frames = int(max_duration_sec * 100)
        print(f"CommonVoiceLocalDataset: {len(self.records)} segments")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row  = self.records.iloc[idx]
        path = os.path.join(self.clips_dir, row["path"])

        # Читаем аудио
        try:
            with open(path, "rb") as f:
                audio_bytes = f.read()
            features = audio_extractor(audio_bytes)
            if features.shape[0] > self.max_frames:
                features = features[:self.max_frames]
        except Exception as e:
            features = torch.zeros(10, 80)

        target = torch.tensor(
            tokenizer.encode(row["sentence"].lower().strip()),
            dtype=torch.long
        )
        return features, None, target, row["sentence"].lower().strip()
    
import torch

def collate_fn(batch):
    features, videos, targets, texts = zip(*batch)

    feat_lengths   = torch.tensor([f.shape[0] for f in features])
    target_lengths = torch.tensor([len(t) for t in targets])

    feat_padded = torch.zeros(len(features), feat_lengths.max(), features[0].shape[1])
    tgt_padded  = torch.zeros(len(targets), target_lengths.max(), dtype=torch.long)

    for i, (f, t) in enumerate(zip(features, targets)):
        feat_padded[i, :f.shape[0]] = f
        tgt_padded[i,  :len(t)]     = t

    # videos — None для аудио датасетов
    video_padded = None
    valid_videos = [v for v in videos if v is not None]


    if len(valid_videos) > 0:
        vid_lengths  = [v.shape[0] for v in valid_videos]
        max_v        = max(vid_lengths)
        _, C, H, W   = valid_videos[0].shape
        video_padded = torch.zeros(len(videos), max_v, C, H, W)
        valid_idx = [i for i, v in enumerate(videos) if v is not None]
        for i, v in zip(valid_idx, valid_videos):
            video_padded[i, :v.shape[0]] = v

    return feat_padded, video_padded, tgt_padded, feat_lengths, target_lengths, texts


In [ ]:
import torch

# from avsr_conformer.utils.utils import wer, DEVICE 
# from avsr_conformer.utils.tokenizer import tokenizer 
# from avsr_conformer.models.model_implementation import AVConformerCTC
# # from 

# from torch.optim import AdamW
# from torch.optim.lr_scheduler import CosineAnnealingLR

# import os, io, cv2, subprocess, tempfile, time
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torchaudio
# import torchaudio.transforms as T



def collate_fn(batch):
    features, videos, targets, texts = zip(*batch)

    feat_lengths   = torch.tensor([f.shape[0] for f in features])
    target_lengths = torch.tensor([len(t) for t in targets])

    feat_padded = torch.zeros(len(features), feat_lengths.max(), features[0].shape[1])
    tgt_padded  = torch.zeros(len(targets), target_lengths.max(), dtype=torch.long)

    for i, (f, t) in enumerate(zip(features, targets)):
        feat_padded[i, :f.shape[0]] = f
        tgt_padded[i,  :len(t)]     = t

    # videos — None для аудио датасетов
    video_padded = None
    valid_videos = [v for v in videos if v is not None]


    if len(valid_videos) > 0:
        vid_lengths  = [v.shape[0] for v in valid_videos]
        max_v        = max(vid_lengths)
        _, C, H, W   = valid_videos[0].shape
        video_padded = torch.zeros(len(videos), max_v, C, H, W)
        valid_idx = [i for i, v in enumerate(videos) if v is not None]
        for i, v in zip(valid_idx, valid_videos):
            video_padded[i, :v.shape[0]] = v

    return feat_padded, video_padded, tgt_padded, feat_lengths, target_lengths, texts


# def finetune_on_loaders(train_loader, val_loader,
#                         checkpoint_path="/conformer_sova_1.pt",
#                         n_epochs=5,
#                         lr=1e-4,
#                         freeze_encoder_epochs=1,
#                         save_path="conformer_finetuned.pt"):

#     print("=" * 60)
#     print(f"Fine-tune: {n_epochs} эпох, lr={lr}")
#     print(f"Checkpoint: {checkpoint_path}")
#     print("=" * 60)

#     # Загружаем pretrain веса
#     ckpt  = torch.load(checkpoint_path, map_location=DEVICE)
#     model = AVConformerCTC(vocab_size=ckpt["vocab_size"]).to(DEVICE)
#     model.load_state_dict(ckpt["model_state"])
#     print(f"Загружен: epoch={ckpt['epoch']}, WER={ckpt['wer']:.3f}\n")

#     # Discriminative lr — энкодер обновляется медленнее
#     optimizer = AdamW([
#         {"params": model.subsampling.parameters(), "lr": lr * 0.1},
#         {"params": model.conformer.parameters(),   "lr": lr * 0.1},
#         {"params": model.pos_enc.parameters(),     "lr": lr * 0.1},
#         {"params": model.video_enc.parameters(),   "lr": lr},
#         {"params": model.fusion.parameters(),      "lr": lr},
#         {"params": model.ctc_head.parameters(),    "lr": lr},
#     ], weight_decay=1e-2)

#     scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
#     best_wer  = float("inf")

#     for epoch in range(n_epochs):

#         # Freeze / Unfreeze
#         if epoch < freeze_encoder_epochs:
#             for name, param in model.named_parameters():
#                 param.requires_grad = ("ctc_head" in name or
#                                        "video_enc" in name or
#                                        "fusion"    in name)
#             if epoch == 0:
#                 trainable = sum(p.numel() for p in model.parameters()
#                                 if p.requires_grad) / 1e6
#                 print(f"  🔒 Энкодер заморожен | обучаемых: {trainable:.1f}M\n")
#         else:
#             for param in model.parameters():
#                 param.requires_grad = True
#             if epoch == freeze_encoder_epochs:
#                 total = sum(p.numel() for p in model.parameters()) / 1e6
#                 print(f"  🔓 Энкодер разморожен | всего: {total:.1f}M\n")

#         # ── TRAIN ──────────────────────────────────────────────
#         model.train()
#         train_losses = []
#         t0 = time.time()

#         for step, (feats, videos, targets, feat_lens, tgt_lens, texts) in enumerate(train_loader):
#             feats   = feats.to(DEVICE)
#             targets = targets.to(DEVICE)

#             # Видео не используем на этом этапе
#             log_probs     = model(feats, video_frames=None)
#             input_lengths = (feat_lens // 4).clamp(max=log_probs.size(1))

#             loss = F.ctc_loss(
#                 log_probs.transpose(0, 1),
#                 targets, input_lengths, tgt_lens,
#                 blank=0, zero_infinity=True,
#             )

#             if torch.isnan(loss) or torch.isinf(loss):
#                 continue  # пропускаем битые батчи

#             optimizer.zero_grad()
#             loss.backward()
#             torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
#             optimizer.step()
#             train_losses.append(loss.item())

#             if step % 100 == 0:
#                 avg = np.mean(train_losses[-100:]) if train_losses else 0
#                 print(f"  [{epoch+1}/{n_epochs}] step {step:4d}/{len(train_loader)}"
#                       f" | loss {loss.item():.4f} | avg {avg:.4f}"
#                       f" | {time.time()-t0:.0f}s")

#         # ── VALIDATION ─────────────────────────────────────────
#         model.eval()
#         val_losses, wer_scores = [], []
#         sample_printed = 0   # печатаем несколько примеров для диагностики

#         with torch.no_grad():
#             for feats, videos, targets, feat_lens, tgt_lens, texts in val_loader:
#                 feats   = feats.to(DEVICE)
#                 targets = targets.to(DEVICE)

#                 log_probs     = model(feats, video_frames=None)
#                 input_lengths = (feat_lens // 4).clamp(max=log_probs.size(1))

#                 loss = F.ctc_loss(
#                     log_probs.transpose(0, 1),
#                     targets, input_lengths, tgt_lens,
#                     blank=0, zero_infinity=True,
#                 )
#                 if not (torch.isnan(loss) or torch.isinf(loss)):
#                     val_losses.append(loss.item())

#                 preds = log_probs.argmax(dim=-1).cpu()
#                 for i, pred_ids in enumerate(preds):
#                     hyp = tokenizer.decode_greedy(pred_ids.tolist())
#                     ref = texts[i]
#                     wer_scores.append(wer(ref, hyp))

#                     # Печатаем первые 3 примера для диагностики
#                     if sample_printed < 3:
#                         print(f"  REF: {ref}")
#                         print(f"  HYP: {hyp}")
#                         print(f"  WER: {wer(ref, hyp):.3f}\n")
#                         sample_printed += 1

#         avg_train = np.mean(train_losses) if train_losses else float("inf")
#         avg_val   = np.mean(val_losses)   if val_losses   else float("inf")
#         avg_wer   = np.mean(wer_scores)   if wer_scores   else float("inf")
#         scheduler.step()

#         print(f"\n{'─'*60}")
#         print(f"Epoch {epoch+1:2d}/{n_epochs} | "
#               f"train={avg_train:.4f} | val={avg_val:.4f} | WER={avg_wer:.3f}")
#         print(f"{'─'*60}\n")

#         if avg_wer < best_wer:
#             best_wer = avg_wer
#             torch.save({
#                 "epoch":       epoch + 1,
#                 "model_state": model.state_dict(),
#                 "optimizer":   optimizer.state_dict(),
#                 "val_loss":    avg_val,
#                 "wer":         avg_wer,
#                 "vocab_size":  VOCAB_SIZE,
#                 "stage":       "finetune_common_voice",
#             }, save_path)
#             print(f"  ✓ Сохранён → {save_path} (WER={avg_wer:.3f})\n")

#     return model

In [36]:
# Укажи свои пути
TRAIN_DATA_DIR = "train_data/cv-corpus-25.0-2026-03-09"
# TSV_PATH  = "./ru/validated.tsv"       # или train.tsv
CLIPS_DIR = "./ru/clips"               # папка с mp3

# Проверка
ds_cv = CommonVoiceLocalDataset(
    tsv_path  = TRAIN_DATA_DIR + "/ru/train.tsv",
    clips_dir = TRAIN_DATA_DIR + CLIPS_DIR,
    max_samples = 5,
)


CommonVoiceLocalDataset: 5 segments


In [18]:
featurs, _, target, text = ds_cv[0]
print(f"Text:     {text}")
print(f"Features: {featurs.shape}")
print(f"Target:   {target}")

Text:     мьянма тесно сотрудничает с советом по правам человека организации объединенных наций.
Features: torch.Size([634, 80])
Target:   tensor([15, 31, 34, 16, 15,  2, 35, 21,  7, 20, 16, 17, 35, 20, 17, 21, 19, 22,
         6, 16, 11, 26,  2,  7, 21, 35, 20, 35, 20, 17,  4,  7, 21, 17, 15, 35,
        18, 17, 35, 18, 19,  2,  4,  2, 15, 35, 26,  7, 14, 17,  4,  7, 13,  2,
        35, 17, 19,  5,  2, 16, 11, 10,  2, 25, 11, 11, 35, 17,  3, 29,  7,  6,
        11, 16,  7, 16, 16, 30, 24, 35, 16,  2, 25, 11, 12, 72])


In [14]:
TRAIN_DATA_DIR = "train_data/cv-corpus-25.0-2026-03-09"
TSV_PATH  = "/ru/validated.tsv"       # или train.tsv
CLIPS_DIR = "/ru/clips"               

ds_cv_train = CommonVoiceLocalDataset(
    tsv_path   = TRAIN_DATA_DIR + "/ru/train.tsv",
    clips_dir  = TRAIN_DATA_DIR+CLIPS_DIR,
    max_samples = 26_900,
)
ds_cv_val = CommonVoiceLocalDataset(
    tsv_path   = TRAIN_DATA_DIR + "/ru/dev.tsv",
    clips_dir  = TRAIN_DATA_DIR+CLIPS_DIR,
    max_samples = 2_000,
)

train_loader = DataLoader(ds_cv_train, batch_size=8, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(ds_cv_val,   batch_size=8, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)

# И дообучаем поверх sova
model = finetune_on_loaders(
    train_loader    = train_loader,
    val_loader      = val_loader,
    checkpoint_path = "conformer_sova_1.pt",
    n_epochs        = 5,
    save_path       = "conformer_cv_ru.pt",
)

супер 1
супер 2
супер 3
CommonVoiceLocalDataset: 26611 segments
супер 1
супер 2
супер 3
CommonVoiceLocalDataset: 2000 segments
Fine-tune: 5 эпох, lr=0.0001
Checkpoint: conformer_sova_1.pt
Загружен: epoch=8, WER=0.882

  🔒 Энкодер заморожен | обучаемых: 0.9M

  [1/5] step    0/3327 | loss 2.2032 | avg 2.2032 | 2s
  [1/5] step  100/3327 | loss 2.0269 | avg 1.9012 | 26s
  [1/5] step  200/3327 | loss 1.6255 | avg 1.8439 | 51s
  [1/5] step  300/3327 | loss 1.9075 | avg 1.7606 | 76s
  [1/5] step  400/3327 | loss 1.6241 | avg 1.7042 | 100s
  [1/5] step  500/3327 | loss 1.6742 | avg 1.6552 | 124s
  [1/5] step  600/3327 | loss 1.6240 | avg 1.5581 | 148s
  [1/5] step  700/3327 | loss 1.2894 | avg 1.5248 | 173s
  [1/5] step  800/3327 | loss 1.3371 | avg 1.4892 | 197s
  [1/5] step  900/3327 | loss 1.6100 | avg 1.4501 | 221s
  [1/5] step 1000/3327 | loss 1.1348 | avg 1.4114 | 245s
  [1/5] step 1100/3327 | loss 1.3681 | avg 1.3600 | 269s
  [1/5] step 1200/3327 | loss 1.7638 | avg 1.3522 | 293s
  [1/

Проверка на качество

In [2]:
from evaluate_performance import load_model, transcribe_video

In [7]:
model = load_model("avsr_conformer\checkpoints\conformer_cv_ru_v4epoch10.pt")

result = transcribe_video(
        video_path="videos/trushin.mp4",
        model=model,
        segment_sec=20.0,
        use_video=True,
        reference_text=None,      # передай строку для подсчёта WER
    )

Loaded checkpoint: epoch=10, WER=0.814

Transcribing: videos/trushin.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
         13 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  ну в целом мори это случай когда еслилпеплохо, чувствуе ша депи да можно, вграду совуйтида пижет свовосьми з градусовоесяте написнно, косенусна, сколько дуддстиградусов на.
    [00:00:19 – 00:00:39]  на, косенус восмидест и градусов на накосенус с таких ста сракандранстак вот, но и дальше чущуть приведениеми припривести
    [00:00:38 – 00:00:58]  приперивести, но на примерт, сказать, чтобыто этот оже саммое что синус, десяти градос на, а вт это тоже сама, что минус косенус срака градусв.
    [00:00:57 – 00:01:17]  градусв так, согласонвот и тебя получается уже вот что минус  всиму сздесять.
    [00:01:16 – 00:01:36]  накойсь ну здвуадциать, накойсьснусорокк первы путь есль знаешь а орву усимустронул угла, <unk> сказать, что, естлел назвать, сину здестиградусов заты, пто симу 

In [5]:
model = load_model("avsr_conformer\checkpoints\conformer_cv_ru_v4.pt")

result = transcribe_video(
        video_path="videos/trushin.mp4",
        model=model,
        segment_sec=20.0,
        use_video=True,
        reference_text=None,      # передай строку для подсчёта WER
    )

Loaded checkpoint: epoch=5, WER=0.798

Transcribing: videos/trushin.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
         13 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  ну вцелон мори это случей когда еслилпеплохо, чувствуе ша депи да можно, вграду совуйтида пижетсвовосьмис градусовтоесяте написно косерусна, сколько двустиградусов на.
    [00:00:19 – 00:00:39]  на, косенус восмидест и градусов на, накосенус с каких ста сракандранстак вот, но и дальше чащуть приведениеми припривести
    [00:00:38 – 00:00:58]  приперивести, но на примерт, сказать, что быуто этот воже саммое, что синус, десяти и градос на, а вт этотоже сама, что минускосенус срака градусв.
    [00:00:57 – 00:01:17]  градусвтак согласонеот и у тебя получается, уже вот что минус  всиму сздесять.
    [00:01:16 – 00:01:36]  накойсь на вздвуадциать, накойснусорокк первы путь есль знаешь а орвул усимустронол угла, <unk> сказать, что, естлел назвать, син здестиградуся заты, пто симу ст

In [8]:
model = load_model("avsr_conformer\checkpoints\conformer_sova_1.pt")

result = transcribe_video(
        video_path="videos/trushin.mp4",
        model=model,
        segment_sec=20.0,
        use_video=True,
        reference_text=None,      # передай строку для подсчёта WER
    )

Loaded checkpoint: epoch=8, WER=0.882

Transcribing: videos/trushin.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
         13 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  ну вцелон мотри это слоченй когда еслинтеплоха чуствоешь депи да можно вгранусо вытида пижеть стогось ь здратусо в оесите написнно косеус на сколько двудцатиградусо на
    [00:00:19 – 00:00:39]  на ковсенус восмидестиградусов на накосенс кскаких ста соракадрадс так вот ну и дальшо чиечуть привидениеими припривести
    [00:00:38 – 00:00:58]  припривести но на примерт сказаюь что быто этоэтогоже самаеш что синус десити граус да а вт это тоже сама что минус ко сенус зрака грабусв
    [00:00:57 – 00:01:17]  градусв таксагасявот им тебя получается уже бот что минус всин сздесять
    [00:01:16 – 00:01:36]  т накось на стдватьсять накос му ссоракак первыю пудть есль д знаешь   орву лу се му стронол вубла сказать что есл назвать всину здестегралисят зат кто сену стрицатиг радуся просл

Видеофайл
    ↓
[1. Извлечение аудиодорожки] — ffmpeg, 16 кГц, моно
    ↓
[2. Сегментация по тишине]  — детекция по уровню RMS
    ↓
[3. Извлечение признаков]   — логарифмическая мел-спектрограмма (80 × T)
    ↓
[4. Акустическое моделирование] — Conformer Encoder (6 блоков, d=256)
    ↓                                    ↑
[4а. Визуальное моделирование]  — VideoEncoder (CNN) + Cross-Attention Fusion
    ↓
[5. Декодирование]          — CTC (жадное) / Attention (beam search)
    ↓
[6. Постобработка]          — сборка сегментов, таймкоды ЧЧ:ММ:СС
    ↓
Текст с таймкодами

                   ВХОД
         ┌──────────────────────────┐
         │   Видеокадры             │   Аудио (16 кГц)
         │   (T_v, 3, 112, 112)     │   (T сэмплов)
         └──────────┬───────────────┘
                    │                        │
                    ▼                        ▼
           [VideoEncoder]          [Лог мел-спектрограмма]
           Conv2D × 3                  (T × 80)
           AdaptivePool                    │
           Linear → 256                   ▼
                    │              [ConvSubsampling]
                    │              Conv2D × 2, stride=2
                    │              Linear → 256
                    │                  (T//4 × 256)
                    │                      │
                    │              [Позиционное кодирование]
                    │                      │
                    │              ┌───────────────┐
                    │              │ ConformerBlock │ × 6
                    │              │               │
                    │              │  ½ FFN        │
                    │              │  Self-Attn    │
                    │              │  Conv Module  │
                    │              │  ½ FFN        │
                    │              └───────┬───────┘
                    │                      │
                    │              (B × T//4 × 256)
                    │                      │
                    └──────[Cross-Attention Fusion]
                                   │
                           (B × T//4 × 256)
                                   │
                         ┌─────────┴─────────┐
                         ▼                   ▼
                    [CTC Head]        [Attention Decoder]
                 Linear(256→80)       (опционально)
                         │
                    log_softmax
                         │
                      ВЫХОД
                  (текст + таймкоды)

        x
        │
        ▼
   ┌─────────┐
   │  ½ FFN  │  LayerNorm → Linear(256→1024) → SiLU → Linear(1024→256)
   └────┬────┘
        │ + residual
        ▼
   ┌──────────────┐
   │ Self-Attention│  MultiHead, 4 головы, d_k=64
   └──────┬───────┘
          │ + residual
          ▼
   ┌──────────────────┐
   │ Convolution Module│
   │                  │
   │  LayerNorm        │
   │  PointwiseConv×2  │  расширение до 2×d
   │  GLU              │  сжатие обратно до d
   │  DepthwiseConv    │  kernel=31, groups=d
   │  BatchNorm + SiLU │
   │  PointwiseConv    │
   └──────┬───────────┘
          │ + residual
          ▼
   ┌─────────┐
   │  ½ FFN  │
   └────┬────┘
        │ + residual
        ▼
   LayerNorm
        │
        ▼
        y

#### **Common Voice 17_0** обучение (v2)

In [15]:
TRAIN_DATA_DIR = "train_data/cv-corpus-25.0-2026-03-09"
# TSV_PATH  = "/ru/validated.tsv"       # или train.tsv
CLIPS_DIR = "/ru/clips"               

ds_cv_train = CommonVoiceLocalDataset(
    tsv_path   = TRAIN_DATA_DIR + "/ru/validated.tsv",
    clips_dir  = TRAIN_DATA_DIR+CLIPS_DIR,
    max_samples = 70_000,
)
ds_cv_val = CommonVoiceLocalDataset(
    tsv_path   = TRAIN_DATA_DIR + "/ru/dev.tsv",
    clips_dir  = TRAIN_DATA_DIR+CLIPS_DIR,
    max_samples = 5_000,
)

train_loader = DataLoader(ds_cv_train, batch_size=8, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(ds_cv_val,   batch_size=8, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)


C:\Users\Tim\AppData\Local\Temp\ipykernel_24032\2688521845.py:15: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(tsv_path, sep="\t")


CommonVoiceLocalDataset: 70000 segments
CommonVoiceLocalDataset: 5000 segments


In [ ]:
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader        = train_loader,
    val_loader          = val_loader,
    checkpoint_path     = CHECKPOINTS_DIR + "/conformer_cv_ru.pt",  # ← от сюда, не от sova
    n_epochs            = 10,
    lr                  = 5e-5,                  # ← меньше lr
    freeze_encoder_epochs = 1,                   
    save_path           =  CHECKPOINTS_DIR + "/conformer_cv_ru_v2.pt",
)


Fine-tune: 10 эпох, lr=5e-05
Checkpoint: avsr_conformer/checkpoints/conformer_cv_ru.pt
Загружен: epoch=5, WER=0.892

  🔒 Энкодер заморожен | обучаемых: 0.9M

  [1/10] step    0/6250 | loss 1.0276 | avg 1.0276 | 0s
  [1/10] step  100/6250 | loss 0.8975 | avg 1.0662 | 26s
  [1/10] step  200/6250 | loss 1.1121 | avg 1.0455 | 50s
  [1/10] step  300/6250 | loss 1.2794 | avg 1.0469 | 75s
  [1/10] step  400/6250 | loss 1.1489 | avg 1.0494 | 99s
  [1/10] step  500/6250 | loss 0.9604 | avg 1.0346 | 124s
  [1/10] step  600/6250 | loss 1.2122 | avg 1.0504 | 148s
  [1/10] step  700/6250 | loss 1.2201 | avg 1.0436 | 171s
  [1/10] step  800/6250 | loss 1.0019 | avg 1.0378 | 196s
  [1/10] step  900/6250 | loss 1.0965 | avg 1.0644 | 220s
  [1/10] step 1000/6250 | loss 1.1314 | avg 1.0716 | 244s
  [1/10] step 1100/6250 | loss 0.8963 | avg 1.0353 | 268s
  [1/10] step 1200/6250 | loss 1.0008 | avg 1.0450 | 292s
  [1/10] step 1300/6250 | loss 1.2046 | avg 1.0601 | 316s
  [1/10] step 1400/6250 | loss 1.311

Exception ignored from cffi callback <function SoundFile._init_virtual_io.<locals>.vio_read at 0x000001FF3ECE32E0>:
Traceback (most recent call last):
  File "c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\soundfile.py", line 1294, in vio_read
    buf = _ffi.buffer(ptr, count)
KeyboardInterrupt: 


In [6]:
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader        = train_loader,
    val_loader          = val_loader,
    checkpoint_path     = CHECKPOINTS_DIR + "/conformer_cv_ru_v2.pt",  # ← от сюда, не от sova
    n_epochs            = 10,
    lr                  = 2e-4,                  # ← чуть больше lr
    freeze_encoder_epochs = 0,                   # ← не морозим, уже адаптирован
    save_path           =  CHECKPOINTS_DIR + "/conformer_cv_ru_v3.pt",
)

Fine-tune: 10 эпох, lr=0.0002
Checkpoint: avsr_conformer/checkpoints/conformer_cv_ru_v2.pt
Загружен: epoch=2, WER=0.886

  🔓 Энкодер разморожен | всего: 13.2M

  [1/10] step    0/6250 | loss 1.0242 | avg 1.0242 | 2s
  [1/10] step  100/6250 | loss 1.0725 | avg 1.0074 | 33s
  [1/10] step  200/6250 | loss 0.7059 | avg 1.0115 | 63s
  [1/10] step  300/6250 | loss 1.0225 | avg 1.0117 | 92s
  [1/10] step  400/6250 | loss 1.1986 | avg 1.0113 | 120s
  [1/10] step  500/6250 | loss 1.2682 | avg 1.0431 | 149s
  [1/10] step  600/6250 | loss 0.9950 | avg 1.0049 | 176s
  [1/10] step  700/6250 | loss 0.9746 | avg 1.0357 | 202s
  [1/10] step  800/6250 | loss 1.3227 | avg 1.0035 | 229s
  [1/10] step  900/6250 | loss 1.0267 | avg 0.9832 | 257s
  [1/10] step 1000/6250 | loss 1.0609 | avg 1.0053 | 284s
  [1/10] step 1100/6250 | loss 0.9057 | avg 0.9982 | 311s
  [1/10] step 1200/6250 | loss 1.0253 | avg 0.9926 | 339s
  [1/10] step 1300/6250 | loss 0.8823 | avg 1.0268 | 366s
  [1/10] step 1400/6250 | loss 0.

KeyboardInterrupt: 

In [ ]:
# ещё 5 эпохи
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader        = train_loader,
    val_loader          = val_loader,
    checkpoint_path     = CHECKPOINTS_DIR + "/conformer_cv_ru_v3.pt",  # ← от сюда, не от sova
    n_epochs            = 10,
    lr                  = 3e-4,                  # ← чуть больше lr
    freeze_encoder_epochs = 0,                   # ← не морозим, уже адаптирован
    save_path           =  CHECKPOINTS_DIR + "/conformer_cv_ru_v4.pt",
)

Fine-tune: 10 эпох, lr=0.0003
Checkpoint: avsr_conformer/checkpoints/conformer_cv_ru_v3.pt
Загружен: epoch=2, WER=0.856

  🔓 Энкодер разморожен | всего: 13.2M

  [1/10] step    0/6250 | loss 0.8019 | avg 0.8019 | 0s


KeyboardInterrupt: 

In [ ]:
torch.save({
    "epoch":       10,
    "model_state": model.state_dict(),
    "val_loss":    0.6912,
    "wer":         0.814,
    "vocab_size":  VOCAB_SIZE,
    "stage":       "finetune_cv_ru_epoch10",
}, "conformer_cv_ru_epoch10.pt")

In [ ]:
# v4 10 epochs
#  
#  lr = 3e-4,                  
#  freeze_encoder_epochs = 0, 

In [19]:
# ещё 5 эпохи
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader        = train_loader,
    val_loader          = val_loader,
    checkpoint_path     = CHECKPOINTS_DIR + "/conformer_cv_ru_v4epoch10.pt",  # ← от сюда, не от sova
    n_epochs            = 15,
    lr                  = 2e-4,                  # ← чуть больше lr
    freeze_encoder_epochs = 0,                   # ← не морозим, уже адаптирован
    save_path           =  CHECKPOINTS_DIR + "/conformer_cv_ru_v5.pt",
)

Fine-tune: 15 эпох, lr=0.0002
Checkpoint: avsr_conformer/checkpoints/conformer_cv_ru_v4epoch10.pt
Загружен: epoch=10, WER=0.814

  🔓 Энкодер разморожен | всего: 13.2M

  [1/15] step    0/8750 | loss 0.7211 | avg 0.7211 | 1s
  [1/15] step  100/8750 | loss 0.8569 | avg 0.6308 | 32s
  [1/15] step  200/8750 | loss 0.4610 | avg 0.6584 | 60s
  [1/15] step  300/8750 | loss 0.5384 | avg 0.6383 | 89s
  [1/15] step  400/8750 | loss 0.6731 | avg 0.6404 | 117s
  [1/15] step  500/8750 | loss 0.5944 | avg 0.6440 | 144s
  [1/15] step  600/8750 | loss 0.5260 | avg 0.6467 | 171s
  [1/15] step  700/8750 | loss 0.6065 | avg 0.6790 | 199s
  [1/15] step  800/8750 | loss 0.6907 | avg 0.6532 | 226s
  [1/15] step  900/8750 | loss 0.8751 | avg 0.6549 | 254s
  [1/15] step 1000/8750 | loss 1.0185 | avg 0.6454 | 282s
  [1/15] step 1100/8750 | loss 0.7304 | avg 0.6421 | 309s
  [1/15] step 1200/8750 | loss 0.4425 | avg 0.6794 | 337s
  [1/15] step 1300/8750 | loss 0.6014 | avg 0.6499 | 366s
  [1/15] step 1400/8750 |

KeyboardInterrupt: 

In [24]:
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader        = train_loader,
    val_loader          = val_loader,
    checkpoint_path     = CHECKPOINTS_DIR + "/conformer_cv_ru_v5.pt",  # ← от сюда, не от sova
    n_epochs            = 5,
    lr                  = 4e-4,                  # ← чуть больше lr
    freeze_encoder_epochs = 0,                   # ← не морозим, уже адаптирован
    save_path           =  CHECKPOINTS_DIR + "/conformer_cv_ru_v5epoch17.pt",
)

Fine-tune: 5 эпох, lr=0.0004
Checkpoint: avsr_conformer/checkpoints/conformer_cv_ru_v5.pt
Загружен: epoch=12, WER=0.743

  🔓 Энкодер разморожен | всего: 13.2M

  [1/5] step    0/8750 | loss 0.8380 | avg 0.8380 | 0s
  [1/5] step  100/8750 | loss 0.4343 | avg 0.4733 | 26s
  [1/5] step  200/8750 | loss 0.4048 | avg 0.4614 | 50s
  [1/5] step  300/8750 | loss 0.4742 | avg 0.5007 | 75s
  [1/5] step  400/8750 | loss 0.4755 | avg 0.5027 | 100s
  [1/5] step  500/8750 | loss 0.3128 | avg 0.4990 | 124s
  [1/5] step  600/8750 | loss 0.4393 | avg 0.4802 | 149s
  [1/5] step  700/8750 | loss 0.7364 | avg 0.4751 | 174s
  [1/5] step  800/8750 | loss 0.5083 | avg 0.5110 | 200s
  [1/5] step  900/8750 | loss 0.3929 | avg 0.5194 | 225s
  [1/5] step 1000/8750 | loss 0.6764 | avg 0.5023 | 250s
  [1/5] step 1100/8750 | loss 0.4711 | avg 0.4979 | 275s
  [1/5] step 1200/8750 | loss 0.3534 | avg 0.5001 | 301s
  [1/5] step 1300/8750 | loss 0.6425 | avg 0.5107 | 327s
  [1/5] step 1400/8750 | loss 0.3816 | avg 0.49

In [26]:
torch.save({
    "epoch":       13,
    "model_state": model.state_dict(),
    "val_loss":    0.5189,
    "wer":         0.74, 
    "vocab_size":  VOCAB_SIZE,
    "stage":       "finetune_cv_ru_v5epoch18",
}, CHECKPOINTS_DIR + "/conformer_cv_ru_v5epoch18.pt")

In [ ]:
# поиск видео-аудио датасета 

In [ ]:
# по ПЗ:
# 
# добавить объём алгоритма + обучения 

In [6]:
from evaluate_performance import load_model, transcribe_video

model = load_model("avsr_conformer\checkpoints\conformer_cv_ru_v5epoch17.pt")

result = transcribe_video(  
        video_path="videos/trushin.mp4",
        model=model,
        segment_sec=20.0,
        use_video=True,
        reference_text=None,      # передай строку для подсчёта WER
    )

Loaded checkpoint: epoch=4, WER=0.701

Transcribing: videos/trushin.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
         13 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  ну в целом мори это случай, когда, еслил пеплохо, чувствую ша депи да можно, в граду сувуйти да пижет свовосьметс драдусов той соте написно косенусна, сколько двудстиградусов на.
    [00:00:19 – 00:00:39]  на, косенус восмидесте градусов на, накосенус скаких ста саракандранстак вод, но дальше чущуть, приведениями припривести
    [00:00:38 – 00:00:58]  препривести, но напримерт, сказать, чтобуто этотоже саммое что синус десятии градусна, а вт этотоже самая, что минус косенус сракааградусв.
    [00:00:57 – 00:01:17]  градусов так цогласонвеот и тебя получаются, уже вот что, минус  всиму сздесять.
    [00:01:16 – 00:01:36]  накось нултвуадцать, накойьсну сорокт первый путь есль знаешь, а пормул усину стройнул в угла <unk> сказать, что, естлен называть, сину здести градусов заты, 

In [27]:
from evaluate_performance import load_model, transcribe_video

model = load_model("avsr_conformer\checkpoints\conformer_cv_ru_v5epoch18.pt")

result = transcribe_video(
        video_path="videos/0511.mp4",
        model=model,
        segment_sec=20.0,
        use_video=True,
        reference_text=None,      # передай строку для подсчёта WER
    )

Loaded checkpoint: epoch=13, WER=0.740

Transcribing: videos/0511.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
         12 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  всем проветнеязават быреструшенные, я учительмы тиматики от и мы сламяюм сегодня не ножшка быболтаем т вы такой истри меня очеом стрим пропоболтать, но там е и заявленой тема, что-то промотиматику, зачем нужна темати ки как-люб
    [00:00:19 – 00:00:39]  потемати, ка как ъе полюбить и, давайте начнем с этуг одальше я буду стараться отведть на, каки-то, ваше вопросыем, да, все что хотите, можете спрашивать, я на это потвечаю вот, если увижу ваше вопросо, смотрити, а, чтотко
    [00:00:38 – 00:00:58]  что такое мотематикгда и как ее волюбить, очне роше ворос о нам жто янм не наит не отвечатьякак яобщей прибил уатиматигда, как этой умногих получается ковото, в действия хорошо о получается быструберии и, они станоется сторок сменныли по-берную куково пы хороше
    [00:00:57 – 00:01:

In [10]:
from evaluate_performance import load_model, transcribe_video

model = load_model("avsr_conformer\checkpoints\conformer_cv_ru_v4epoch10.pt")

result = transcribe_video(
        video_path="videos/0511.mp4",
        model=model,
        segment_sec=20.0,
        use_video=True,
        reference_text=None,      # передай строку для подсчёта WER
    )

Loaded checkpoint: epoch=10, WER=0.814

Transcribing: videos/0511.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
         12 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  всем рвет ня залт бы реструшенный, я учительмо тиматикем от и мы сламеем сегодня неможка быболтаем т вы такой и стри меня чем стрим пропоболтать, ном, в там заявлны тема, что-то проматиматьику, зачем нужно ытиматеи кекатлюб
    [00:00:19 – 00:00:39]  потсематеи кекат ъю полюбить и, давайте насчнем с этууа одальшше я обуду стараться отведтиь на каки-то, ваше вопросыем, дам, всё, что хотитея можете спрашивать, я она это потвечаю вот десле увижу вашего опроса, смотритим а, что тако
    [00:00:38 – 00:00:58]  что такоемыэтиматекода и как ие полювить, точене роше гопроса ам жто я н не нает а не вотвечать,я как яобще плибил насиматигда, как этой умногих олучается уковото, в дествие хорошого получается быструвлериси и анистануются спорок цсменныи по-бъернул куково пыхорошо
    [00:00:5

In [7]:
# from torch.utils.data import Dataset
import torch
import json
import os

class YouTubeAVDataset(torch.utils.data.Dataset):
    def __init__(self, metadata_path: str, language=None,
                 max_duration_sec=15.0, use_video=True, max_samples=None):
        ''' 
        Датасет для fine-tune на YouTube (аудио + видео) 
        Parameters:
            metadata_path    — путь к JSON с метаданными (см. yt_dataset.json)
            language         — фильтр по языку (например, "ru" или "en")
            max_duration_sec — максимальная длительность сегмента в секундах
            use_video        — использовать видео (True/False)
            max_samples      — максимальное количество сегментов (для отладки)
        '''
        self.use_video = use_video
        
        with open(metadata_path, encoding="utf-8") as f:
            self.records = json.load(f)
        # Фильтр по длине
        self.records = [
            r for r in self.records
            # if len(r['transcription'].strip()) >= 3 
            if len(r['transcription'].strip()) >= 2  
            # and len(r['transcription'].strip()) <= max_words 
            and r["duration"] >= 0.5
            and r["duration"] <= max_duration_sec
            and os.path.exists(r["audio_path"])
            and os.path.exists(r["video_path"])
            ]

        if max_samples and max_samples <= len(self.records):
            self.records = self.records[:max_samples]
            
        # Фильтр по языку
        if language:
            self.records = [r for r in self.records
                            if r.get("language", "ru") == language]

        # Фильтр по существующим файлам — убираем битые записи
        self.records = [
            r for r in self.records
            if os.path.exists(r["audio_path"])
            and os.path.exists(r["video_path"])
        ]

        self.audio_extractor = LogMelFeatureExtractor()
        self.video_extractor = VideoFrameExtractor()
        self.max_frames      = int(max_duration_sec * 100)
        print(f"YouTubeAVDataset: {len(self.records)} segments "
              f"(language={language or 'all'})")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]

        # Аудио — с защитой от ошибок
        try:
            with open(rec["audio_path"], "rb") as f:
                audio_bytes = f.read()
            features = self.audio_extractor(audio_bytes)
            if features.shape[0] > self.max_frames:
                features = features[:self.max_frames]
        except Exception as e:
            print(f"  ⚠ Аудио ошибка [{idx}]: {e}")
            # Возвращаем пустой тензор — collate_fn отфильтрует
            features = torch.zeros(10, 80)

        # Видео — с защитой от ошибок
        if not self.use_video:
            frames = None
        else:
            try:
                frames = self.video_extractor.extract(
                    rec["video_path"], 0, rec["duration"]
                )
            except Exception as e:
                print(f"  ⚠ Видео ошибка [{idx}]: {e}")
                frames = None

            # Если видео не извлеклось — заглушка из нулей
            if frames is None:
                frames = torch.zeros(1, 3, 112, 112)

        # Текст
        target = torch.tensor(
            tokenizer.encode(rec["transcription"]), dtype=torch.long
        )

        return features, frames, target, rec["transcription"]

# ============================================================
# ЭТАП 2 — Fine-tune на YouTube (аудио + видео)
# ============================================================
def finetune_youtube(checkpoint_path="conformer_sova.pt",
                     metadata_path="yt_dataset.json",
                     language="ru",
                     n_epochs=5,
                     batch_size=4,        # меньше батч — видео тяжелее
                     lr=2e-5,
                     freeze_encoder_epochs=2,
                     save_path="conformer_finetuned.pt",
                     hf_dataset_name=None,
                     hf_language="ru"):

    if hf_dataset_name:
        from datasets import load_dataset, Audio
        print(f"Loading {hf_dataset_name}...")
        hf_ds = load_dataset(hf_dataset_name, hf_language, 
                             trust_remote_code=True)
        hf_ds = hf_ds.cast_column("audio", Audio(decode=False))
        train_ds = SovaAVDataset(hf_ds["train"], max_samples=20_000)
        val_ds = SovaAVDataset(hf_ds["validation"], max_samples=5_000)
    else:

        print("=" * 60)
        print(f"ЭТАП 2: Fine-tune на YouTube ({language})")
        print("=" * 60)

        # Загружаем веса pretrain
        ckpt  = torch.load(checkpoint_path, map_location=DEVICE)
        model = AVConformerCTC(vocab_size=ckpt["vocab_size"]).to(DEVICE)
        model.load_state_dict(ckpt["model_state"])
        print(f"Загружен претрейн: epoch={ckpt['epoch']}, WER={ckpt['wer']:.3f}\n")

        # Датасет
        full_ds  = YouTubeAVDataset(metadata_path, language=language)
        val_size = max(1, int(len(full_ds) * 0.1))
        train_ds, val_ds = torch.utils.data.random_split(
            full_ds, [len(full_ds) - val_size, val_size]
        )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=collate_fn, 
        num_workers=0, # parallel
        pin_memory=True, # ускоряет при достаточной RAM
        # prefetch_factor=2 # загружать заранее
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=0,
    )
    print(f"Train: {len(train_ds)} | Val: {len(val_ds)}\n")

    optimizer = AdamW([
        {"params": model.subsampling.parameters(), "lr": lr * 0.1},
        {"params": model.conformer.parameters(),   "lr": lr * 0.1},
        {"params": model.pos_enc.parameters(),     "lr": lr * 0.1},
        {"params": model.video_enc.parameters(),   "lr": lr},
        {"params": model.fusion.parameters(),      "lr": lr},
        {"params": model.ctc_head.parameters(),    "lr": lr},
    ], weight_decay=1e-2)

    # optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
    best_wer  = float("inf")

    for epoch in range(n_epochs):

        # Freeze / Unfreeze энкодера
        if epoch < freeze_encoder_epochs:
            for name, param in model.named_parameters():
                param.requires_grad = "ctc_head" in name
            if epoch == 0:
                print("  🔒 Энкодер заморожен\n")
        else:
            for param in model.parameters():
                param.requires_grad = True
            if epoch == freeze_encoder_epochs:
                print("  🔓 Энкодер разморожен — end-to-end\n")

        # ── TRAIN ──────────────────────────────────────────
        model.train()
        train_losses = []
        t0 = time.time()

        for step, (feats, videos, targets, feat_lens, tgt_lens, texts) in enumerate(train_loader):
            feats   = feats.to(DEVICE)
            targets = targets.to(DEVICE)

            # Видео на GPU если есть
            video_tensor = None # пока так
            # if videos is not None:
            #     # print(videos[0].shape)  
            #     # video_tensor = videos.to(DEVICE) if videos is not None else None
            #     if videos is not None and not all(v is None for v in videos):
            #         # video_tensor = torch.stack([v for v in videos if v is not None]).to(DEVICE)
            #         valid_videos = [v for v in videos if v is not None]
            #         max_t = max(v.shape[0] for v in valid_videos)
            #         C, H, W = valid_videos[0].shape[1], valid_videos[0].shape[2], valid_videos[0].shape[3]

            #         video_tensor = torch.zeros(len(valid_videos), max_t, C, H, W)
            #         for i, v in enumerate(valid_videos):
            #             video_tensor[i, :v.shape[0]] = v
            #         video_tensor = video_tensor.to(DEVICE)

            #     else:
            #         video_tensor = None

            log_probs     = model(feats, video_tensor)
            input_lengths = (feat_lens // 4).clamp(max=log_probs.size(1))

            loss = F.ctc_loss(
                log_probs.transpose(0, 1),
                targets, input_lengths, tgt_lens,
                blank=0, zero_infinity=True,
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_losses.append(loss.item())

            if step % 50 == 0:
                print(f"  [{epoch+1}/{n_epochs}] step {step:3d}/{len(train_loader)}"
                      f" | loss {loss.item():.4f} | {time.time()-t0:.0f}s")

        # ── VALIDATION ─────────────────────────────────────
        model.eval()
        val_losses, wer_scores = [], []
        printed_samples = 0

        with torch.no_grad():
            for feats, videos, targets, feat_lens, tgt_lens, texts in val_loader:
                feats   = feats.to(DEVICE)
                targets = targets.to(DEVICE)

                log_probs     = model(feats, video_frames=None)
                input_lengths = (feat_lens // 4).clamp(max=log_probs.size(1))

                loss = F.ctc_loss(
                    log_probs.transpose(0, 1),
                    targets, input_lengths, tgt_lens,
                    blank=0, zero_infinity=True,
                )
                val_losses.append(loss.item())

                preds = log_probs.argmax(dim=-1).cpu()
                for i, pred_ids in enumerate(preds):
                    hyp = tokenizer.decode_greedy(pred_ids.tolist())

                    if printed_samples < 3:
                        print(f"REF: {texts[i]}")
                        print(f"HYP: {hyp}")
                        print(f"WER: {wer(texts[i], hyp):.3f}")
                        print()
                        printed_samples += 1

                    wer_scores.append(wer(texts[i], hyp))

        avg_train = np.mean(train_losses)
        avg_val   = np.mean(val_losses)
        avg_wer   = np.mean(wer_scores)
        scheduler.step()

        print(f"\n{'─'*60}")
        print(f"Epoch {epoch+1:2d}/{n_epochs} | "
              f"train={avg_train:.4f} | val={avg_val:.4f} | WER={avg_wer:.3f}")
        print(f"{'─'*60}\n")

        if avg_wer < best_wer:
            best_wer = avg_wer
            torch.save({
                "epoch":       epoch + 1,
                "model_state": model.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "val_loss":    avg_val,
                "wer":         avg_wer,
                "vocab_size":  VOCAB_SIZE,
                "stage":       f"finetune_youtube_{language}",
            }, save_path)
            print(f"  ✓ Сохранён → {save_path} (WER={avg_wer:.3f})\n")

    return model


In [ ]:
# Проверяем датасет
ds = YouTubeAVDataset("./yt_dataset.json")
features, frames, target, text = ds[0]
print(f"Audio features: {features.shape}")   # (T, 80)
print(f"Video frames:   {frames.shape}")      # (T_v, 3, 112, 112)
print(f"Transcription:  {text}")


YouTubeAVDataset: 15101 segments (language=all)
Audio features: torch.Size([307, 80])
Video frames:   torch.Size([32, 3, 112, 112])
Transcription:  всем привет с вами снова борис трушин и


In [40]:
# ── ЭТАП 2: Fine-tune на YouTube ──────────────────────
# (раскомментируй после этапа 1)
model = finetune_youtube(
        checkpoint_path = CHECKPOINTS_DIR + "/conformer_cv_ru_v5epoch18.pt",
        metadata_path   = "yt_dataset.json",
        language        = "ru",
        n_epochs        = 5,
        lr              = 1e-5,
        freeze_encoder_epochs = 1,
        save_path       = CHECKPOINTS_DIR + "/conformer_youtube_ru.pt",
)

ЭТАП 2: Fine-tune на YouTube (ru)
Загружен претрейн: epoch=13, WER=0.740

YouTubeAVDataset: 15101 segments (language=ru)
Train: 13591 | Val: 1510

  🔒 Энкодер заморожен

  [1/5] step   0/3398 | loss 6.2467 | 1s
  [1/5] step  50/3398 | loss 5.1692 | 35s
  [1/5] step 100/3398 | loss 7.8264 | 65s
  [1/5] step 150/3398 | loss 3.2341 | 95s
  [1/5] step 200/3398 | loss 5.8734 | 125s
  [1/5] step 250/3398 | loss 4.0156 | 155s
  [1/5] step 300/3398 | loss 3.7181 | 185s
  [1/5] step 350/3398 | loss 6.3829 | 219s
  [1/5] step 400/3398 | loss 0.1455 | 247s
  [1/5] step 450/3398 | loss 2.7471 | 278s
  [1/5] step 500/3398 | loss 2.5184 | 309s
  [1/5] step 550/3398 | loss 3.3702 | 341s
  [1/5] step 600/3398 | loss 0.0000 | 370s
  [1/5] step 650/3398 | loss 1.3427 | 400s
  [1/5] step 700/3398 | loss 3.0956 | 430s
  [1/5] step 750/3398 | loss 4.5749 | 462s
  [1/5] step 800/3398 | loss 7.4640 | 492s
  [1/5] step 850/3398 | loss 1.8587 | 522s
  [1/5] step 900/3398 | loss 4.1484 | 552s
  [1/5] step 950/3

KeyboardInterrupt: 

In [41]:
# смешанные данные
from torch.utils.data import ConcatDataset

TRAIN_DATA_DIR = "train_data/cv-corpus-25.0-2026-03-09"
CLIPS_DIR = "/ru/clips"               

CHECKPOINTS_DIR = "avsr_conformer/checkpoints"


ds_cv = CommonVoiceLocalDataset(
    tsv_path    = TRAIN_DATA_DIR + "/ru/validated.tsv",
    clips_dir   = TRAIN_DATA_DIR + CLIPS_DIR,
    max_samples = 40_000,
)

ds_yt = YouTubeAVDataset(
    metadata_path = "yt_dataset.json",
    language      = "ru"
    # max_words     = 12,      # только короткие сегменты
)

# Смешиваем в пропорции 70% CV + 30% YouTube
ds_combined = ConcatDataset([ds_cv, ds_yt])

train_loader = DataLoader(
    ds_combined, batch_size=8, shuffle=True,
    collate_fn=collate_fn, num_workers=0,
)

ds_cv_val = CommonVoiceLocalDataset(
    tsv_path   = TRAIN_DATA_DIR + "/ru/dev.tsv",
    clips_dir  = TRAIN_DATA_DIR+CLIPS_DIR,
    max_samples = 4_000,
)

val_loader   = DataLoader(ds_cv_val,   batch_size=8, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)


C:\Users\Tim\AppData\Local\Temp\ipykernel_2812\2688521845.py:15: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(tsv_path, sep="\t")


CommonVoiceLocalDataset: 40000 segments
YouTubeAVDataset: 15101 segments (language=ru)
CommonVoiceLocalDataset: 4000 segments


In [31]:
model = finetune_on_loaders(
    train_loader        = train_loader,
    val_loader          = val_loader,      # CV val как прежде
    checkpoint_path     = CHECKPOINTS_DIR + "/conformer_cv_ru_v5epoch18.pt",
    n_epochs            = 5,
    lr                  = 2e-5,            # ← в 3 раза меньше чем раньше
    freeze_encoder_epochs = 0,             # ← не морозим
    save_path           = CHECKPOINTS_DIR + "/conformer_mixed.pt",
)

Fine-tune: 5 эпох, lr=2e-05
Checkpoint: avsr_conformer/checkpoints/conformer_cv_ru_v5epoch18.pt
Загружен: epoch=13, WER=0.740

  🔓 Энкодер разморожен | всего: 13.2M

  [1/5] step    0/2501 | loss 0.3420 | avg 0.3420 | 0s
  [1/5] step  100/2501 | loss 0.6951 | avg 0.3820 | 24s
  [1/5] step  200/2501 | loss 0.2926 | avg 0.3992 | 47s
  [1/5] step  300/2501 | loss 0.3074 | avg 0.3758 | 70s
  [1/5] step  400/2501 | loss 0.3790 | avg 0.3804 | 93s
  [1/5] step  500/2501 | loss 0.4090 | avg 0.3985 | 116s
  [1/5] step  600/2501 | loss 0.4893 | avg 0.3778 | 139s
  [1/5] step  700/2501 | loss 0.3972 | avg 0.3885 | 162s
  [1/5] step  800/2501 | loss 0.3379 | avg 0.4024 | 185s
  [1/5] step  900/2501 | loss 0.3289 | avg 0.4308 | 209s
  [1/5] step 1000/2501 | loss 0.2397 | avg 0.3957 | 232s
  [1/5] step 1100/2501 | loss 0.6455 | avg 0.3893 | 255s
  [1/5] step 1200/2501 | loss 0.5896 | avg 0.3838 | 279s
  [1/5] step 1300/2501 | loss 0.4773 | avg 0.3823 | 302s
  [1/5] step 1400/2501 | loss 0.2623 | avg

In [34]:
model = finetune_on_loaders(
    train_loader        = train_loader,
    val_loader          = val_loader,      # CV val как прежде
    checkpoint_path     = CHECKPOINTS_DIR + "/conformer_mixed.pt",
    n_epochs            = 10,
    lr                  = 3e-5,            # ← в 3 раза меньше чем раньше
    freeze_encoder_epochs = 0,             # ← не морозим
    save_path           = CHECKPOINTS_DIR + "/conformer_mixed_1.pt",
)

Fine-tune: 10 эпох, lr=3e-05
Checkpoint: avsr_conformer/checkpoints/conformer_mixed.pt
Загружен: epoch=5, WER=0.727

  🔓 Энкодер разморожен | всего: 13.2M

  [1/10] step    0/2501 | loss 0.3344 | avg 0.3344 | 0s
  [1/10] step  100/2501 | loss 0.5073 | avg 0.3667 | 18s
  [1/10] step  200/2501 | loss 0.3284 | avg 0.3921 | 35s
  [1/10] step  300/2501 | loss 0.2375 | avg 0.3614 | 52s
  [1/10] step  400/2501 | loss 0.3946 | avg 0.3810 | 69s
  [1/10] step  500/2501 | loss 0.3159 | avg 0.3816 | 86s
  [1/10] step  600/2501 | loss 0.5041 | avg 0.3794 | 103s
  [1/10] step  700/2501 | loss 0.3174 | avg 0.3775 | 120s
  [1/10] step  800/2501 | loss 0.3720 | avg 0.3925 | 137s
  [1/10] step  900/2501 | loss 0.3476 | avg 0.3962 | 154s
  [1/10] step 1000/2501 | loss 0.3732 | avg 0.3909 | 171s
  [1/10] step 1100/2501 | loss 0.3117 | avg 0.3787 | 188s
  [1/10] step 1200/2501 | loss 0.3612 | avg 0.4113 | 205s
  [1/10] step 1300/2501 | loss 0.2285 | avg 0.3594 | 223s
  [1/10] step 1400/2501 | loss 0.2479 |

Exception ignored from cffi callback <function SoundFile._init_virtual_io.<locals>.vio_read at 0x000002C424DD72E0>:
Traceback (most recent call last):
  File "c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\soundfile.py", line 1294, in vio_read
    buf = _ffi.buffer(ptr, count)
KeyboardInterrupt: 


  [9/10] step  100/2501 | loss 0.2662 | avg 0.3669 | 18s


KeyboardInterrupt: 

In [ ]:
# кажется стагнировались лучше 3й эпохи 0.27 wer обучение не идёт

In [36]:
# ── ЭТАП: Fine-tune на YouTube ──────────────────────
# пробуем ютуб
model = finetune_youtube(
        checkpoint_path = CHECKPOINTS_DIR + "/conformer_mixed.pt",
        metadata_path   = "yt_dataset.json",
        language        = "ru",
        n_epochs        = 5,
        lr              = 1e-5,
        freeze_encoder_epochs = 1,
        save_path       = CHECKPOINTS_DIR + "/conformer_youtube_ru.pt",
)

ЭТАП 2: Fine-tune на YouTube (ru)
Загружен претрейн: epoch=5, WER=0.727

YouTubeAVDataset: 8 segments (language=ru)
Train: 7 | Val: 1

  🔒 Энкодер заморожен

  [1/5] step   0/2 | loss 8.8686 | 1s
REF: да и возьмём
HYP: равный
WER: 1.000


────────────────────────────────────────────────────────────
Epoch  1/5 | train=9.6692 | val=11.1823 | WER=1.000
────────────────────────────────────────────────────────────

  ✓ Сохранён → avsr_conformer/checkpoints/conformer_youtube_ru.pt (WER=1.000)

  🔓 Энкодер разморожен — end-to-end

  [2/5] step   0/2 | loss 8.6769 | 1s
REF: да и возьмём
HYP: равный
WER: 1.000


────────────────────────────────────────────────────────────
Epoch  2/5 | train=9.6773 | val=11.2735 | WER=1.000
────────────────────────────────────────────────────────────

  [3/5] step   0/2 | loss 10.2048 | 1s
REF: да и возьмём
HYP: равный
WER: 1.000


────────────────────────────────────────────────────────────
Epoch  3/5 | train=9.0974 | val=11.3186 | WER=1.000
───────────────────

In [38]:
# ── ЭТАП: Fine-tune на YouTube ──────────────────────
# пробуем ютуб
model = finetune_youtube(
        checkpoint_path = CHECKPOINTS_DIR + "/conformer_mixed.pt",
        metadata_path   = "yt_dataset.json",
        language        = "ru",
        n_epochs        = 5,
        lr              = 1e-5,
        freeze_encoder_epochs = 1,
        save_path       = CHECKPOINTS_DIR + "/conformer_youtube_ru.pt",
)

ЭТАП 2: Fine-tune на YouTube (ru)
Загружен претрейн: epoch=5, WER=0.727

YouTubeAVDataset: 15101 segments (language=ru)
Train: 13591 | Val: 1510

  🔒 Энкодер заморожен

  [1/5] step   0/3398 | loss 0.0000 | 1s
  [1/5] step  50/3398 | loss 1.6745 | 31s


KeyboardInterrupt: 

In [42]:
model = finetune_on_loaders(
    train_loader        = train_loader,
    val_loader          = val_loader,      # CV val как прежде
    checkpoint_path     = CHECKPOINTS_DIR + "/conformer_mixed.pt",
    n_epochs            = 5,
    lr                  = 2e-5,            # ← в 3 раза меньше чем раньше
    freeze_encoder_epochs = 0,             # ← не морозим
    save_path           = CHECKPOINTS_DIR + "/conformer_mixed_1.pt",
)

Fine-tune: 5 эпох, lr=2e-05
Checkpoint: avsr_conformer/checkpoints/conformer_mixed.pt
Загружен: epoch=5, WER=0.727

  🔓 Энкодер разморожен | всего: 13.2M

  [1/5] step    0/6888 | loss 3.0721 | avg 3.0721 | 1s
  [1/5] step  100/6888 | loss 1.1114 | avg 1.1401 | 50s
  [1/5] step  200/6888 | loss 0.5061 | avg 1.1300 | 100s
  [1/5] step  300/6888 | loss 0.9060 | avg 1.0279 | 152s
  [1/5] step  400/6888 | loss 1.0789 | avg 1.0159 | 202s
  [1/5] step  500/6888 | loss 0.3969 | avg 0.9521 | 252s
  [1/5] step  600/6888 | loss 0.6332 | avg 0.9598 | 300s
  [1/5] step  700/6888 | loss 1.2534 | avg 0.9098 | 349s
  [1/5] step  800/6888 | loss 0.4077 | avg 0.8472 | 396s
  [1/5] step  900/6888 | loss 1.1348 | avg 0.9361 | 445s
  [1/5] step 1000/6888 | loss 0.8604 | avg 0.9680 | 495s
  [1/5] step 1100/6888 | loss 1.0857 | avg 0.8767 | 545s
  [1/5] step 1200/6888 | loss 0.3511 | avg 0.8350 | 595s
  [1/5] step 1300/6888 | loss 0.4011 | avg 0.8356 | 646s
  [1/5] step 1400/6888 | loss 1.1667 | avg 0.8053 

#### *проверка на видео*

In [7]:
# conformer_cv_ru_v5epoch17
from evaluate_performance import load_model, transcribe_video

model = load_model("avsr_conformer/checkpoints/conformer_cv_ru_v5epoch17_best.pt")

result = transcribe_video(
        video_path="videos/0511.mp4",
        model=model,
        segment_sec=20.0,
        use_video=True,
        reference_text=None,      # передай строку для подсчёта WER
    )

Loaded checkpoint: epoch=4, WER=0.701

Transcribing: videos/0511.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
         12 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  всем приветнеязават быреструшенны, я учительмы тематике от и мы сламиюм сегодня не ножшка поболтаем т вы такой истри мени очеом стрим пропоболтать, ном там е и заявленой тема, что-то промотиматику, зачем нужна м темати ки как-люб
    [00:00:19 – 00:00:39]  потемати, ка как ъе полюбить и, давайте ночнем с этуг одальше я буду стараться отведть на, каки-то, ваше вопросым, дат, все что хотите, можете спрашивать, и и на это потвечаю вот, если увижу ваше вопросо смотрити, а, чтотко
    [00:00:38 – 00:00:58]  что такоемоэтематикгда и как ее полювить, точене роше вопрос о там что ян не наит не отвечатья как яобщей прибил натематигда, как этой умногих получается ковото, в девствия хорошо о получается быструберии и, ани станоется сторок сменныи по-бернуло куково пы хорошо
    [00:00:57 – 0

In [ ]:
from evaluate_performance import load_model, transcribe_video

model = load_model("avsr_conformer/checkpoints/conformer_mixed.pt")

result = transcribe_video(
        video_path="videos/0511.mp4",
        model=model,
        segment_sec=20.0,
        use_video=True,
        reference_text=None,      # передай строку для подсчёта WER
    )

Loaded checkpoint: epoch=5, WER=0.727

Transcribing: videos/0511.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
         12 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  всем прветнеязават быреструшенные, я учительмы тиматике от и мы сламеюм сегодня не ножку поболтаем т вы такой изстри меня очеом стрим пропоболтать, но, там е и заявленой тема, что-то промотиматику, зачем нужна отемате ки как-люб
    [00:00:19 – 00:00:39]  потсемати, ка как ъе полюбить, и, давайте начнем с этуг одальше я буду стараться отведть на, каки-то, ваше вопросыем, да, все что хотите, можете спрашивать, я на это потвечаю вот, если увижу ваше вопросо, смотрити, а, чтот ко
    [00:00:38 – 00:00:58]  что такое мотематикгда и как ее полюбить, очне роше ворос о нам жто янм не наит не отвечать,я, как яобщей прибил уатиматигда, как этой умногих получается ковото,  в действия хорошого получается быструберии и, они станоется сторок сменныли по-бъерную куково пы хороше
    [00:00:57 

In [ ]:
# хронология обучения

# 1) sova_rudevices - 20_000 тренировочная, epoch=8, WER=0.882 
# 2) common voice 17_0 - сначала на 40_000, потом на 70_000 train, на 70_000 13 эпох, WER=0.74
# 3) mixed - смешанные данные common voice + youtube, 5 эпох, WER=0.727


# - написаить выполненное оьучение 
# - расписать теорию,
# 


теперь можно расписать по теории из того, что уже есть
вот в структуре


```markdown
                   ВХОД
         ┌──────────────────────────┐
         │   Видеокадры             │   Аудио (16 кГц)
         │   (T_v, 3, 112, 112)     │   (T сэмплов)
         └──────────┬───────────────┘
                    │                        │
                    ▼                        ▼
           [VideoEncoder]          [Лог мел-спектрограмма]
           Conv2D × 3                  (T × 80)
           AdaptivePool                    │
           Linear → 256                   ▼
                    │              [ConvSubsampling]
                    │              Conv2D × 2, stride=2
                    │              Linear → 256
                    │                  (T//4 × 256)
                    │                      │
                    │              [Позиционное кодирование]
                    │                      │
                    │              ┌───────────────┐
                    │              │ ConformerBlock │ × 6
                    │              │               │
                    │              │  ½ FFN        │
                    │              │  Self-Attn    │
                    │              │  Conv Module  │
                    │              │  ½ FFN        │
                    │              └───────┬───────┘
                    │                      │
                    │              (B × T//4 × 256)
                    │                      │
                    └──────[Cross-Attention Fusion]
                                   │
                           (B × T//4 × 256)
                                   │
                         ┌─────────┴─────────┐
                         ▼                   ▼
                    [CTC Head]        [Attention Decoder]
                 Linear(256→80)       (опционально)
                         │
                    log_softmax
                         │
                      ВЫХОД
                  (текст + таймкоды)
```


можно поочерёдно расписать пункты - VideoEncoder, перевод извлечение мелспектограммы из аудио, что есть мелспектограмма, CTC head. ConvSamplingLayer, ConformerBlock - уже есть, 
можно расписать по метрике WER - что это как считается

In [ ]:
# закинуть сгенерированные ответы

In [ ]:
# структура презы:
# 
# НАЗВАНИЕ
# - описание проблемы
# - существующие решения
# - задача исследования 

# получение хорошего, близкого к оригиналу результата распознавания - задача, требующая высоких вычислительных ресурсов и большого объёма данных

# - резульльтат который есть - WER=0.727 - продемонстрировать на видео
# 
# задача исследования, в первую очередь - в создании открытой отечественной разработки, которая может стать полезным фундаментом для дальнейшего обучения и экспериментов
# процесс обучения - итеративный
#
# задача от отдела аспирантуры​ 
# насколько открытый код

In [ ]:
# ============================================================
# ОБНОВЛЁННЫЙ TRAIN STEP — гибридная функция потерь
# ============================================================
LAMBDA_CTC = 0.7   # вес CTC в суммарном лоссе

def compute_loss(model, feats, targets, feat_lens, tgt_lens,
                 video_tensor=None, use_attention=True):
    """
    Вычисляет гибридный loss: λ·L_CTC + (1-λ)·L_Attention
    """
    B = feats.size(0)

    # Подготовка teacher-forcing токенов для Attention Decoder:
    # вход:  <blank> + target[:-1]  (сдвиг вправо)
    # выход: target  (то что предсказываем)
    if use_attention:
        # Создаём сдвинутые входы: [<blank>, t1, t2, ..., t_{L-1}]
        bos = torch.zeros(B, 1, dtype=torch.long, device=feats.device)
        tgt_in  = torch.cat([bos, targets[:, :-1]], dim=1)  # вход декодера
        tgt_out = targets                                     # что предсказываем
    else:
        tgt_in = None

    print(f'tgt_in в compute_loss: {tgt_in}')
    # здесь с
    # print(f"targets shape: {targets.shape}")
    # print(f"tgt_lens: {tgt_lens}")

    ctc_logits, attn_logits = model(feats, video_tensor, tgt_in)

    # CTC loss
    input_lengths = (feat_lens // 4).clamp(max=ctc_logits.size(1))
    loss_ctc = F.ctc_loss(
        ctc_logits.transpose(0, 1),
        targets, input_lengths, tgt_lens,
        blank=0, zero_infinity=True,
    )

    loss_attn = torch.tensor(0.0, device=feats.device)

    # Attention loss (cross-entropy)
    if (attn_logits is not None
            and not torch.isnan(attn_logits).any()
            and not torch.isinf(attn_logits).any()
            and not torch.isnan(loss_ctc)
            and not torch.isinf(loss_ctc)):
        
        # attn_logits: (B, L, vocab) → (B*L, vocab)
        # tgt_out:     (B, L)        → (B*L,)
        # print(f"    attn stats: min={attn_logits.min():.3f} "
        #   f"max={attn_logits.max():.3f} "
        #   f"nan={torch.isnan(attn_logits).sum().item()} "
        #   f"inf={torch.isinf(attn_logits).sum().item()}")

        loss_attn = F.cross_entropy(
            attn_logits.reshape(-1, attn_logits.size(-1)),
            tgt_out.reshape(-1),
            ignore_index=0,   # игнорируем padding
        )
         # Защита от nan в attn loss
        if torch.isnan(loss_attn) or torch.isinf(loss_attn):
            loss_attn = torch.tensor(0.0, device=feats.device)

        
    if loss_attn.item() == 0.0:
        loss      = loss_ctc
    else:
        loss = LAMBDA_CTC * loss_ctc + (1 - LAMBDA_CTC) * loss_attn

    return loss, loss_ctc, loss_attn


# ============================================================
# ОБНОВЛЁННЫЙ finetune_on_loaders
# ============================================================
def finetune_on_loaders(train_loader, val_loader,
                        checkpoint_path="conformer_sova_1.pt",
                        n_epochs=5, lr=1e-4,
                        freeze_encoder_epochs=1,
                        use_attention=True,       # ← новый флаг
                        save_path="conformer_finetuned.pt"):

    print("=" * 60)
    print(f"Fine-tune: {n_epochs} эпох, lr={lr}, attention={use_attention}")
    print("=" * 60)

    ckpt  = torch.load(checkpoint_path, map_location=DEVICE)
    model = AVConformerCTC(vocab_size=ckpt["vocab_size"]).to(DEVICE)

    # Загружаем только те веса которые совпадают
    # (нужно если старый чекпоинт без attention decoder)
    state = ckpt["model_state"]
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print(f"  Новые параметры (инициализированы случайно): {len(missing)}")
    print(f"Загружен: epoch={ckpt['epoch']}, WER={ckpt['wer']:.3f}\n")

    optimizer = AdamW([
        {"params": model.subsampling.parameters(), "lr": lr * 0.1},
        {"params": model.conformer.parameters(),   "lr": lr * 0.1},
        {"params": model.pos_enc.parameters(),     "lr": lr * 0.1},
        {"params": model.video_enc.parameters(),   "lr": lr},
        {"params": model.fusion.parameters(),      "lr": lr},
        {"params": model.ctc_head.parameters(),    "lr": lr},
        {"params": model.decoder.parameters(),     "lr": lr},      # ← новое
        {"params": model.decoder_embed.parameters(),"lr": lr},     # ← новое
        {"params": model.decoder_pos.parameters(), "lr": lr},      # ← новое
        {"params": model.output_proj.parameters(), "lr": lr},      # ← новое
    ], weight_decay=1e-2)

    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
    best_wer  = float("inf")

    for epoch in range(n_epochs):

        if epoch < freeze_encoder_epochs:
            for name, param in model.named_parameters():
                param.requires_grad = not any(
                    k in name for k in ["subsampling", "conformer", "pos_enc"]
                )
            if epoch == 0:
                n = sum(p.numel() for p in model.parameters()
                        if p.requires_grad) / 1e6
                print(f"  🔒 Энкодер заморожен | обучаемых: {n:.1f}M\n")
        else:
            for param in model.parameters():
                param.requires_grad = True
            if epoch == freeze_encoder_epochs:
                n = sum(p.numel() for p in model.parameters()) / 1e6
                print(f"  🔓 Энкодер разморожен | всего: {n:.1f}M\n")

        # ── TRAIN ──────────────────────────────────────────────
        model.train()
        train_losses, ctc_losses, attn_losses = [], [], []
        t0 = time.time()

        for step, (feats, videos, targets, feat_lens, tgt_lens, texts) in enumerate(train_loader):
            if step == 0:
                print(f"targets shape: {targets.shape}")
                print(f"tgt_lens: {tgt_lens[:8]}")
                print(f"texts: {list(texts[:3])}")
                print(f"target[0]: {targets[0]}")
                print(f"decoded: '{tokenizer.decode_greedy(targets[0].tolist())}'")
                break

            feats   = feats.to(DEVICE)
            targets = targets.to(DEVICE)
            # print(f"увеличиваем  step {step}, {step % 100 == 0}")
            video_tensor = None
            if videos is not None:
                valid = [v for v in videos if v is not None]
                if valid:
                    max_t = max(v.shape[0] for v in valid)
                    C, H, W = valid[0].shape[1:]
                    vt = torch.zeros(len(videos), max_t, C, H, W)
                    vi = [i for i, v in enumerate(videos) if v is not None]
                    for i, v in zip(vi, valid):
                        vt[i, :v.shape[0]] = v
                    video_tensor = vt.to(DEVICE)

            print(f"texts пример: {texts[:2]}")

            loss, l_ctc, l_attn = compute_loss(
                model, feats, targets, feat_lens, tgt_lens,
                video_tensor, use_attention=use_attention,
            )

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"  ⚠ step {step}: loss={loss.item():.4f} "
                     f"| ctc={l_ctc.item():.4f} "
                     f"| attn={l_attn.item():.4f}")
                print(f"  feats: min={feats.min():.3f} max={feats.max():.3f}")
                print(f"  targets shape: {targets.shape}, "
                     f"tgt_lens: {tgt_lens[:4]}, "
                    f"feat_lens//4: {(feat_lens//4)[:4]}")

                continue

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            train_losses.append(loss.item())
            ctc_losses.append(l_ctc.item())
            attn_losses.append(l_attn.item())

            # print(s)
            if step % 100 == 0:
                # print('Ураа!! step {step}')
                print(f'[{epoch+1}/{n_epochs}]')
                print(f"  [{epoch+1}/{n_epochs}] step {step:4d}/{len(train_loader)}"
                      f" | loss {loss.item():.4f}"
                      f" | ctc {l_ctc.item():.4f}"
                      f" | attn {l_attn.item():.4f}"
                      f" | {time.time()-t0:.0f}s",
                      flush=True)

        # ── VALIDATION ─────────────────────────────────────────
        model.eval()
        val_losses, wer_scores = [], []
        sample_printed = 0

        with torch.no_grad():
            for feats, videos, targets, feat_lens, tgt_lens, texts in val_loader:
                feats   = feats.to(DEVICE)
                targets = targets.to(DEVICE)

                # На валидации — только CTC (быстро и без teacher forcing)
                ctc_logits, _ = model(feats, video_frames=None,
                                      target_tokens=None)
                input_lengths = (feat_lens // 4).clamp(max=ctc_logits.size(1))

                loss_val = F.ctc_loss(
                    ctc_logits.transpose(0, 1),
                    targets, input_lengths, tgt_lens,
                    blank=0, zero_infinity=True,
                )
                if not (torch.isnan(loss_val) or torch.isinf(loss_val)):
                    val_losses.append(loss_val.item())

                preds = ctc_logits.argmax(dim=-1).cpu()
                for i, pred_ids in enumerate(preds):
                    hyp = tokenizer.decode_greedy(pred_ids.tolist())
                    wer_scores.append(wer(texts[i], hyp))
                    if sample_printed < 3:
                        print(f"  REF: {texts[i]}")
                        print(f"  HYP: {hyp}")
                        print(f"  WER: {wer(texts[i], hyp):.3f}\n")
                        sample_printed += 1

        avg_train = np.mean(train_losses) if train_losses else float("inf")
        avg_val   = np.mean(val_losses)   if val_losses   else float("inf")
        avg_wer   = np.mean(wer_scores)   if wer_scores   else float("inf")
        avg_ctc   = np.mean(ctc_losses)   if ctc_losses   else float("inf")
        avg_attn  = np.mean(attn_losses)  if attn_losses  else float("inf")
        scheduler.step()

        print(f"\n{'─'*60}")
        print(f"Epoch {epoch+1:2d}/{n_epochs} | "
              f"loss={avg_train:.4f} | ctc={avg_ctc:.4f} | attn={avg_attn:.4f} | "
              f"val={avg_val:.4f} | WER={avg_wer:.3f}")
        print(f"{'─'*60}\n")

        if avg_wer < best_wer:
            best_wer = avg_wer
            torch.save({
                "epoch":       epoch + 1,
                "model_state": model.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "val_loss":    avg_val,
                "wer":         avg_wer,
                "vocab_size":  VOCAB_SIZE,
                "stage":       "hybrid_ctc_attention",
            }, save_path)
            print(f"  ✓ Сохранён → {save_path} (WER={avg_wer:.3f})\n")

    return model

In [ ]:
TRAIN_DATA_DIR = "train_data/cv-corpus-25.0-2026-03-09"
# TSV_PATH  = "/ru/validated.tsv"       # или train.tsv
CLIPS_DIR = "/ru/clips"               

ds_cv_train = CommonVoiceLocalDataset(
    tsv_path   = TRAIN_DATA_DIR + "/ru/validated.tsv",
    clips_dir  = TRAIN_DATA_DIR+CLIPS_DIR,
    max_samples = 70_000,
)
ds_cv_val = CommonVoiceLocalDataset(
    tsv_path   = TRAIN_DATA_DIR + "/ru/dev.tsv",
    clips_dir  = TRAIN_DATA_DIR+CLIPS_DIR,
    max_samples = 5_000,
)

train_loader = DataLoader(ds_cv_train, batch_size=8, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(ds_cv_val,   batch_size=8, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)


C:\Users\Tim\AppData\Local\Temp\ipykernel_27132\2688521845.py:15: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(tsv_path, sep="\t")


CommonVoiceLocalDataset: 70000 segments
CommonVoiceLocalDataset: 5000 segments


In [ ]:
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_cv_ru_v5epoch17_best.pt",
    n_epochs              = 5,
    lr                    = 1e-4,
    freeze_encoder_epochs = 2,    # сначала учим только decoder
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_hybrid.pt",
)

Fine-tune: 5 эпох, lr=0.0001, attention=True
  Новые параметры (инициализированы случайно): 58
Загружен: epoch=5, WER=0.727

  🔒 Энкодер заморожен | обучаемых: 4.2M

увеличиваем  step 0, True
Ураа!! step {step}
[1/5]
  [1/5] step    0/8750 | loss 0.2707 | ctc 0.2707 | attn 0.0000 | 1s
увеличиваем  step 1, False
увеличиваем  step 2, False
увеличиваем  step 3, False
увеличиваем  step 4, False
увеличиваем  step 5, False
увеличиваем  step 6, False
увеличиваем  step 7, False
увеличиваем  step 8, False
увеличиваем  step 9, False
увеличиваем  step 10, False
увеличиваем  step 11, False
увеличиваем  step 12, False
увеличиваем  step 13, False
увеличиваем  step 14, False
увеличиваем  step 15, False
увеличиваем  step 16, False
увеличиваем  step 17, False
увеличиваем  step 18, False
увеличиваем  step 19, False
увеличиваем  step 20, False
увеличиваем  step 21, False
увеличиваем  step 22, False
увеличиваем  step 23, False
увеличиваем  step 24, False
увеличиваем  step 25, False
увеличиваем  step 26, F

KeyboardInterrupt: 

In [12]:
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_cv_ru_v5epoch17_best.pt",
    n_epochs              = 5,
    lr                    = 4e-4,
    freeze_encoder_epochs = 0,    # сразу 
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_hybrid.pt",
)

Fine-tune: 5 эпох, lr=0.0004, attention=True
  Новые параметры (инициализированы случайно): 58
Загружен: epoch=4, WER=0.701

  🔓 Энкодер разморожен | всего: 16.5M

Ураа!! step {step}
[1/5]
  [1/5] step    0/8750 | loss 0.2977 | ctc 0.2977 | attn 0.0000 | 0s


c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Ураа!! step {step}
[1/5]
  [1/5] step  100/8750 | loss 0.2949 | ctc 0.2949 | attn 0.0000 | 23s
Ураа!! step {step}
[1/5]
  [1/5] step  200/8750 | loss 0.3472 | ctc 0.3472 | attn 0.0000 | 45s
Ураа!! step {step}
[1/5]
  [1/5] step  300/8750 | loss 0.3144 | ctc 0.3144 | attn 0.0000 | 68s
Ураа!! step {step}
[1/5]
  [1/5] step  400/8750 | loss 0.3532 | ctc 0.3532 | attn 0.0000 | 91s
Ураа!! step {step}
[1/5]
  [1/5] step  500/8750 | loss 0.4445 | ctc 0.4445 | attn 0.0000 | 113s
Ураа!! step {step}
[1/5]
  [1/5] step  600/8750 | loss 0.3430 | ctc 0.3430 | attn 0.0000 | 136s
Ураа!! step {step}
[1/5]
  [1/5] step  700/8750 | loss 0.4616 | ctc 0.4616 | attn 0.0000 | 159s
Ураа!! step {step}
[1/5]
  [1/5] step  800/8750 | loss 0.2881 | ctc 0.2881 | attn 0.0000 | 181s
Ураа!! step {step}
[1/5]
  [1/5] step  900/8750 | loss 0.2609 | ctc 0.2609 | attn 0.0000 | 204s
Ураа!! step {step}
[1/5]
  [1/5] step 1000/8750 | loss 0.5835 | ctc 0.5835 | attn 0.0000 | 226s
Ураа!! step {step}
[1/5]
  [1/5] step 1100/8

Exception ignored from cffi callback <function SoundFile._init_virtual_io.<locals>.vio_read at 0x000002AC1D67BA30>:
Traceback (most recent call last):
  File "c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\soundfile.py", line 1294, in vio_read
    buf = _ffi.buffer(ptr, count)
KeyboardInterrupt: 


Ураа!! step {step}
[5/5]
  [5/5] step 4700/8750 | loss 0.2576 | ctc 0.2576 | attn 0.0000 | 1540s
Ураа!! step {step}
[5/5]
  [5/5] step 4800/8750 | loss 0.5278 | ctc 0.5278 | attn 0.0000 | 1576s
Ураа!! step {step}
[5/5]
  [5/5] step 4900/8750 | loss 0.2488 | ctc 0.2488 | attn 0.0000 | 1613s
Ураа!! step {step}
[5/5]
  [5/5] step 5000/8750 | loss 0.3181 | ctc 0.3181 | attn 0.0000 | 1649s
Ураа!! step {step}
[5/5]
  [5/5] step 5100/8750 | loss 0.2888 | ctc 0.2888 | attn 0.0000 | 1684s
Ураа!! step {step}
[5/5]
  [5/5] step 5200/8750 | loss 0.3461 | ctc 0.3461 | attn 0.0000 | 1719s
Ураа!! step {step}
[5/5]
  [5/5] step 5300/8750 | loss 0.4247 | ctc 0.4247 | attn 0.0000 | 1754s
Ураа!! step {step}
[5/5]
  [5/5] step 5400/8750 | loss 0.3162 | ctc 0.3162 | attn 0.0000 | 1789s
Ураа!! step {step}
[5/5]
  [5/5] step 5500/8750 | loss 0.3396 | ctc 0.3396 | attn 0.0000 | 1823s
Ураа!! step {step}
[5/5]
  [5/5] step 5600/8750 | loss 0.1835 | ctc 0.1835 | attn 0.0000 | 1859s
Ураа!! step {step}
[5/5]
  [5/

In [13]:
train_loader = DataLoader(ds_cv_train, batch_size=8, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)

In [14]:
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_hybrid.pt",
    n_epochs              = 4,
    lr                    = 6e-4, 
    freeze_encoder_epochs = 0,    # сразу 
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_hybrid_1.pt",
)

Fine-tune: 4 эпох, lr=0.0006, attention=True
Загружен: epoch=2, WER=0.703

  🔓 Энкодер разморожен | всего: 16.5M

Ураа!! step {step}
[1/4]
  [1/4] step    0/8750 | loss 0.3314 | ctc 0.3314 | attn 0.0000 | 1s


c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Ураа!! step {step}
[1/4]
  [1/4] step  100/8750 | loss 0.3737 | ctc 0.3737 | attn 0.0000 | 24s
Ураа!! step {step}
[1/4]
  [1/4] step  200/8750 | loss 0.3125 | ctc 0.3125 | attn 0.0000 | 48s
Ураа!! step {step}
[1/4]
  [1/4] step  300/8750 | loss 0.4946 | ctc 0.4946 | attn 0.0000 | 72s
Ураа!! step {step}
[1/4]
  [1/4] step  400/8750 | loss 0.4105 | ctc 0.4105 | attn 0.0000 | 96s
Ураа!! step {step}
[1/4]
  [1/4] step  500/8750 | loss 0.3486 | ctc 0.3486 | attn 0.0000 | 120s
Ураа!! step {step}
[1/4]
  [1/4] step  600/8750 | loss 0.4606 | ctc 0.4606 | attn 0.0000 | 144s
Ураа!! step {step}
[1/4]
  [1/4] step  700/8750 | loss 0.4499 | ctc 0.4499 | attn 0.0000 | 167s
Ураа!! step {step}
[1/4]
  [1/4] step  800/8750 | loss 0.4200 | ctc 0.4200 | attn 0.0000 | 191s
Ураа!! step {step}
[1/4]
  [1/4] step  900/8750 | loss 0.3231 | ctc 0.3231 | attn 0.0000 | 215s
Ураа!! step {step}
[1/4]
  [1/4] step 1000/8750 | loss 0.3528 | ctc 0.3528 | attn 0.0000 | 238s
Ураа!! step {step}
[1/4]
  [1/4] step 1100/8

In [15]:
# другие данные

In [16]:
import os
import pandas as pd

from torch.utils.data import Dataset

class OpenSTTDataset(Dataset):
    def __init__(self, manifest_path, data_dir,
                 max_duration_sec=15.0,
                 min_duration_sec=1.0,
                 max_samples=None):
        """
        manifest_path — путь к .csv файлу
        data_dir      — папка где лежит public_youtube1120/
        """
        # Читаем манифест — без заголовка, три колонки
        df = pd.read_csv(manifest_path, header=None,
                         names=["audio_path", "text_path", "duration"])

        # Фильтруем по длительности
        df = df[
            (df["duration"] >= min_duration_sec) &
            (df["duration"] <= max_duration_sec)
        ]

        # Перемешиваем и берём нужное количество
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
        if max_samples:
            df = df.iloc[:max_samples]

        self.records    = df
        self.data_dir   = data_dir
        self.max_frames = int(max_duration_sec * 100)

        print(f"OpenSTTDataset: {len(self.records)} segments "
              f"(duration {min_duration_sec}–{max_duration_sec}s)")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]

        audio_path = os.path.join(self.data_dir, row["audio_path"])
        text_path  = os.path.join(self.data_dir, row["text_path"])

        # Читаем текст
        try:
            with open(text_path, encoding="utf-8") as f:
                text = f.read().strip().lower()
        except Exception:
            text = ""

        # Читаем аудио
        try:
            with open(audio_path, "rb") as f:
                audio_bytes = f.read()
            features = audio_extractor(audio_bytes)
            if features.shape[0] > self.max_frames:
                features = features[:self.max_frames]
        except Exception:
            features = torch.zeros(10, 80)

        target = torch.tensor(
            tokenizer.encode(text), dtype=torch.long
        )

        return features, None, target, text

In [19]:
import torch
import pandas as pd

DATA_DIR     = "train_data"  
MANIFEST     = DATA_DIR + "/public_youtube1120_hq/public_youtube1120_hq.csv"

# Смотрим что внутри
df = pd.read_csv(MANIFEST, header=None,
                 names=["audio_path", "text_path", "duration"])
print(f"Всего записей: {len(df)}")
print(f"Длительность: min={df.duration.min():.1f} "
      f"max={df.duration.max():.1f} "
      f"mean={df.duration.mean():.1f}")
print(df.head(3))

# Проверяем один сэмпл
ds = OpenSTTDataset(MANIFEST, DATA_DIR, max_samples=5)
feat, _, tgt, text = ds[0]
print(f"Text:     {text}")
print(f"Features: {feat.shape}")

Всего записей: 369245
Длительность: min=0.6 max=11.6 mean=2.8
                                     audio_path  \
0  public_youtube1120_hq/1/54/a013e35242a9.opus   
1  public_youtube1120_hq/0/97/00ac1b10918b.opus   
2  public_youtube1120_hq/e/87/bfcf058d8f7d.opus   

                                     text_path  duration  
0  public_youtube1120_hq/1/54/a013e35242a9.txt     0.639  
1  public_youtube1120_hq/0/97/00ac1b10918b.txt     0.639  
2  public_youtube1120_hq/e/87/bfcf058d8f7d.txt     0.640  
OpenSTTDataset: 5 segments (duration 1.0–15.0s)
Text:     ну у нас же все же хорошо вон смотри митька растёт
Features: torch.Size([352, 80])


In [ ]:
# import torch
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from torch.utils.data import DataLoader, Subset

# # Загружаем манифест
# MANIFEST = "train_data/public_youtube1120_hq/public_youtube1120_hq.csv"
# DATA_ROOT = "train_data"

# df = pd.read_csv(MANIFEST, header=None, names=["audio_path", "text_path", "duration"])

# # Добавляем бины длительности
# bins = [0, 1, 2, 3, 4, 5, 12]
# labels = ['0-1', '1-2', '2-3', '3-4', '4-5', '5+']
# df['dur_bin'] = pd.cut(df['duration'], bins=bins, labels=labels, right=False)

# # Разделение
# train_val_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['dur_bin'])
# val_ratio = 0.1 / 0.9   # 0.1111
# train_df, val_df = train_test_split(train_val_df, test_size=val_ratio, random_state=42, stratify=train_val_df['dur_bin'])

# # Создаём датасеты (предполагаем, что OpenSTTDataset уже определён)
# full_dataset = OpenSTTDataset(MANIFEST, DATA_ROOT, 55_000)   # без max_samples, чтобы взять все данные

# train_dataset = Subset(full_dataset, train_df.index.tolist())
# val_dataset   = Subset(full_dataset, val_df.index.tolist())
# test_dataset  = Subset(full_dataset, test_df.index.tolist())

# # Создаём DataLoader'ы
# batch_size = 8
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
# val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=0)
# test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

NameError: name 'OpenSTTDataset' is not defined

In [18]:
from torch.utils.data import Dataset

# Важно: передаём уже отфильтрованный df напрямую
class OpenSTTDatasetFromDF(Dataset):
    """Версия OpenSTTDataset принимающая готовый DataFrame"""
    def __init__(self, dataframe, data_dir, max_duration_sec=15.0, 
                 max_samples=None, shuffle=True, seed=42):

        df = dataframe.reset_index(drop=True)
        # Перемешиваем и берём нужное количество
        if shuffle:
            df = dataframe.sample(frac=1, random_state=seed).reset_index(drop=True)
        
        if max_samples and max_samples < len(df):
            df = df.iloc[:max_samples]

        self.records = df.reset_index(drop=True)
        self.data_dir   = data_dir
        
        self.max_frames = int(max_duration_sec * 100)
        print(f"OpenSTTDatasetFromDF: {len(self.records)} segments")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]

        text_path  = os.path.join(self.data_dir, row["text_path"])
        audio_path = os.path.join(self.data_dir, row["audio_path"])

        try:
            with open(text_path, encoding="utf-8") as f:
                text = f.read().strip().lower()
        except Exception:
            text = ""

        try:
            with open(audio_path, "rb") as f:
                audio_bytes = f.read()
            features = audio_extractor(audio_bytes)
            if features.shape[0] > self.max_frames:
                features = features[:self.max_frames]
        except Exception:
            features = torch.zeros(10, 80)

        target = torch.tensor(tokenizer.encode(text), dtype=torch.long)
        
        max_target_len = self.max_frames // 4 - 1
        if len(target) > max_target_len:
            target = target[:max_target_len]
            text = tokenizer.decode_greedy(target.tolist())[:len(text)]

        return features, None, target, text


def collate_fn(batch):

    batch = [(f, v, t, txt) for f, v, t, txt in batch if len(t) > 0]
    if len(batch) == 0:
        
        # print("wh")
        return None   # обработать в train loop

    features, videos, targets, texts = zip(*batch)

    feat_lengths   = torch.tensor([f.shape[0] for f in features])
    target_lengths = torch.tensor([len(t) for t in targets])

    feat_padded = torch.zeros(len(features), feat_lengths.max(), features[0].shape[1])
    tgt_padded  = torch.zeros(len(targets), target_lengths.max(), dtype=torch.long)

    for i, (f, t) in enumerate(zip(features, targets)):
        feat_padded[i, :f.shape[0]] = f
        tgt_padded[i,  :len(t)]     = t

    # videos — None для аудио датасетов
    video_padded = None
    valid_videos = [v for v in videos if v is not None]


    if len(valid_videos) > 0:
        vid_lengths  = [v.shape[0] for v in valid_videos]
        max_v        = max(vid_lengths)
        _, C, H, W   = valid_videos[0].shape
        video_padded = torch.zeros(len(videos), max_v, C, H, W)
        valid_idx = [i for i, v in enumerate(videos) if v is not None]
        for i, v in zip(valid_idx, valid_videos):
            video_padded[i, :v.shape[0]] = v

    return feat_padded, video_padded, tgt_padded, feat_lengths, target_lengths, texts


In [5]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

MANIFEST  = "train_data/public_youtube1120_hq/public_youtube1120_hq.csv"
DATA_ROOT = "train_data/"

# ── Загружаем манифест ─────────────────────────────────────
df = pd.read_csv(MANIFEST, header=None,
                 names=["audio_path", "text_path", "duration"])

# Фильтруем слишком короткие (шум) и длинные (не влезут в батч)
df = df[(df["duration"] >= 1.0) & (df["duration"] <= 15.0)]
df = df.reset_index(drop=True)   # ← важно: сбрасываем индекс после фильтрации
print(f"После фильтрации: {len(df)} записей")

# ── Стратификация по длине ─────────────────────────────────
bins   = [0, 2, 4, 6, 8, 16]
labels = ['0-2', '2-4', '4-6', '6-8', '8+']
df['dur_bin'] = pd.cut(df['duration'], bins=bins, labels=labels, right=False)

# ── Разделение 80 / 10 / 10 ───────────────────────────────
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['dur_bin']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['dur_bin']
)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

# Проверяем распределение по бинам
# print("\nРаспределение по длине:")
# for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
#     counts = split_df['dur_bin'].value_counts().sort_index()
#     print(f"  {split_name}: {dict(counts)}")

# ── Датасет и DataLoader'ы ─────────────────────────────────
# Создаём полный датасет БЕЗ перемешивания внутри OpenSTTDataset
# чтобы индексы из train_df/val_df/test_df совпадали


# val_dataset   = OpenSTTDatasetFromDF(val_df,   DATA_ROOT, max_samples=5_000)
# train_dataset = OpenSTTDatasetFromDF(train_df, DATA_ROOT, max_samples=20_000)
# # test_dataset  = OpenSTTDatasetFromDF(test_df,  DATA_ROOT, max_samples=5_000)

# train_loader = DataLoader(
#     train_dataset, batch_size=8, shuffle=True,
#     collate_fn=collate_fn, num_workers=0, pin_memory=True,
# )
# val_loader = DataLoader(
#     val_dataset, batch_size=8, shuffle=False,
#     collate_fn=collate_fn, num_workers=0, pin_memory=True,
# )

# test_loader = DataLoader(
#     test_dataset, batch_size=8, shuffle=False,
#     collate_fn=collate_fn, num_workers=0, pin_memory=True,
# )

После фильтрации: 364839 записей
Train: 291871  Val: 36484  Test: 36484


In [104]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

def openstt_loader_maker(train_samples=20_000, val_samples=5_000, test_samples=5_000, test_included=False):
    '''
    Return train_loader, val_loader, test_loader (if test_included=True) для OpenSTT датасета
    '''
    MANIFEST  = "train_data/public_youtube1120_hq/public_youtube1120_hq.csv"
    DATA_ROOT = "train_data/"

    # ── Загружаем манифест ─────────────────────────────────────
    df = pd.read_csv(MANIFEST, header=None,
                    names=["audio_path", "text_path", "duration"])

    # Фильтруем слишком короткие (шум) и длинные (не влезут в батч)
    df = df[(df["duration"] >= 1.0) & (df["duration"] <= 15.0)]
    df = df.reset_index(drop=True)   # ← важно: сбрасываем индекс после фильтрации
    print(f"После фильтрации: {len(df)} записей")

    # ── Стратификация по длине ─────────────────────────────────
    bins   = [0, 2, 4, 6, 8, 16]
    labels = ['0-2', '2-4', '4-6', '6-8', '8+']
    df['dur_bin'] = pd.cut(df['duration'], bins=bins, labels=labels, right=False)

    if test_included:
        # ── Разделение 80 / 10 / 10 ───────────────────────────────
        train_df, temp_df = train_test_split(
            df, test_size=0.2, random_state=42, stratify=df['dur_bin']
        )
        val_df, test_df = train_test_split(
            temp_df, test_size=0.5, random_state=42, stratify=temp_df['dur_bin']
        )

        print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

        # Проверяем распределение по бинам
        # print("\nРаспределение по длине:")
        # for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        #     counts = split_df['dur_bin'].value_counts().sort_index()
        #     print(f"  {split_name}: {dict(counts)}")

        # ── Датасет и DataLoader'ы ─────────────────────────────────
        # Создаём полный датасет БЕЗ перемешивания внутри OpenSTTDataset
        # чтобы индексы из train_df/val_df/test_df совпадали

        val_dataset   = OpenSTTDatasetFromDF(val_df,   DATA_ROOT, max_samples=val_samples)
        train_dataset = OpenSTTDatasetFromDF(train_df, DATA_ROOT, max_samples=train_samples)
        test_dataset  = OpenSTTDatasetFromDF(test_df,  DATA_ROOT, max_samples=test_samples)

        train_loader = DataLoader(
            train_dataset, batch_size=8, shuffle=True,
            collate_fn=collate_fn, num_workers=0, pin_memory=True,
        )
        val_loader = DataLoader(
            val_dataset, batch_size=8, shuffle=False,
            collate_fn=collate_fn, num_workers=0, pin_memory=True,
        )
        test_loader = DataLoader(
            test_dataset, batch_size=8, shuffle=False,
            collate_fn=collate_fn, num_workers=0, pin_memory=True,
        )

        return train_loader, val_loader, test_loader

    else:
        # ── Разделение 80 / 10 / 10 ───────────────────────────────
        train_df, val_df = train_test_split(
            df, test_size=0.2, random_state=42, stratify=df['dur_bin']
        )
        
        print(f"Train: {len(train_df)}  Val: {len(val_df)}")

        # Проверяем распределение по бинам
        # print("\nРаспределение по длине:")
        # for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        #     counts = split_df['dur_bin'].value_counts().sort_index()
        #     print(f"  {split_name}: {dict(counts)}")

        # ── Датасет и DataLoader'ы ─────────────────────────────────
        # Создаём полный датасет БЕЗ перемешивания внутри OpenSTTDataset
        # чтобы индексы из train_df/val_df/test_df совпадали

        val_dataset   = OpenSTTDatasetFromDF(val_df,   DATA_ROOT, max_samples=val_samples)
        train_dataset = OpenSTTDatasetFromDF(train_df, DATA_ROOT, max_samples=train_samples)

        train_loader = DataLoader(
            train_dataset, batch_size=8, shuffle=True,
            collate_fn=collate_fn, num_workers=0, pin_memory=True,
        )
        val_loader = DataLoader(
            val_dataset, batch_size=8, shuffle=False,
            collate_fn=collate_fn, num_workers=0, pin_memory=True,
        )
        

        return train_loader, val_loader

In [24]:
from pyctcdecode import build_ctcdecoder

# tokenizer.id2char это dict {0: "<blank>", 1: "<unk>", 2: "а", ...}
# pyctcdecode хочет список где blank в позиции 0
labels = [tokenizer.id2char[i] for i in range(len(tokenizer.id2char))]

# Заменяем спец-токены на пустую строку (для blank pyctcdecode так и ожидает)
labels[0] = ""   # blank
# unk и остальные оставляем как есть

beam_decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=None,   # пока без LM
)

print(f"Beam decoder создан, labels: {len(labels)}")

Beam decoder создан, labels: 77


In [ ]:
# kenlm/build/bin/lmplz -o 5 < russian_corpus.txt > ru_5gram.arpa

# # Использовать
# decoder = build_ctcdecoder(
#     labels=tokenizer.id2char,
#     kenlm_model_path="ru_5gram.arpa",
#     alpha=0.5,   # вес LM
#     beta=1.0,    # бонус за длину
# )

In [2]:
# ============================================================
# ОБНОВЛЁННЫЙ TRAIN STEP — гибридная функция потерь
# ============================================================

from pyctcdecode import build_ctcdecoder

# tokenizer.id2char это dict {0: "<blank>", 1: "<unk>", 2: "а", ...}
# pyctcdecode хочет список где blank в позиции 0
labels = [tokenizer.id2char[i] for i in range(len(tokenizer.id2char))]

# Заменяем спец-токены на пустую строку (для blank pyctcdecode так и ожидает)
labels[0] = ""   # blank
# unk и остальные оставляем как есть

beam_decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path="C:/kenlm_models/ru_4gram.bin",   # пока без LM
)

# ----------------------------------------

LAMBDA_CTC = 0.7   # вес CTC в суммарном лоссе
# print(f"Beam decoder создан, labels: {len(labels)}")

def compute_loss(model, feats, targets, feat_lens, tgt_lens,
                 video_tensor=None, use_attention=True):
    """
    Вычисляет гибридный loss: λ·L_CTC + (1-λ)·L_Attention
    """
    B = feats.size(0)

    # Подготовка teacher-forcing токенов для Attention Decoder:
    # вход:  <blank> + target[:-1]  (сдвиг вправо)
    # выход: target  (то что предсказываем)
    if use_attention:
        # BOS = <unk> вместо <blank>
        bos = torch.ones(B, 1, dtype=torch.long, device=feats.device)
        tgt_in  = torch.cat([bos, targets[:, :-1]], dim=1)  # вход декодера
        tgt_out = targets                                     # что предсказываем
    else:
        tgt_in = None

    # print(f'tgt_in в compute_loss: {tgt_in}')
    # здесь с
    
    ctc_logits, attn_logits = model(feats, video_tensor, tgt_in)

    # if ctc_logits == 0.0:
    #     print(f"ctc_logits: {ctc_logits}")
    # CTC loss
    input_lengths = (feat_lens // 4).clamp(max=ctc_logits.size(1))
    loss_ctc = F.ctc_loss(
        ctc_logits.transpose(0, 1),
        targets, input_lengths, tgt_lens,
        blank=0, zero_infinity=True,
    )

    loss_attn = torch.tensor(0.0, device=feats.device)

    # Attention loss (cross-entropy)
    if (attn_logits is not None
            and not torch.isnan(attn_logits).any()
            and not torch.isinf(attn_logits).any()
            and not torch.isnan(loss_ctc)
            and not torch.isinf(loss_ctc)):
        
        # attn_logits: (B, L, vocab) → (B*L, vocab)
        # tgt_out:     (B, L)        → (B*L,)
        # print(f"    attn stats: min={attn_logits.min():.3f} "
        #   f"max={attn_logits.max():.3f} "
        #   f"nan={torch.isnan(attn_logits).sum().item()} "
        #   f"inf={torch.isinf(attn_logits).sum().item()}")

        loss_attn = F.cross_entropy(
            attn_logits.reshape(-1, attn_logits.size(-1)),
            tgt_out.reshape(-1),
            ignore_index=0,   # игнорируем padding
        )
         # Защита от nan в attn loss
        if torch.isnan(loss_attn) or torch.isinf(loss_attn):
            loss_attn = torch.tensor(0.0, device=feats.device)

        
    if loss_attn.item() == 0.0:
        loss   = loss_ctc
    else:
        loss = LAMBDA_CTC * loss_ctc + (1 - LAMBDA_CTC) * loss_attn

    return loss, loss_ctc, loss_attn


# ============================================================
# ОБНОВЛЁННЫЙ finetune_on_loaders
# ============================================================
def finetune_on_loaders(train_loader, val_loader,
                        checkpoint_path="conformer_sova_1.pt",
                        n_epochs=5, lr=1e-4,
                        freeze_encoder_epochs=1,
                        use_attention=True,       # ← новый флаг
                        save_path="conformer_finetuned.pt"):

    print("=" * 60)
    print(f"Fine-tune: {n_epochs} эпох, lr={lr}, attention={use_attention}")
    print("=" * 60)

    ckpt  = torch.load(checkpoint_path, map_location=DEVICE)
    model = AVConformerCTC(vocab_size=ckpt["vocab_size"]).to(DEVICE)

    # Загружаем только те веса которые совпадают
    # (нужно если старый чекпоинт без attention decoder)
    state = ckpt["model_state"]
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print(f"  Новые параметры (инициализированы случайно): {len(missing)}")
    print(f"Загружен: epoch={ckpt['epoch']}, WER={ckpt['wer']:.3f}\n")

    optimizer = AdamW([
        {"params": model.subsampling.parameters(), "lr": lr * 0.1},
        {"params": model.conformer.parameters(),   "lr": lr * 0.1},
        {"params": model.pos_enc.parameters(),     "lr": lr * 0.1},
        {"params": model.video_enc.parameters(),   "lr": lr},
        {"params": model.fusion.parameters(),      "lr": lr},
        {"params": model.ctc_head.parameters(),    "lr": lr},
        {"params": model.decoder.parameters(),     "lr": lr},      # ← новое
        {"params": model.decoder_embed.parameters(),"lr": lr},     # ← новое
        {"params": model.decoder_pos.parameters(), "lr": lr},      # ← новое
        {"params": model.output_proj.parameters(), "lr": lr},      # ← новое
    ], weight_decay=1e-2)

    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
    best_wer  = float("inf")

    for epoch in range(n_epochs):

        if epoch < freeze_encoder_epochs:
            for name, param in model.named_parameters():
                param.requires_grad = not any(
                    k in name for k in ["subsampling", "conformer", "pos_enc"]
                )
            if epoch == 0:
                n = sum(p.numel() for p in model.parameters()
                        if p.requires_grad) / 1e6
                print(f"  🔒 Энкодер заморожен | обучаемых: {n:.1f}M\n")
        else:
            for param in model.parameters():
                param.requires_grad = True
            if epoch == freeze_encoder_epochs:
                n = sum(p.numel() for p in model.parameters()) / 1e6
                print(f"  🔓 Энкодер разморожен | всего: {n:.1f}M\n")

        # ── TRAIN ──────────────────────────────────────────────
        model.train()
        train_losses, ctc_losses, attn_losses = [], [], []
        t0 = time.time()

        for step, batch_data in enumerate(train_loader):

            if batch_data is None:
                continue
            feats, videos, targets, feat_lens, tgt_lens, texts = batch_data
            feats   = feats.to(DEVICE)
            targets = targets.to(DEVICE)

            video_tensor = None
            if videos is not None:
                valid = [v for v in videos if v is not None]
                if valid:
                    max_t = max(v.shape[0] for v in valid)
                    C, H, W = valid[0].shape[1:]
                    vt = torch.zeros(len(videos), max_t, C, H, W)
                    vi = [i for i, v in enumerate(videos) if v is not None]
                    for i, v in zip(vi, valid):
                        vt[i, :v.shape[0]] = v
                    video_tensor = vt.to(DEVICE)

            loss, l_ctc, l_attn = compute_loss(
                model, feats, targets, feat_lens, tgt_lens,
                video_tensor, use_attention=use_attention,
            )

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"  ⚠ step {step}: loss={loss.item():.4f} "
                     f"| ctc={l_ctc.item():.4f} "
                     f"| attn={l_attn.item():.4f}")
                print(f"  feats: min={feats.min():.3f} max={feats.max():.3f}")
                print(f"  targets shape: {targets.shape}, "
                     f"tgt_lens: {tgt_lens[:4]}, "
                    f"feat_lens//4: {(feat_lens//4)[:4]}")

                continue

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            train_losses.append(loss.item())
            ctc_losses.append(l_ctc.item())
            attn_losses.append(l_attn.item())

            # print(s)
            if step % 100 == 0:
                print(f"  [{epoch+1}/{n_epochs}] step {step:4d}/{len(train_loader)}"
                      f" | loss {loss.item():.4f}"
                      f" | ctc {l_ctc.item():.4f}"
                      f" | attn {l_attn.item():.4f}"
                      f" | {time.time()-t0:.0f}s",
                      flush=True)

        # ── VALIDATION ─────────────────────────────────────────
        model.eval()
        wer_scores_greedy, wer_scores_beam = [], []
        sample_printed = 0
        val_losses, val_ctc_losses, val_attn_losses = [], [], []
        batches_valid, batches_seen = 0, 0
        with torch.no_grad():
            # for feats, videos, targets, feat_lens, tgt_lens, texts in val_loader:

            for batch_data in val_loader:
                batches_seen += 1
                if batch_data is None:
                    continue

                feats, videos, targets, feat_lens, tgt_lens, texts = batch_data
                batches_valid += 1
                feats   = feats.to(DEVICE)
                targets = targets.to(DEVICE)

                # На валидации — только CTC (быстро и без teacher forcing)
                ctc_logits, attn_logits = model(feats, video_frames=None,
                                      target_tokens=None)
                input_lengths = (feat_lens // 4).clamp(max=ctc_logits.size(1))

                ctc_loss_val = F.ctc_loss(
                    ctc_logits.transpose(0, 1),
                    targets, input_lengths, tgt_lens,
                    blank=0, zero_infinity=True,
                )
                ctc_inf_or_nan = torch.isnan(ctc_loss_val) or torch.isinf(ctc_loss_val)
                if not ctc_inf_or_nan:
                    val_ctc_losses.append(ctc_loss_val.item())

                loss_attn_val = torch.tensor(0.0, device=DEVICE)
                if attn_logits is not None:
                    loss_attn_val = F.cross_entropy(
                        attn_logits.reshape(-1, attn_logits.size(-1)),
                        targets.reshape(-1),
                        ignore_index=0,
                    )
                attn_inf_or_nan = torch.isnan(loss_attn_val) or torch.isinf(loss_attn_val)
                if not attn_inf_or_nan:
                    val_attn_losses.append(loss_attn_val.item())

                # Общий val loss = λ·CTC + (1-λ)·Attention (та же формула что в train)
                if not ctc_inf_or_nan and not attn_inf_or_nan:
                    loss_val = LAMBDA_CTC * ctc_loss_val + (1 - LAMBDA_CTC) * loss_attn_val
                    val_losses.append(loss_val.item())
                # if loss_attn_val.item() > 0:
                # else:
                    # loss_val = ctc_loss_val


                # greedy
                preds = ctc_logits.argmax(dim=-1).cpu()

                # beam search
                log_probs_np = ctc_logits.cpu().numpy()

                for i, pred_ids in enumerate(preds):
                    hyp_greedy = tokenizer.decode_greedy(pred_ids.tolist())
                    wer_greedy = wer(texts[i], hyp_greedy)
                    wer_scores_greedy.append(wer_greedy)

                    # beam search используем реальную длину
                    T_real = input_lengths[i]
                    hyp_beam = beam_decoder.decode(
                        log_probs_np[i, :T_real, :],
                        beam_width=50
                        )
                    wer_beam = wer(texts[i], hyp_beam)
                    wer_scores_beam.append(wer_beam)

                    if sample_printed < 3:
                        print(f"  REF: {texts[i]}")
                        print(f"  GREEDY: {hyp_greedy}")
                        print(f"  BEAM: {hyp_beam}")
                        print(f"  WER greedy={wer_greedy:.3f}, beam={wer_beam:.3f}\n")
                        sample_printed += 1

        avg_train = np.mean(train_losses) if train_losses else float("inf")
        avg_val   = np.mean(val_losses)   if val_losses   else float("inf")
        avg_ctc   = np.mean(ctc_losses)   if ctc_losses   else float("inf")
        avg_attn  = np.mean(attn_losses)  if attn_losses  else float("inf")
        
        avg_wer_greedy   = np.mean(wer_scores_greedy)   if wer_scores_greedy   else float("inf")
        avg_wer_beam     = np.mean(wer_scores_beam)     if wer_scores_beam     else float("inf")
        scheduler.step()

        print(f"\n{'─'*60}")
        print(f"Epoch {epoch+1:2d}/{n_epochs} | "
              f"loss={avg_train:.4f} | ctc={avg_ctc:.4f} | attn={avg_attn:.4f} | "
              f"val={avg_val:.4f} | WER greedy={avg_wer_greedy:.3f}, WER beam={avg_wer_beam:.3f}")
        print(f"{'─'*60}\n")

        # print(f"Батчей всего: {batches_seen}, валидных: {batches_valid}")
        # print(f"val_losses: {len(val_losses)}, wer_scores_greedy: {len(wer_scores_greedy)}, wer_scores_beam: {len(wer_scores_beam)}")
        if avg_wer_beam < best_wer:
            best_wer = avg_wer_beam
            torch.save({
                "epoch":       epoch + 1,
                "model_state": model.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "val_loss":    avg_val,
                "wer":         avg_wer_beam,
                "vocab_size":  VOCAB_SIZE,
                "stage":       "hybrid_ctc_attention",
            }, save_path)
            print(f"  ✓ Сохранён → {save_path} (WER={avg_wer_beam:.3f})\n")

    return model

Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
No known unigrams provided, decoding results might be a lot worse.


In [4]:
# проверить существование директориии

import os
os.path.exists("kenLM/ru_4gram.bin")

True

In [6]:
import shutil
import os

# Создаём папку с латинским именем рядом с диском C
target_dir = "C:/kenlm_models"
os.makedirs(target_dir, exist_ok=True)

# Копируем модель туда
src  = "kenLM/ru_4gram.bin"
dst  = f"{target_dir}/ru_4gram.bin"

if not os.path.exists(dst):
    print(f"Копируем {os.path.getsize(src) / 1024**3:.2f} GB...")
    shutil.copy(src, dst)
    print(f"Готово: {dst}")
else:
    print(f"Уже есть: {dst}")

# Используем
from pyctcdecode import build_ctcdecoder

beam_decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=dst,   # путь без кириллицы
)

print("Beam decoder с LM создан!")

Уже есть: C:/kenlm_models/ru_4gram.bin


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
No known unigrams provided, decoding results might be a lot worse.


Beam decoder с LM создан!


In [5]:
import os
size = os.path.getsize("kenLM/ru_4gram.bin")
print(f"Размер: {size / 1024**3:.2f} GB")

Размер: 10.81 GB


In [7]:
import kenlm

ModuleNotFoundError: No module named 'kenlm'

In [ ]:
# val_ctc_losses, val_attn_losses, val_losses, wer_scores = [], [], [], []
#         sample_printed = 0

#         batches_seen, batches_valid = 0, 0
#         with torch.no_grad():

#             # for feats, videos, targets, feat_lens, tgt_lens, texts in val_loader:
#             for batch_data in val_loader:
#                 batches_seen += 1
#                 if batch_data is None:
#                     continue
#                 batches_valid += 1
#                 feats, videos, targets, feat_lens, tgt_lens, texts = batch_data
                
#                 feats   = feats.to(DEVICE)
#                 targets = targets.to(DEVICE)

#                 # На валидации — только CTC (быстро и без teacher forcing)
#                 ctc_logits, attn_logits = model(feats, video_frames=None,
#                                       target_tokens=None)
#                 input_lengths = (feat_lens // 4).clamp(max=ctc_logits.size(1))

#                 ctc_loss_val = F.ctc_loss(
#                     ctc_logits.transpose(0, 1),
#                     targets, input_lengths, tgt_lens,
#                     blank=0, zero_infinity=True,
#                 )
#                 ctc_inf_or_nan = torch.isnan(ctc_loss_val) or torch.isinf(ctc_loss_val) 
#                 if not ctc_inf_or_nan:
#                     val_ctc_losses.append(ctc_loss_val.item())
                
#                 loss_attn_val = torch.tensor(0.0, device=DEVICE)
#                 if attn_logits is not None:
#                     loss_attn_val = F.cross_entropy(
#                         attn_logits.reshape(-1, attn_logits.size(-1)),
#                         targets.reshape(-1),
#                         ignore_index=0,
#                     )
#                 attn_inf_or_nan = torch.isnan(loss_attn_val) or torch.isinf(loss_attn_val)
#                 if not attn_inf_or_nan:
#                     val_attn_losses.append(loss_attn_val.item())

#                 # Общий val loss = λ·CTC + (1-λ)·Attention (та же формула что в train)
#                 if not ctc_inf_or_nan and not attn_inf_or_nan:
#                     loss_val = LAMBDA_CTC * ctc_loss_val + (1 - LAMBDA_CTC) * loss_attn_val
#                     val_losses.append(loss_val.item())
#                 # if loss_attn_val.item() > 0:
#                 # else:
#                     # loss_val = ctc_loss_val

#                 preds = ctc_logits.argmax(dim=-1).cpu()
#                 for i, pred_ids in enumerate(preds):
#                     hyp = tokenizer.decode_greedy(pred_ids.tolist())
#                     wer_scores.append(wer(texts[i], hyp))
#                     if sample_printed < 3:
#                         print(f"  REF: {texts[i]}")
#                         print(f"  HYP: {hyp}")
#                         print(f"  WER: {wer(texts[i], hyp):.3f}\n")
#                         sample_printed += 1

#         avg_train = np.mean(train_losses) if train_losses else float("inf")
#         avg_ctc   = np.mean(ctc_losses)   if ctc_losses   else float("inf")
#         avg_attn  = np.mean(attn_losses)  if attn_losses  else float("inf")

#         avg_val   = np.mean(val_losses)   if val_losses   else float("inf")
#         avg_attn_val  = np.mean(val_attn_losses)  if val_attn_losses  else float("inf")
#         avg_ctc_val   = np.mean(val_ctc_losses)   if val_ctc_losses   else float("inf")
#         avg_wer   = np.mean(wer_scores)   if wer_scores   else float("inf")
        

In [93]:
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_cv_ru_v5epoch17_best.pt",
    n_epochs              = 5,
    lr                    = 4e-5,
    freeze_encoder_epochs = 1,    # сразу 
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_hybrid.pt",
)

Fine-tune: 5 эпох, lr=4e-05, attention=True
  Новые параметры (инициализированы случайно): 58
Загружен: epoch=4, WER=0.701

  🔒 Энкодер заморожен | обучаемых: 4.4M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


RuntimeError: Boolean value of Tensor with more than one value is ambiguous

In [ ]:
# теперь чтобы 

In [ ]:
# ютуб ещё раз 

In [97]:
from torch.utils.data import ConcatDataset, DataLoader

CHECKPOINTS_DIR = "avsr_conformer/checkpoints"
DATA_ROOT = "train_data/public_youtube1120_hq/"

# OpenSTT датасет (уже разделён через train_df)
train_openstt = OpenSTTDatasetFromDF(
    train_df, DATA_ROOT, max_samples=10_000
)

# YouTube датасет
ds_yt = YouTubeAVDataset(
    metadata_path = "yt_dataset.json",
    language      = "ru",
    max_samples=3_000
)

# Смешиваем: ~55k OpenSTT + ~15k YouTube
ds_combined = ConcatDataset([train_openstt, ds_yt])
print(f"Смешанный датасет: {len(ds_combined)} сегментов")

# Val — только OpenSTT для стабильного бенчмарка
# ds_val = OpenSTTDatasetFromDF(
#     val_df, DATA_ROOT, max_samples=3_000
# )
# val_dataset   = OpenSTTDatasetFromDF(val_df,   DATA_ROOT, max_samples=5_000)

train_loader = DataLoader(
    ds_combined, batch_size=8, shuffle=True,
    collate_fn=collate_fn, num_workers=0, pin_memory=True,
)

# _, val_loader = openstt_loader_maker(max_samples=20_000, test_included=False)
val_loader = DataLoader(
    val_dataset, batch_size=8, shuffle=False,
    collate_fn=collate_fn, num_workers=0, pin_memory=True,
)



OpenSTTDatasetFromDF: 10000 segments
YouTubeAVDataset: 3000 segments (language=ru)
Смешанный датасет: 13000 сегментов


In [98]:
# Запуск
model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_cv_ru_v5epoch17_best.pt",
    n_epochs              = 5, 
    lr                    = 2e-5,          # чуть меньше — уже адаптирован
    freeze_encoder_epochs = 1,             
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_hybrid_mixed.pt",
)

Fine-tune: 5 эпох, lr=2e-05, attention=True
  Новые параметры (инициализированы случайно): 58
Загружен: epoch=4, WER=0.701

  🔒 Энкодер заморожен | обучаемых: 4.4M

  [1/5] step    0/1625 | loss 5.3923 | ctc 5.8218 | attn 4.3899 | 1s


c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
  [1/5] step  100/1625 | loss 2.6542 | ctc 2.3169 | attn 3.4411 | 49s
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
  [1/5] step  200/1625 | loss 4.6828 | ctc 5.3343 | attn 3.1626 | 93s
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
  [1/5] step  300/1625 | loss 0.9301 | ctc 0.0000 | attn 3.1003 | 140s
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
  [1/5] step  500/1625 | loss 0.8827 | ctc 0.0000 | attn 2.9424 | 235s
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
  [1/5] step  600/1625 | loss 2.3195 | ctc 2.1096 | attn 2.8093 | 281s
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
  [1/5] step  700/1625 | loss 0.7682 | ctc 0.0000 | attn 2.5607 | 327s
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
  [1/5] step  800/1625 | loss 5.5610 | ctc 6.8253 | attn 2.6111 | 382s
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
  [1/5] step  900/1625 | loss 0.7664 | ctc 0.0000 | attn 2.5547 | 432s
wh
wh
wh
wh
wh
wh
wh
wh
wh
wh
  [1/5] step 1000/1625 

KeyboardInterrupt: 

In [105]:
train_loader, val_loader = openstt_loader_maker(train_samples=40_000, val_samples=4_000, test_included=False)

После фильтрации: 364839 записей
Train: 291871  Val: 72968
OpenSTTDatasetFromDF: 4000 segments
OpenSTTDatasetFromDF: 40000 segments


In [106]:
# Запуск
model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_cv_ru_v5epoch17_best.pt",
    n_epochs              = 5, 
    lr                    = 2e-5,          # чуть меньше — уже адаптирован
    freeze_encoder_epochs = 1,             
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_hybrid.pt",
)

Fine-tune: 5 эпох, lr=2e-05, attention=True
  Новые параметры (инициализированы случайно): 58
Загружен: epoch=4, WER=0.701

  🔒 Энкодер заморожен | обучаемых: 4.4M

  [1/5] step    0/5000 | loss 3.3384 | ctc 2.8489 | attn 4.4807 | 1s


c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step  100/5000 | loss 2.6758 | ctc 2.3670 | attn 3.3964 | 37s
  [1/5] step  200/5000 | loss 2.8950 | ctc 2.8076 | attn 3.0991 | 74s
  [1/5] step  300/5000 | loss 3.5790 | ctc 3.8517 | attn 2.9428 | 110s
  [1/5] step  400/5000 | loss 4.4086 | ctc 5.0608 | attn 2.8866 | 147s
  [1/5] step  500/5000 | loss 2.8885 | ctc 2.9362 | attn 2.7770 | 184s
  [1/5] step  600/5000 | loss 2.9818 | ctc 3.0868 | attn 2.7368 | 222s
  [1/5] step  700/5000 | loss 3.1499 | ctc 3.3387 | attn 2.7095 | 258s
  [1/5] step  800/5000 | loss 3.6689 | ctc 4.0817 | attn 2.7057 | 295s
  [1/5] step  900/5000 | loss 2.5927 | ctc 2.5702 | attn 2.6453 | 330s
  [1/5] step 1000/5000 | loss 2.1054 | ctc 1.9367 | attn 2.4990 | 365s
  [1/5] step 1100/5000 | loss 2.8637 | ctc 3.0168 | attn 2.5064 | 401s
  [1/5] step 1200/5000 | loss 2.4723 | ctc 2.3866 | attn 2.6722 | 437s
  [1/5] step 1300/5000 | loss 2.5435 | ctc 2.5122 | attn 2.6164 | 472s
  [1/5] step 1400/5000 | loss 2.5144 | ctc 2.4486 | attn 2.6679 | 507s
  [1/5] 

In [107]:
# чуть адаптировалась 
model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_hybrid.pt",
    n_epochs              = 5, 
    lr                    = 3e-5,          # чуть меньше — уже адаптирован
    freeze_encoder_epochs = 0,             
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_hybrid_1.pt",
)

Fine-tune: 5 эпох, lr=3e-05, attention=True
Загружен: epoch=2, WER=0.980

  🔓 Энкодер разморожен | всего: 16.7M

  [1/5] step    0/5000 | loss 2.0773 | ctc 2.1455 | attn 1.9181 | 0s


c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step  100/5000 | loss 1.9791 | ctc 2.0158 | attn 1.8934 | 19s
  [1/5] step  200/5000 | loss 2.3568 | ctc 2.5308 | attn 1.9507 | 37s
  [1/5] step  300/5000 | loss 2.0963 | ctc 2.1172 | attn 2.0475 | 55s
  [1/5] step  400/5000 | loss 1.9409 | ctc 1.8188 | attn 2.2260 | 73s
  [1/5] step  500/5000 | loss 1.8035 | ctc 1.7567 | attn 1.9126 | 91s
  [1/5] step  600/5000 | loss 2.2760 | ctc 2.3880 | attn 2.0146 | 109s
  [1/5] step  700/5000 | loss 2.2950 | ctc 2.3750 | attn 2.1084 | 127s
  [1/5] step  800/5000 | loss 2.1669 | ctc 2.2500 | attn 1.9729 | 145s
  [1/5] step  900/5000 | loss 2.0554 | ctc 2.1243 | attn 1.8944 | 163s
  [1/5] step 1000/5000 | loss 2.1019 | ctc 2.1936 | attn 1.8881 | 181s
  [1/5] step 1100/5000 | loss 2.4648 | ctc 2.6992 | attn 1.9178 | 199s
  [1/5] step 1200/5000 | loss 2.3050 | ctc 2.4312 | attn 2.0107 | 217s
  [1/5] step 1300/5000 | loss 1.8892 | ctc 1.9181 | attn 1.8218 | 235s
  [1/5] step 1400/5000 | loss 1.9942 | ctc 2.0894 | attn 1.7721 | 252s
  [1/5] ste

In [ ]:
# hard to adapt
model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_hybrid.pt",
    n_epochs              = 7, 
    lr                    = 1.5e-5,          # чуть меньше — уже адаптирован
    freeze_encoder_epochs = 2,             
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_hybrid_1.pt",
)

In [ ]:
# адаптировалась
model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_hybrid.pt",
    n_epochs              = 7, 
    lr                    = 4e-5,          # чуть меньше — уже адаптирован
    freeze_encoder_epochs = 0,             
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_hybrid_1.pt",
)

In [ ]:
# эксперимент со смешанным CV + OpenSTT

In [6]:
from torch.utils.data import ConcatDataset
import os

DATA_ROOT_OPENSTT = "train_data/public_youtube1120_hq/"
CV_DATA_DIR       = "train_data/cv-corpus-25.0-2026-03-09/ru"

# Common Voice — 60% от смеси  
ds_cv = CommonVoiceLocalDataset(
    tsv_path    = CV_DATA_DIR + "/validated.tsv",
    clips_dir   = CV_DATA_DIR + "/clips",
    max_samples = 35_000,
)

# OpenSTT — 40% от смеси
ds_openstt = OpenSTTDatasetFromDF(
    train_df, DATA_ROOT_OPENSTT, max_samples=12_000, shuffle=True
)

# Смешиваем — ConcatDataset перемешает их при shuffle=True
ds_combined = ConcatDataset([ds_cv, ds_openstt])
print(f"Смешанный датасет: {len(ds_combined)} сегментов")

# Val оставляем на CV — это стабильный бенчмарк
ds_cv_val = CommonVoiceLocalDataset(
    tsv_path    = CV_DATA_DIR + "/dev.tsv",
    clips_dir   = CV_DATA_DIR + "/clips",
    max_samples = 3_000,
)

train_loader = DataLoader(
    ds_combined, batch_size=8, shuffle=True,
    collate_fn=collate_fn, num_workers=0, pin_memory=True,
)
val_loader = DataLoader(
    ds_cv_val, batch_size=8, shuffle=False,
    collate_fn=collate_fn, num_workers=0, pin_memory=True,
)


C:\Users\Tim\AppData\Local\Temp\ipykernel_14164\3887216578.py:15: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(tsv_path, sep="\t")


CommonVoiceLocalDataset: 35000 segments
OpenSTTDatasetFromDF: 12000 segments
Смешанный датасет: 47000 сегментов
CommonVoiceLocalDataset: 3000 segments


In [ ]:
# 1
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_cv_ru_v5epoch17_best.pt",
    n_epochs              = 5,
    lr                    = 4e-5,
    freeze_encoder_epochs = 1,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_cv_openstt_mix.pt",
)

Fine-tune: 5 эпох, lr=4e-05, attention=True
  Новые параметры (инициализированы случайно): 58
Загружен: epoch=4, WER=0.701

  🔒 Энкодер заморожен | обучаемых: 4.4M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/5000 | loss 1.4533 | ctc 0.1821 | attn 4.4195 | 2s
  [1/5] step  100/5000 | loss 1.3485 | ctc 0.5191 | attn 3.2836 | 26s
  [1/5] step  200/5000 | loss 1.0860 | ctc 0.2784 | attn 2.9703 | 49s
  [1/5] step  300/5000 | loss 1.2429 | ctc 0.5046 | attn 2.9654 | 72s
  [1/5] step  400/5000 | loss 1.1371 | ctc 0.4880 | attn 2.6517 | 95s
  [1/5] step  500/5000 | loss 1.0473 | ctc 0.3339 | attn 2.7118 | 117s
  [1/5] step  600/5000 | loss 1.1911 | ctc 0.5759 | attn 2.6266 | 141s
  [1/5] step  700/5000 | loss 1.1205 | ctc 0.4905 | attn 2.5906 | 163s
  [1/5] step  800/5000 | loss 1.2893 | ctc 0.7151 | attn 2.6293 | 187s
  [1/5] step  900/5000 | loss 1.0416 | ctc 0.3795 | attn 2.5865 | 209s
  [1/5] step 1000/5000 | loss 0.9461 | ctc 0.2471 | attn 2.5773 | 234s
  [1/5] step 1100/5000 | loss 1.0571 | ctc 0.4189 | attn 2.5463 | 257s
  [1/5] step 1200/5000 | loss 1.0081 | ctc 0.3605 | attn 2.5193 | 280s
  [1/5] step 1300/5000 | loss 1.0373 | ctc 0.3576 | attn 2.6233 | 303s
  [1/5] step

In [13]:
# 2 
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_cv_openstt_mix_safe.pt",
    n_epochs              = 5,
    lr                    = 1.42e-5,
    freeze_encoder_epochs = 2,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_cv_openstt_mix_safe_1.pt",
)

Fine-tune: 5 эпох, lr=1.42e-05, attention=True
Загружен: epoch=3, WER=0.702

  🔒 Энкодер заморожен | обучаемых: 4.4M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/5000 | loss 0.6360 | ctc 0.2434 | attn 1.8136 | 3s
  [1/5] step  100/5000 | loss 0.8210 | ctc 0.4777 | attn 1.8509 | 35s
  [1/5] step  200/5000 | loss 0.8140 | ctc 0.4377 | attn 1.9429 | 67s
  [1/5] step  300/5000 | loss 0.7193 | ctc 0.3460 | attn 1.8392 | 97s
  [1/5] step  400/5000 | loss 0.7213 | ctc 0.3423 | attn 1.8586 | 127s
  [1/5] step  500/5000 | loss 0.7248 | ctc 0.3118 | attn 1.9640 | 157s
  [1/5] step  600/5000 | loss 0.6928 | ctc 0.2943 | attn 1.8880 | 187s
  [1/5] step  700/5000 | loss 0.6859 | ctc 0.3240 | attn 1.7717 | 217s
  [1/5] step  800/5000 | loss 0.7100 | ctc 0.3485 | attn 1.7947 | 247s
  [1/5] step  900/5000 | loss 0.6622 | ctc 0.2770 | attn 1.8177 | 277s
  [1/5] step 1000/5000 | loss 0.8225 | ctc 0.4874 | attn 1.8278 | 306s
  [1/5] step 1100/5000 | loss 0.5521 | ctc 0.1977 | attn 1.6153 | 336s
  [1/5] step 1200/5000 | loss 0.7992 | ctc 0.4409 | attn 1.8739 | 365s
  [1/5] step 1300/5000 | loss 0.7027 | ctc 0.4018 | attn 1.6056 | 394s
  [1/5] ste

In [15]:
model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_cv_openstt_mix_safe.pt",
    n_epochs              = 5,
    lr                    = 1e-5,
    freeze_encoder_epochs = 0,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_specaug.pt",
)

Fine-tune: 5 эпох, lr=1e-05, attention=True
Загружен: epoch=3, WER=0.702

  🔓 Энкодер разморожен | всего: 16.7M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/5000 | loss 0.6952 | ctc 0.3665 | attn 1.6813 | 1s
  [1/5] step  100/5000 | loss 0.6915 | ctc 0.3010 | attn 1.8629 | 34s
  [1/5] step  200/5000 | loss 0.7095 | ctc 0.3425 | attn 1.8104 | 64s
  [1/5] step  300/5000 | loss 0.7793 | ctc 0.4564 | attn 1.7480 | 95s
  [1/5] step  400/5000 | loss 0.6475 | ctc 0.2541 | attn 1.8275 | 127s
  [1/5] step  500/5000 | loss 0.7870 | ctc 0.4146 | attn 1.9039 | 158s
  [1/5] step  600/5000 | loss 0.7097 | ctc 0.3696 | attn 1.7300 | 187s
  [1/5] step  700/5000 | loss 0.8038 | ctc 0.4182 | attn 1.9604 | 217s
  [1/5] step  800/5000 | loss 0.6636 | ctc 0.2783 | attn 1.8196 | 248s
  [1/5] step  900/5000 | loss 0.7668 | ctc 0.4393 | attn 1.7491 | 279s
  [1/5] step 1000/5000 | loss 0.9908 | ctc 0.6955 | attn 1.8767 | 310s
  [1/5] step 1100/5000 | loss 0.8124 | ctc 0.4559 | attn 1.8817 | 341s
  [1/5] step 1200/5000 | loss 0.9838 | ctc 0.6986 | attn 1.8393 | 370s
  [1/5] step 1300/5000 | loss 0.8644 | ctc 0.5414 | attn 1.8335 | 399s
  [1/5] ste

KeyboardInterrupt: 

In [33]:
# specaug 

In [ ]:
# 25/15
model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_cv_openstt_mix_safe.pt",
    n_epochs              = 5,
    lr                    = 1.2e-5,
    freeze_encoder_epochs = 0,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_specaug_1.pt",
)

Fine-tune: 5 эпох, lr=1.2e-05, attention=True
Загружен: epoch=3, WER=0.702

  🔓 Энкодер разморожен | всего: 16.7M

  [1/5] step    0/5000 | loss 0.7023 | ctc 0.3655 | attn 1.7126 | 0s


c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step  100/5000 | loss 0.7743 | ctc 0.4208 | attn 1.8348 | 24s
  [1/5] step  200/5000 | loss 0.6448 | ctc 0.2407 | attn 1.8572 | 47s
  [1/5] step  300/5000 | loss 0.6139 | ctc 0.2401 | attn 1.7353 | 71s
  [1/5] step  400/5000 | loss 0.6935 | ctc 0.3225 | attn 1.8067 | 95s
  [1/5] step  500/5000 | loss 0.7280 | ctc 0.3728 | attn 1.7936 | 120s
  [1/5] step  600/5000 | loss 0.8850 | ctc 0.5536 | attn 1.8794 | 144s
  [1/5] step  700/5000 | loss 0.9070 | ctc 0.5556 | attn 1.9611 | 168s
  [1/5] step  800/5000 | loss 0.7563 | ctc 0.4254 | attn 1.7489 | 193s
  [1/5] step  900/5000 | loss 0.9764 | ctc 0.5903 | attn 2.1349 | 218s
  [1/5] step 1000/5000 | loss 0.6011 | ctc 0.2263 | attn 1.7256 | 246s
  [1/5] step 1100/5000 | loss 0.6847 | ctc 0.3573 | attn 1.6670 | 273s
  [1/5] step 1200/5000 | loss 0.7248 | ctc 0.3874 | attn 1.7369 | 302s
  [1/5] step 1300/5000 | loss 0.6823 | ctc 0.3781 | attn 1.5947 | 330s
  [1/5] step 1400/5000 | loss 0.9682 | ctc 0.6874 | attn 1.8105 | 370s
  [1/5] st

In [ ]:
# 25/15
model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_specaug_1.pt",
    n_epochs              = 5,
    lr                    = 2.35e-5,
    freeze_encoder_epochs = 0,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_specaug_2.pt",
)

Fine-tune: 5 эпох, lr=2.35e-05, attention=True
Загружен: epoch=3, WER=0.581

  🔓 Энкодер разморожен | всего: 16.7M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/5000 | loss 0.5110 | ctc 0.3467 | attn 1.0037 | 3s
  [1/5] step  100/5000 | loss 0.6341 | ctc 0.4346 | attn 1.2326 | 31s
  [1/5] step  200/5000 | loss 0.6755 | ctc 0.4545 | attn 1.3383 | 62s
  [1/5] step  300/5000 | loss 0.9995 | ctc 0.8299 | attn 1.5083 | 91s
  [1/5] step  400/5000 | loss 0.4077 | ctc 0.2839 | attn 0.7791 | 119s
  [1/5] step  500/5000 | loss 0.5799 | ctc 0.4078 | attn 1.0964 | 146s
  [1/5] step  600/5000 | loss 0.4367 | ctc 0.3001 | attn 0.8463 | 172s
  [1/5] step  700/5000 | loss 0.9564 | ctc 0.7468 | attn 1.5851 | 198s
  [1/5] step  800/5000 | loss 0.6819 | ctc 0.4634 | attn 1.3372 | 225s
  [1/5] step  900/5000 | loss 1.1396 | ctc 1.0335 | attn 1.4581 | 251s
  [1/5] step 1000/5000 | loss 0.5527 | ctc 0.4117 | attn 0.9756 | 278s
  [1/5] step 1100/5000 | loss 0.5896 | ctc 0.4335 | attn 1.0581 | 304s
  [1/5] step 1200/5000 | loss 0.6385 | ctc 0.4781 | attn 1.1196 | 330s
  [1/5] step 1300/5000 | loss 0.4530 | ctc 0.2659 | attn 1.0142 | 357s
  [1/5] ste

In [31]:
# 30/20 - train
model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_specaug_2.pt",
    n_epochs              = 5,
    lr                    = 2.48e-5,
    freeze_encoder_epochs = 0,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_specaug_3.pt",
)

Fine-tune: 5 эпох, lr=2.48e-05, attention=True
Загружен: epoch=5, WER=0.580

  🔓 Энкодер разморожен | всего: 16.7M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/6250 | loss 0.4316 | ctc 0.3685 | attn 0.6210 | 0s
  [1/5] step  100/6250 | loss 0.4603 | ctc 0.3903 | attn 0.6704 | 28s
  [1/5] step  200/6250 | loss 0.4107 | ctc 0.3360 | attn 0.6348 | 56s
  [1/5] step  300/6250 | loss 0.3805 | ctc 0.3236 | attn 0.5514 | 83s
  [1/5] step  400/6250 | loss 0.4732 | ctc 0.3858 | attn 0.7353 | 111s
  [1/5] step  500/6250 | loss 0.3411 | ctc 0.2733 | attn 0.5444 | 140s
  [1/5] step  600/6250 | loss 0.4947 | ctc 0.4279 | attn 0.6950 | 167s
  [1/5] step  700/6250 | loss 0.4401 | ctc 0.3600 | attn 0.6803 | 196s
  [1/5] step  800/6250 | loss 0.4602 | ctc 0.4158 | attn 0.5933 | 224s
  [1/5] step  900/6250 | loss 0.5236 | ctc 0.4447 | attn 0.7604 | 253s
  [1/5] step 1000/6250 | loss 0.3826 | ctc 0.3022 | attn 0.6236 | 281s
  [1/5] step 1100/6250 | loss 0.6354 | ctc 0.6055 | attn 0.7252 | 309s
  [1/5] step 1200/6250 | loss 0.4578 | ctc 0.3734 | attn 0.7107 | 338s
  [1/5] step 1300/6250 | loss 0.5550 | ctc 0.4619 | attn 0.8346 | 366s
  [1/5] ste

In [32]:
# 4 30/20 - train
model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_specaug_3.pt",
    n_epochs              = 5,
    lr                    = 5.48e-5,
    freeze_encoder_epochs = 0,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_specaug_4.pt",
)

Fine-tune: 5 эпох, lr=5.48e-05, attention=True
Загружен: epoch=5, WER=0.578

  🔓 Энкодер разморожен | всего: 16.7M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/6250 | loss 0.4487 | ctc 0.3905 | attn 0.6234 | 1s
  [1/5] step  100/6250 | loss 0.4509 | ctc 0.3790 | attn 0.6665 | 23s
  [1/5] step  200/6250 | loss 0.4051 | ctc 0.3745 | attn 0.4968 | 43s
  [1/5] step  300/6250 | loss 0.2955 | ctc 0.2463 | attn 0.4433 | 63s
  [1/5] step  400/6250 | loss 0.6059 | ctc 0.5778 | attn 0.6903 | 82s
  [1/5] step  500/6250 | loss 0.5028 | ctc 0.4663 | attn 0.6122 | 103s
  [1/5] step  600/6250 | loss 0.5307 | ctc 0.4067 | attn 0.9025 | 123s
  [1/5] step  700/6250 | loss 0.3822 | ctc 0.3508 | attn 0.4765 | 142s
  [1/5] step  800/6250 | loss 0.5564 | ctc 0.5531 | attn 0.5663 | 162s
  [1/5] step  900/6250 | loss 0.3032 | ctc 0.2532 | attn 0.4533 | 182s
  [1/5] step 1000/6250 | loss 0.5747 | ctc 0.5470 | attn 0.6579 | 204s
  [1/5] step 1100/6250 | loss 0.4109 | ctc 0.3637 | attn 0.5525 | 226s
  [1/5] step 1200/6250 | loss 0.3844 | ctc 0.3358 | attn 0.5304 | 251s
  [1/5] step 1300/6250 | loss 0.3555 | ctc 0.2643 | attn 0.6289 | 273s
  [1/5] step

In [ ]:
# 5 30/25 - common_voice/open_stt train
model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_specaug_4.pt",
    n_epochs              = 5,
    lr                    = 5.48e-5,
    freeze_encoder_epochs = 0,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_specaug_5.pt",
)

Fine-tune: 5 эпох, lr=5.48e-05, attention=True
Загружен: epoch=5, WER=0.576

  🔓 Энкодер разморожен | всего: 16.7M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/6250 | loss 0.4421 | ctc 0.3848 | attn 0.6141 | 0s
  [1/5] step  100/6250 | loss 0.2413 | ctc 0.2356 | attn 0.2583 | 26s
  [1/5] step  200/6250 | loss 0.3297 | ctc 0.2941 | attn 0.4367 | 56s
  [1/5] step  300/6250 | loss 0.4528 | ctc 0.4570 | attn 0.4403 | 86s
  [1/5] step  400/6250 | loss 0.5025 | ctc 0.4851 | attn 0.5548 | 115s
  [1/5] step  500/6250 | loss 0.8175 | ctc 0.9022 | attn 0.5634 | 144s
  [1/5] step  600/6250 | loss 0.3676 | ctc 0.3725 | attn 0.3531 | 168s
  [1/5] step  700/6250 | loss 0.5555 | ctc 0.5337 | attn 0.6209 | 198s
  [1/5] step  800/6250 | loss 0.4310 | ctc 0.4256 | attn 0.4470 | 229s
  [1/5] step  900/6250 | loss 0.3889 | ctc 0.3510 | attn 0.5028 | 259s
  [1/5] step 1000/6250 | loss 0.6830 | ctc 0.6933 | attn 0.6522 | 290s
  [1/5] step 1100/6250 | loss 0.2671 | ctc 0.2747 | attn 0.2442 | 323s
  [1/5] step 1200/6250 | loss 0.2296 | ctc 0.2147 | attn 0.2745 | 354s
  [1/5] step 1300/6250 | loss 0.2535 | ctc 0.2556 | attn 0.2471 | 384s
  [1/5] ste

KeyboardInterrupt: 

In [16]:
# 5 31/32/3.500 - common_voice/open_stt train
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_specaug_5.pt",
    n_epochs              = 10,
    lr                    = 7.2e-5,
    freeze_encoder_epochs = 0,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_specaug_6.pt",
)

Fine-tune: 10 эпох, lr=7.2e-05, attention=True
Загружен: epoch=3, WER=0.576

  🔓 Энкодер разморожен | всего: 16.7M

  [1/10] step    0/7875 | loss 0.8140 | ctc 0.8372 | attn 0.7443 | 1s
  [1/10] step  100/7875 | loss 1.2116 | ctc 1.2475 | attn 1.1038 | 25s
  [1/10] step  200/7875 | loss 0.7458 | ctc 0.7589 | attn 0.7065 | 48s
  [1/10] step  300/7875 | loss 1.4430 | ctc 1.5482 | attn 1.1273 | 71s
  [1/10] step  400/7875 | loss 0.5048 | ctc 0.5100 | attn 0.4893 | 94s
  [1/10] step  500/7875 | loss 0.7444 | ctc 0.7681 | attn 0.6732 | 117s
  [1/10] step  600/7875 | loss 0.7393 | ctc 0.7712 | attn 0.6435 | 140s
  [1/10] step  700/7875 | loss 0.8903 | ctc 0.8965 | attn 0.8715 | 164s
  [1/10] step  800/7875 | loss 1.1743 | ctc 1.2238 | attn 1.0258 | 187s
  [1/10] step  900/7875 | loss 1.0175 | ctc 1.0979 | attn 0.7763 | 211s
  [1/10] step 1000/7875 | loss 0.6836 | ctc 0.6741 | attn 0.7119 | 235s
  [1/10] step 1100/7875 | loss 0.8703 | ctc 0.9263 | attn 0.7022 | 258s
  [1/10] step 1200/7875 | 

KeyboardInterrupt: 

In [ ]:
bst_chkpt = ""
if 

In [10]:
# 5 30/25 - common_voice/open_stt train

CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_specaug_5.pt",
    n_epochs              = 5,
    lr                    = 5.48e-5,
    freeze_encoder_epochs = 0,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_specaug_6.pt",
)

Fine-tune: 5 эпох, lr=5.48e-05, attention=True
Загружен: epoch=3, WER=0.576

  🔓 Энкодер разморожен | всего: 16.7M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/6875 | loss 1.0034 | ctc 1.0817 | attn 0.7686 | 3s
  [1/5] step  100/6875 | loss 1.0568 | ctc 1.0894 | attn 0.9588 | 35s
  [1/5] step  200/6875 | loss 1.1472 | ctc 1.2107 | attn 0.9566 | 63s
  [1/5] step  300/6875 | loss 1.1482 | ctc 1.2148 | attn 0.9485 | 90s
  [1/5] step  400/6875 | loss 0.8343 | ctc 0.8532 | attn 0.7778 | 117s
  [1/5] step  500/6875 | loss 0.6840 | ctc 0.7080 | attn 0.6120 | 144s
  [1/5] step  600/6875 | loss 0.7735 | ctc 0.7594 | attn 0.8157 | 171s
  [1/5] step  700/6875 | loss 0.8016 | ctc 0.8354 | attn 0.7003 | 198s
  [1/5] step  800/6875 | loss 1.0010 | ctc 1.0922 | attn 0.7275 | 224s
  [1/5] step  900/6875 | loss 0.6668 | ctc 0.6738 | attn 0.6456 | 251s
  [1/5] step 1000/6875 | loss 0.8087 | ctc 0.7478 | attn 0.9914 | 277s
  [1/5] step 1100/6875 | loss 0.8542 | ctc 0.8419 | attn 0.8910 | 304s
  [1/5] step 1200/6875 | loss 0.7174 | ctc 0.7228 | attn 0.7015 | 329s
  [1/5] step 1300/6875 | loss 0.5734 | ctc 0.5979 | attn 0.4998 | 355s
  [1/5] ste

In [ ]:
# 5 30/25 - common_voice/open_stt train

CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_specaug_5.pt",
    n_epochs              = 5,
    lr                    = 5.48e-5,
    freeze_encoder_epochs = 0,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_specaug_6.pt",
)

In [13]:
# 5 35/12 - common_voice/open_stt train

CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_specaug_5.pt",
    n_epochs              = 5,
    lr                    = 1.2e-4,
    freeze_encoder_epochs = 0,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_specaug_6.pt",
)

Fine-tune: 5 эпох, lr=0.00012, attention=True
Загружен: epoch=3, WER=0.576

  🔓 Энкодер разморожен | всего: 16.7M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/5875 | loss 1.2098 | ctc 1.3277 | attn 0.8560 | 1s
  [1/5] step  100/5875 | loss 1.1635 | ctc 1.1939 | attn 1.0724 | 36s
  [1/5] step  200/5875 | loss 1.0553 | ctc 1.1290 | attn 0.8341 | 72s
  [1/5] step  300/5875 | loss 0.6783 | ctc 0.7605 | attn 0.4318 | 111s
  [1/5] step  400/5875 | loss 0.8108 | ctc 0.8061 | attn 0.8251 | 149s
  [1/5] step  500/5875 | loss 0.8512 | ctc 0.8734 | attn 0.7844 | 188s
  [1/5] step  600/5875 | loss 1.3452 | ctc 1.4424 | attn 1.0537 | 228s
  [1/5] step  700/5875 | loss 0.9139 | ctc 0.9334 | attn 0.8555 | 265s
  [1/5] step  800/5875 | loss 0.8862 | ctc 0.9222 | attn 0.7781 | 304s
  [1/5] step  900/5875 | loss 0.9474 | ctc 1.0038 | attn 0.7781 | 340s
  [1/5] step 1000/5875 | loss 0.7534 | ctc 0.8178 | attn 0.5600 | 376s
  [1/5] step 1100/5875 | loss 0.7352 | ctc 0.7678 | attn 0.6373 | 415s
  [1/5] step 1200/5875 | loss 0.8608 | ctc 0.9088 | attn 0.7168 | 454s
  [1/5] step 1300/5875 | loss 0.7084 | ctc 0.7608 | attn 0.5511 | 491s
  [1/5] st

KeyboardInterrupt: 

In [ ]:
# обуч язык мод KenLM

In [7]:
# 35/12 - common_voice/open_stt train

# посмотреть ошибку модели на с kenLM
# 
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader          = train_loader,
    val_loader            = val_loader,
    checkpoint_path       = CHECKPOINTS_DIR + "/conformer_specaug_5.pt",
    n_epochs              = 5,
    lr                    = 5.8e-5,
    freeze_encoder_epochs = 0,
    use_attention         = True,
    save_path             = CHECKPOINTS_DIR + "/conformer_specaug_6.pt",
)

Fine-tune: 5 эпох, lr=5.8e-05, attention=True
Загружен: epoch=3, WER=0.576

  🔓 Энкодер разморожен | всего: 16.7M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/5875 | loss 0.7316 | ctc 0.7560 | attn 0.6584 | 3s
  [1/5] step  100/5875 | loss 0.8104 | ctc 0.8677 | attn 0.6388 | 26s
  [1/5] step  200/5875 | loss 0.8352 | ctc 0.8781 | attn 0.7066 | 49s
  [1/5] step  300/5875 | loss 1.2686 | ctc 1.3141 | attn 1.1323 | 73s
  [1/5] step  400/5875 | loss 0.7965 | ctc 0.8675 | attn 0.5834 | 94s
  [1/5] step  500/5875 | loss 0.8764 | ctc 0.9222 | attn 0.7393 | 114s
  [1/5] step  600/5875 | loss 0.8096 | ctc 0.8492 | attn 0.6906 | 135s
  [1/5] step  700/5875 | loss 1.0067 | ctc 1.0828 | attn 0.7781 | 157s
  [1/5] step  800/5875 | loss 0.8256 | ctc 0.8738 | attn 0.6812 | 178s
  [1/5] step  900/5875 | loss 0.7685 | ctc 0.8042 | attn 0.6616 | 200s
  [1/5] step 1000/5875 | loss 0.7241 | ctc 0.7319 | attn 0.7007 | 222s
  [1/5] step 1100/5875 | loss 0.8993 | ctc 0.9337 | attn 0.7963 | 243s
  [1/5] step 1200/5875 | loss 0.8248 | ctc 0.8543 | attn 0.7364 | 265s
  [1/5] step 1300/5875 | loss 0.9533 | ctc 0.9978 | attn 0.8198 | 287s
  [1/5] step

In [ ]:
# # 3
# CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

# model = finetune_on_loaders(
#     train_loader          = train_loader,
#     val_loader            = val_loader,
#     checkpoint_path       = CHECKPOINTS_DIR + "/conformer_cv_ru_v5epoch17_best.pt",
#     n_epochs              = 5,
#     lr                    = 1.3e-5,
#     freeze_encoder_epochs = 3,
#     use_attention         = True,
#     save_path             = CHECKPOINTS_DIR + "/conformer_cv_openstt_mix_safe.pt",
# )

In [ ]:
# open stt - и так с yt


In [ ]:
# не знаю почему batch None
# ctc = 0.0

In [ ]:
print(f"Размер val_dataset: {len(val_dataset)}")

empty_count = 0
for i in range(min(20, len(val_dataset))):
    item = val_dataset[i]
    if item is None:
        empty_count += 1
        continue
    feat, _, tgt, text = item
    if len(tgt) == 0:
        empty_count += 1
    print(f"[{i}] text='{text[:50]}' tgt_len={len(tgt)} feat={feat.shape}")

print(f"\nПустых из первых 20: {empty_count}")

Размер val_dataset: 3000
[0] text='' tgt_len=0 feat=torch.Size([10, 80])
[1] text='' tgt_len=0 feat=torch.Size([10, 80])
[2] text='' tgt_len=0 feat=torch.Size([10, 80])
[3] text='' tgt_len=0 feat=torch.Size([10, 80])
[4] text='' tgt_len=0 feat=torch.Size([10, 80])
[5] text='' tgt_len=0 feat=torch.Size([10, 80])
[6] text='' tgt_len=0 feat=torch.Size([10, 80])
[7] text='' tgt_len=0 feat=torch.Size([10, 80])
[8] text='' tgt_len=0 feat=torch.Size([10, 80])
[9] text='' tgt_len=0 feat=torch.Size([10, 80])
[10] text='' tgt_len=0 feat=torch.Size([10, 80])
[11] text='' tgt_len=0 feat=torch.Size([10, 80])
[12] text='' tgt_len=0 feat=torch.Size([10, 80])
[13] text='' tgt_len=0 feat=torch.Size([10, 80])
[14] text='' tgt_len=0 feat=torch.Size([10, 80])
[15] text='' tgt_len=0 feat=torch.Size([10, 80])
[16] text='' tgt_len=0 feat=torch.Size([10, 80])
[17] text='' tgt_len=0 feat=torch.Size([10, 80])
[18] text='' tgt_len=0 feat=torch.Size([10, 80])
[19] text='' tgt_len=0 feat=torch.Size([10, 80])

Пуст

In [ ]:
# добавить attention decoder в конце

In [ ]:
# conformer_specaug_1

from evaluate_performance import load_model, transcribe_video

# model = load_model("avsr_conformer/checkpoints/conformer_specaug_1.pt")

result = transcribe_video(
        video_path="videos/0511.mp4",
        model=model,
        segment_sec=20.0,
        use_video=True,
        reference_text=None,      # передай строку для подсчёта WER
    )

RuntimeError: Error(s) in loading state_dict for AVConformerCTC:
	Unexpected key(s) in state_dict: "decoder_embed.weight", "decoder_pos.weight", "decoder.layers.0.self_attn.in_proj_weight", "decoder.layers.0.self_attn.in_proj_bias", "decoder.layers.0.self_attn.out_proj.weight", "decoder.layers.0.self_attn.out_proj.bias", "decoder.layers.0.multihead_attn.in_proj_weight", "decoder.layers.0.multihead_attn.in_proj_bias", "decoder.layers.0.multihead_attn.out_proj.weight", "decoder.layers.0.multihead_attn.out_proj.bias", "decoder.layers.0.linear1.weight", "decoder.layers.0.linear1.bias", "decoder.layers.0.linear2.weight", "decoder.layers.0.linear2.bias", "decoder.layers.0.norm1.weight", "decoder.layers.0.norm1.bias", "decoder.layers.0.norm2.weight", "decoder.layers.0.norm2.bias", "decoder.layers.0.norm3.weight", "decoder.layers.0.norm3.bias", "decoder.layers.1.self_attn.in_proj_weight", "decoder.layers.1.self_attn.in_proj_bias", "decoder.layers.1.self_attn.out_proj.weight", "decoder.layers.1

In [ ]:
# from evaluate_performance import load_model, transcribe_video

model = load_model("avsr_conformer/checkpoints/conformer_sova_1.pt")

result = transcribe_video(
        video_path="videos/0511.mp4",
        model=model,
        segment_sec=20.0,
        use_video=False, # требует обучения на аудио видео
        reference_text=None,      # передай строку для подсчёта WER
        decoder=beam_decoder,
    )

Loaded checkpoint: epoch=8, WER=0.882

Transcribing: videos/0511.mp4
  [1/3] Extracting audio...
  [2/3] Splitting into segments...
         12 segments found
  [3/3] Transcribing...
    [00:00:00 – 00:00:20]  всемпривет мея урву бореструшины й я учительм тиматики вот и мысвамяно сегодне неможшко ублолтаем т вуде такой и  три меня чтоом стрим пропубалтась но там ие заявальные темаь что-т промутиматику зачем мужно тс мати еикат юп
    [00:00:19 – 00:00:39]  потимати пеикак не пулюбить и давайте начним се это ва дальше я буду старатцяотвититьна каки-то вашиво просым дам всё что хотите можты спрашивать яно это потеча вот десли вижувашго просо мотрите чтота
    [00:00:38 – 00:00:58]  чт тако я мыэтимати кода ни как е плюбить точе роше просто том что я не наю т не от чать я как я ообще плибил симатик да как это могихполучается л кого-то в дест ви хорошо получается бы стры бега и а не чтонатся поропцимено е побъе ыо пкулкого-то хорошо
    [00:00:57 – 00:01:17]  укого-ты хорошо получается не 

In [ ]:
# попытка с новым yt_dataset

In [5]:
import json
import os
import re
from pathlib import Path

YT_DOWNLOADS = "./yt_downloads"   # папка с .mp4 и .vtt
OLD_JSON     = "./yt_dataset.json"
NEW_JSON     = "./yt_dataset_fixed.json"


import re

def parse_vtt(vtt_path):
    """
    Парсер для YouTube auto-captions с двухстрочной структурой:
      Строка 1: предыдущий контекст (пропускаем)
      Строка 2: новые слова с word-level timing
    """
    with open(vtt_path, encoding="utf-8") as f:
        content = f.read()
    
    # Убираем мета-информацию
    content = re.sub(r'^\s*WEBVTT.*$',  '', content, flags=re.M)
    content = re.sub(r'^\s*Kind:.*$',   '', content, flags=re.M)
    content = re.sub(r'^\s*Language:.*$','', content, flags=re.M)
    
    cleaned_cues = []
    
    # Регулярка для cue с тайм-кодами
    cue_pattern = re.compile(
        r'(\d+):(\d+):(\d+)\.(\d+)\s*-->\s*(\d+):(\d+):(\d+)\.(\d+)[^\n]*\n'
        r'(.*?)(?=\n\n|\n\d+:|\Z)',
        re.DOTALL
    )
    
    for match in cue_pattern.finditer(content):
        h1, m1, s1, ms1 = int(match.group(1)), int(match.group(2)), int(match.group(3)), int(match.group(4))
        h2, m2, s2, ms2 = int(match.group(5)), int(match.group(6)), int(match.group(7)), int(match.group(8))
        body            = match.group(9)
        
        start_sec = h1*3600 + m1*60 + s1 + ms1/1000
        end_sec   = h2*3600 + m2*60 + s2 + ms2/1000
        duration  = end_sec - start_sec
        
        # Пропускаем "пустые" cue-разделители (10мс длительность)
        if duration < 0.1:
            continue
        
        # Разбиваем на строки
        lines = [l.strip() for l in body.split("\n") if l.strip()]
        
        if not lines:
            continue
        
        # YouTube auto-captions: ищем строку с word-level разметкой "<00:00:00.000><c>слово</c>"
        # Она содержит ТОЛЬКО новые слова этого cue
        new_words_line = None
        for line in lines:
            if "<c>" in line:
                new_words_line = line
                break
        
        if new_words_line:
            # Извлекаем слова из word-level разметки
            words = re.findall(r'<c>([^<]+)</c>', new_words_line)
            
            # Первое слово может быть БЕЗ <c> тегов в начале
            # Пример: "в<00:00:01.319><c> лекциях</c>..."
            first_word_match = re.match(r'^([^\s<]+)', new_words_line)
            if first_word_match:
                first_word = first_word_match.group(1).strip()
                if first_word and first_word not in words:
                    words = [first_word] + words
            
            text = " ".join(w.strip() for w in words if w.strip()).lower()
            text = re.sub(r'\s+', ' ', text).strip()
            
            if text:
                cleaned_cues.append({
                    "start_sec": start_sec,
                    "end_sec":   end_sec,
                    "text":      text,
                })
    
    return cleaned_cues

def extract_text_for_segment(cues, seg_start, seg_end):
    """
    Берёт текст из cues которые ПЕРЕСЕКАЮТСЯ с временным окном [seg_start, seg_end].
    """
    matched_texts = []
    
    for cue in cues:
        # Проверяем пересечение окон
        overlap_start = max(cue["start_sec"], seg_start)
        overlap_end   = min(cue["end_sec"],   seg_end)
        
        if overlap_end > overlap_start:
            # Сколько процентов cue попадает в наш сегмент
            cue_duration     = cue["end_sec"] - cue["start_sec"]
            overlap_duration = overlap_end - overlap_start
            
            # Если cue хотя бы на 30% пересекается с сегментом — берём
            if cue_duration > 0 and overlap_duration / cue_duration >= 0.3:
                matched_texts.append(cue["text"])
    
    return " ".join(matched_texts).strip()


# ============================================================
# Основной процесс
# ============================================================

# Загружаем старый датасет
with open(OLD_JSON, encoding="utf-8") as f:
    old_dataset = json.load(f)

print(f"Загружено сегментов: {len(old_dataset)}")

# Группируем сегменты по видео ID
by_video = {}
for entry in old_dataset:
    seg_id = entry["segment_id"]
    video_id = seg_id.split("_")[0]   # "6hwENpQqKP0_0001" → "6hwENpQqKP0"
    by_video.setdefault(video_id, []).append(entry)

print(f"Уникальных видео: {len(by_video)}")

# Кэшируем .vtt файлы (не парсим один и тот же дважды)
vtt_cache = {}

new_dataset = []
empty_count = 0
short_count = 0

for video_id, segments in by_video.items():
    vtt_path = f"{YT_DOWNLOADS}/{video_id}.ru.vtt"
    
    # Пытаемся разные имена .vtt файла
    if not os.path.exists(vtt_path):
        for candidate in [f"{YT_DOWNLOADS}/{video_id}.vtt",
                          f"{YT_DOWNLOADS}/{video_id}.ru-RU.vtt"]:
            if os.path.exists(candidate):
                vtt_path = candidate
                break
        else:
            print(f"  Не найден .vtt для {video_id}")
            continue
    
    if vtt_path not in vtt_cache:
        try:
            vtt_cache[vtt_path] = parse_vtt(vtt_path)
        except Exception as e:
            print(f"  Ошибка парсинга {vtt_path}: {e}")
            continue
    
    cues = vtt_cache[vtt_path]
    
    for entry in segments:
        new_text = extract_text_for_segment(
            cues, entry["start_sec"], entry["end_sec"]
        )
        
        if not new_text:
            empty_count += 1
            continue
        
        if len(new_text.split()) < 2:
            short_count += 1
            continue
        
        new_entry = {
            **entry,
            "transcription": new_text,
        }
        new_dataset.append(new_entry)

print(f"\nИтоги:")
print(f"  Было сегментов:    {len(old_dataset)}")
print(f"  Стало сегментов:   {len(new_dataset)}")
print(f"  Пропущено (пусто): {empty_count}")
print(f"  Пропущено (мало):  {short_count}")

# Сохраняем
with open(NEW_JSON, "w", encoding="utf-8") as f:
    json.dump(new_dataset, f, ensure_ascii=False, indent=2)

print(f"\nСохранено: {NEW_JSON}")

Загружено сегментов: 15108
Уникальных видео: 37
  Не найден .vtt для u
  Не найден .vtt для V7bU
  Не найден .vtt для s
  Не найден .vtt для qYvXk

Итоги:
  Было сегментов:    15108
  Стало сегментов:   12092
  Пропущено (пусто): 1627
  Пропущено (мало):  0

Сохранено: ./yt_dataset_fixed.json


In [3]:
cues = parse_vtt("yt_downloads/_d-QtptazZg.ru.vtt")
for cue in cues[:6]:
    print(f"[{cue['start_sec']:.2f} - {cue['end_sec']:.2f}]: {cue['text']}")

[0.32 - 5.33]: в лекциях шла речь о том что одно из
[5.34 - 7.55]: применений предела это возможность
[7.56 - 10.10]: задавать и рационально вещественные
[10.11 - 13.81]: числа как предел последовательности
[13.82 - 17.63]: рациональных вещественных чисел приведу
[17.64 - 20.87]: такой пример из практики посмотрите вот


In [8]:
from torch.utils.data import ConcatDataset

# DATA_ROOT_OPENSTT = "train_data/public_youtube1120_hq/"
CV_DATA_DIR       = "train_data/cv-corpus-25.0-2026-03-09/ru"

# Common Voice — 60% от смеси  
ds_cv = CommonVoiceLocalDataset(
    tsv_path    = CV_DATA_DIR + "/validated.tsv",
    clips_dir   = CV_DATA_DIR + "/clips",
    max_samples = 30_000,
)

ds_yt = YouTubeAVDataset(
    metadata_path="./yt_dataset_fixed.json",
    use_video=False,    # ВРЕМЕННО без видео
)

# Смешиваем с CV для предотвращения забывания
ds_combined = ConcatDataset([
    ds_cv,  
    ds_yt,    # все 12k
])


# Val оставляем на CV — это стабильный бенчмарк
ds_cv_val = CommonVoiceLocalDataset(
    tsv_path    = CV_DATA_DIR + "/dev.tsv",
    clips_dir   = CV_DATA_DIR + "/clips",
    max_samples = 3_300,
)

from torch.utils.data import WeightedRandomSampler

ds_combined = ConcatDataset([ds_cv, ds_yt])

# Веса: сэмплы в 2 раза вероятнее
weights_cv = [1.0] * len(ds_cv)
weights_yt = [1.7] * len(ds_yt)
weights    = weights_cv + weights_yt

sampler = WeightedRandomSampler(
    weights,
    num_samples=30_000,    # сколько брать на эпоху
    replacement=True,
)

# train_loader = DataLoader(
#     ds_combined,
#     batch_size=8,
#     sampler=sampler,
#     collate_fn=collate_fn,
# )

train_loader = DataLoader(
    ds_combined, batch_size=8, shuffle=False, sampler=sampler,
    collate_fn=collate_fn, num_workers=0, pin_memory=True,
)
val_loader = DataLoader(
    ds_cv_val, batch_size=8, shuffle=False,
    collate_fn=collate_fn, num_workers=0, pin_memory=True,
)

C:\Users\Tim\AppData\Local\Temp\ipykernel_25000\3686687031.py:15: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(tsv_path, sep="\t")


CommonVoiceLocalDataset: 30000 segments
YouTubeAVDataset: 12088 segments (language=all)
CommonVoiceLocalDataset: 3300 segments


In [28]:
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader, val_loader,
    checkpoint_path=f"{CHECKPOINTS_DIR}/conformer_specaug_6_kenlm.pt",  # текущий лучший
    n_epochs=4,
    lr=1e-5,
    freeze_encoder_epochs=1,
    use_attention=True,
    save_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio.pt",
)

Fine-tune: 4 эпох, lr=1e-05, attention=True
Загружен: epoch=5, WER=0.537

  🔒 Энкодер заморожен | обучаемых: 4.4M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/4] step    0/3636 | loss 1.2511 | ctc 1.4709 | attn 0.7384 | 2s
  [1/4] step  100/3636 | loss 0.8547 | ctc 0.9361 | attn 0.6650 | 21s
  [1/4] step  200/3636 | loss 1.1520 | ctc 1.4108 | attn 0.5483 | 40s
  [1/4] step  300/3636 | loss 0.8341 | ctc 0.9508 | attn 0.5620 | 58s
  [1/4] step  400/3636 | loss 1.4058 | ctc 1.5830 | attn 0.9923 | 76s
  [1/4] step  500/3636 | loss 1.4632 | ctc 1.6888 | attn 0.9367 | 95s
  [1/4] step  600/3636 | loss 1.7965 | ctc 2.2514 | attn 0.7348 | 113s
  [1/4] step  700/3636 | loss 1.4184 | ctc 1.5741 | attn 1.0553 | 131s
  [1/4] step  800/3636 | loss 2.2700 | ctc 2.7037 | attn 1.2582 | 149s
  [1/4] step  900/3636 | loss 1.4076 | ctc 1.6276 | attn 0.8941 | 167s
  [1/4] step 1000/3636 | loss 1.0570 | ctc 1.2256 | attn 0.6635 | 185s
  [1/4] step 1100/3636 | loss 1.4083 | ctc 1.6561 | attn 0.8301 | 203s
  [1/4] step 1200/3636 | loss 1.1951 | ctc 1.3741 | attn 0.7774 | 222s
  [1/4] step 1300/3636 | loss 1.4582 | ctc 1.6230 | attn 1.0738 | 240s
  [1/4] step 

In [ ]:
# !!
model = finetune_on_loaders(
    train_loader, val_loader,
    checkpoint_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio.pt",  # текущий лучший
    n_epochs=4,
    lr=1.43e-5,
    freeze_encoder_epochs=0,
    use_attention=True,
    save_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio_1.pt",
)

Fine-tune: 4 эпох, lr=1.43e-05, attention=True
Загружен: epoch=1, WER=0.558

  🔓 Энкодер разморожен | всего: 16.7M

  [1/4] step    0/3636 | loss 1.4707 | ctc 1.7609 | attn 0.7938 | 0s


c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/4] step  100/3636 | loss 1.2072 | ctc 1.3575 | attn 0.8564 | 16s
  [1/4] step  200/3636 | loss 1.1061 | ctc 1.2955 | attn 0.6642 | 31s
  [1/4] step  300/3636 | loss 1.2155 | ctc 1.4058 | attn 0.7715 | 46s
  [1/4] step  400/3636 | loss 1.0176 | ctc 1.1632 | attn 0.6778 | 62s
  [1/4] step  500/3636 | loss 1.2892 | ctc 1.5190 | attn 0.7531 | 77s
  [1/4] step  600/3636 | loss 1.1805 | ctc 1.3739 | attn 0.7293 | 92s
  [1/4] step  700/3636 | loss 1.9477 | ctc 2.3878 | attn 0.9210 | 107s
  [1/4] step  800/3636 | loss 1.1402 | ctc 1.3292 | attn 0.6994 | 123s
  [1/4] step  900/3636 | loss 1.8526 | ctc 2.2318 | attn 0.9679 | 138s
  [1/4] step 1000/3636 | loss 1.7330 | ctc 2.0748 | attn 0.9357 | 153s
  [1/4] step 1100/3636 | loss 0.6231 | ctc 0.6413 | attn 0.5804 | 169s
  [1/4] step 1200/3636 | loss 1.4698 | ctc 1.8181 | attn 0.6571 | 184s
  [1/4] step 1300/3636 | loss 1.1085 | ctc 1.2870 | attn 0.6922 | 199s
  [1/4] step 1400/3636 | loss 1.0553 | ctc 1.2338 | attn 0.6388 | 215s
  [1/4] step

In [31]:
model = finetune_on_loaders(
    train_loader, val_loader,
    checkpoint_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio_1.pt",  # текущий лучший
    n_epochs=4,
    lr=3.43e-5,
    freeze_encoder_epochs=0,
    use_attention=True,
    save_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio_2.pt",
)

Fine-tune: 4 эпох, lr=3.43e-05, attention=True
Загружен: epoch=4, WER=0.572

  🔓 Энкодер разморожен | всего: 16.7M

  [1/4] step    0/3636 | loss 1.1859 | ctc 1.4227 | attn 0.6332 | 0s
  [1/4] step  100/3636 | loss 1.2574 | ctc 1.5065 | attn 0.6764 | 23s
  [1/4] step  200/3636 | loss 0.9360 | ctc 1.1391 | attn 0.4622 | 44s
  [1/4] step  300/3636 | loss 1.4778 | ctc 1.7491 | attn 0.8447 | 66s
  [1/4] step  400/3636 | loss 1.4897 | ctc 1.7581 | attn 0.8637 | 87s
  [1/4] step  500/3636 | loss 1.1308 | ctc 1.3405 | attn 0.6413 | 109s
  [1/4] step  600/3636 | loss 0.7625 | ctc 0.8482 | attn 0.5624 | 130s
  [1/4] step  700/3636 | loss 1.2015 | ctc 1.3754 | attn 0.7959 | 152s
  [1/4] step  800/3636 | loss 1.1682 | ctc 1.3614 | attn 0.7177 | 173s
  [1/4] step  900/3636 | loss 0.9685 | ctc 1.1581 | attn 0.5262 | 195s
  [1/4] step 1000/3636 | loss 0.8742 | ctc 1.0043 | attn 0.5709 | 217s
  [1/4] step 1100/3636 | loss 0.9919 | ctc 1.1819 | attn 0.5488 | 239s
  [1/4] step 1200/3636 | loss 0.9806 |

In [ ]:
# 50 - CV, 12k - yt
model = finetune_on_loaders(
    train_loader, val_loader,
    checkpoint_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio_1.pt",  # текущий лучший
    n_epochs=4,
    lr=3.43e-5,
    freeze_encoder_epochs=0,
    use_attention=True,
    save_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio_2.pt",
)

In [35]:
# weight 30/12 - CV/yt
model = finetune_on_loaders(
    train_loader, val_loader,
    checkpoint_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio_2.pt",  # текущий лучший
    n_epochs=5,
    lr=3.43e-5,
    freeze_encoder_epochs=0,
    use_attention=True,
    save_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio_3.pt",
)

Fine-tune: 5 эпох, lr=3.43e-05, attention=True
Загружен: epoch=2, WER=0.561

  🔓 Энкодер разморожен | всего: 16.7M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/3750 | loss 0.6879 | ctc 0.8251 | attn 0.3677 | 2s
  [1/5] step  100/3750 | loss 0.6569 | ctc 0.7176 | attn 0.5151 | 25s
  [1/5] step  200/3750 | loss 0.9550 | ctc 1.1269 | attn 0.5539 | 45s
  [1/5] step  300/3750 | loss 0.8757 | ctc 0.9483 | attn 0.7065 | 65s
  [1/5] step  400/3750 | loss 0.9942 | ctc 1.1454 | attn 0.6412 | 85s
  [1/5] step  500/3750 | loss 0.6761 | ctc 0.8101 | attn 0.3632 | 106s
  [1/5] step  600/3750 | loss 1.1462 | ctc 1.3594 | attn 0.6486 | 127s
  [1/5] step  700/3750 | loss 0.9365 | ctc 1.0868 | attn 0.5858 | 147s
  [1/5] step  800/3750 | loss 1.2606 | ctc 1.3702 | attn 1.0048 | 166s
  [1/5] step  900/3750 | loss 1.0196 | ctc 1.2425 | attn 0.4996 | 186s
  [1/5] step 1000/3750 | loss 1.2076 | ctc 1.4122 | attn 0.7301 | 205s
  [1/5] step 1100/3750 | loss 0.8114 | ctc 0.9666 | attn 0.4493 | 225s
  [1/5] step 1200/3750 | loss 1.1790 | ctc 1.4290 | attn 0.5957 | 244s
  [1/5] step 1300/3750 | loss 1.3787 | ctc 1.5638 | attn 0.9468 | 263s
  [1/5] step

In [10]:
# weight 30/12 - CV/yt
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

model = finetune_on_loaders(
    train_loader, val_loader,
    checkpoint_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio_3.pt",  # текущий лучший
    n_epochs=5,
    lr=1.23e-4,
    freeze_encoder_epochs=0,
    use_attention=True,
    save_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio_4.pt",
)

Fine-tune: 5 эпох, lr=0.000123, attention=True
Загружен: epoch=5, WER=0.560

  🔓 Энкодер разморожен | всего: 16.7M



c:\Users\Tim\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  [1/5] step    0/3750 | loss 1.1578 | ctc 1.4264 | attn 0.5310 | 2s
  [1/5] step  100/3750 | loss 0.7440 | ctc 0.8291 | attn 0.5455 | 26s
  [1/5] step  200/3750 | loss 0.7738 | ctc 0.9262 | attn 0.4182 | 47s
  [1/5] step  300/3750 | loss 1.2329 | ctc 1.4624 | attn 0.6975 | 68s
  [1/5] step  400/3750 | loss 0.7087 | ctc 0.8020 | attn 0.4909 | 89s
  [1/5] step  500/3750 | loss 0.7466 | ctc 0.8281 | attn 0.5563 | 109s
  [1/5] step  600/3750 | loss 0.7375 | ctc 0.8973 | attn 0.3646 | 129s
  [1/5] step  700/3750 | loss 0.6364 | ctc 0.7183 | attn 0.4451 | 149s
  [1/5] step  800/3750 | loss 1.0260 | ctc 1.1864 | attn 0.6519 | 169s
  [1/5] step  900/3750 | loss 0.7585 | ctc 0.8794 | attn 0.4762 | 189s
  [1/5] step 1000/3750 | loss 0.8984 | ctc 1.0057 | attn 0.6480 | 209s
  [1/5] step 1100/3750 | loss 1.0270 | ctc 1.1774 | attn 0.6762 | 228s
  [1/5] step 1200/3750 | loss 0.7124 | ctc 0.8221 | attn 0.4564 | 250s
  [1/5] step 1300/3750 | loss 0.8655 | ctc 0.9775 | attn 0.6043 | 269s
  [1/5] step

KeyboardInterrupt: 

In [ ]:
# ============================
# video in
CHECKPOINTS_DIR = "avsr_conformer/checkpoints"

# weight 30/12 - CV/yt
model = finetune_on_loaders(
    train_loader, val_loader,
    checkpoint_path=f"{CHECKPOINTS_DIR}/conformer_yt_audio_3.pt",  # текущий лучший
    n_epochs=5,
    lr=1.2e-5,
    freeze_encoder_epochs=2,
    use_attention=True,
    save_path=f"{CHECKPOINTS_DIR}/conformer_yt_video.pt",
)